In [ ]:
# Clone repo for lib/ access (Colab only)
import os
if not os.path.exists('/content/trading-strategies'):
    !git clone https://github.com/r-giov/trading-strategies.git /content/trading-strategies
os.chdir('/content/trading-strategies')

In [ ]:
#
 
!
p
i
p
 
i
n
s
t
a
l
l
 
y
f
i
n
a
n
c
e

#
 
!
p
i
p
 
i
n
s
t
a
l
l
 
T
A
-
L
i
b
 

#
 
!
p
i
p
 
i
n
s
t
a
l
l
 
n
u
m
p
y

#
 
!
p
i
p
 
i
n
s
t
a
l
l
 
p
a
n
d
a
s

#
 
!
p
i
p
 
i
n
s
t
a
l
l
 
v
e
c
t
o
r
b
t

#
 
!
p
i
p
 
i
n
s
t
a
l
l
 
s
c
i
p
y

In [ ]:
import sys, os
# Ensure lib/ is importable (for Google Sheets logging)
for _p in ['/content/trading-strategies', os.path.join(os.getcwd(), '..')]:
    if os.path.isdir(os.path.join(_p, 'lib')) and _p not in sys.path:
        sys.path.insert(0, _p)

import yfinance as yf
import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import warnings
from scipy import stats
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('future.no_silent_downcasting', True)
from lib.data_manager import download, download_multi, setup_colab_secrets

In [ ]:
# Data: yfinance (daily) | Alpaca (intraday) — auto-selected by data_manager
# For Colab: run setup_colab_secrets() if using Alpaca intraday features
# DOWNLOAD STOCK DATA FROM 2018 USING YFINANCE

# Configuration - Change these variables as needed
TICKER = 'SPY'  # Any ticker symbol (e.g., 'AAPL', 'MSFT', 'GOOGL')
START_DATE = '2018-01-01'  # Any start date in YYYY-MM-DD format

# Download data from start date onwards
stock_data = download(TICKER, START_DATE)

if not stock_data.empty:
    print(f"Successfully downloaded {len(stock_data)} records for {TICKER} from {START_DATE}")
    print(f"Data range: {stock_data.index.min().date()} to {stock_data.index.max().date()}")
    print("\nFirst 5 rows:")
    print(stock_data.head())
else:
    print(f"Failed to download {TICKER} data from yfinance")

# Display the downloaded data
stock_data

In [ ]:
#
 
T
E
C
H
N
I
C
A
L
 
A
N
A
L
Y
S
I
S
 
I
N
D
I
C
A
T
O
R
S
 
U
S
I
N
G
 
T
A
-
L
I
B


i
f
 
"
s
t
o
c
k
_
d
a
t
a
"
 
n
o
t
 
i
n
 
l
o
c
a
l
s
(
)
:

 
 
 
 
r
a
i
s
e
 
V
a
l
u
e
E
r
r
o
r
(
"
P
l
e
a
s
e
 
r
u
n
 
t
h
e
 
s
t
o
c
k
 
d
a
t
a
 
d
o
w
n
l
o
a
d
 
c
e
l
l
 
f
i
r
s
t
"
)


#
 
E
x
t
r
a
c
t
 
O
H
L
C
V
 
d
a
t
a
 
(
h
a
n
d
l
i
n
g
 
m
u
l
t
i
-
l
e
v
e
l
 
c
o
l
u
m
n
s
 
f
r
o
m
 
y
f
i
n
a
n
c
e
)

i
f
 
i
s
i
n
s
t
a
n
c
e
(
s
t
o
c
k
_
d
a
t
a
.
c
o
l
u
m
n
s
,
 
p
d
.
M
u
l
t
i
I
n
d
e
x
)
:

 
 
 
 
c
l
o
s
e
 
=
 
s
t
o
c
k
_
d
a
t
a
[
(
"
C
l
o
s
e
"
,
 
T
I
C
K
E
R
)
]
.
v
a
l
u
e
s

 
 
 
 
h
i
g
h
 
=
 
s
t
o
c
k
_
d
a
t
a
[
(
"
H
i
g
h
"
,
 
T
I
C
K
E
R
)
]
.
v
a
l
u
e
s

 
 
 
 
l
o
w
 
=
 
s
t
o
c
k
_
d
a
t
a
[
(
"
L
o
w
"
,
 
T
I
C
K
E
R
)
]
.
v
a
l
u
e
s

 
 
 
 
o
p
e
n
_
 
=
 
s
t
o
c
k
_
d
a
t
a
[
(
"
O
p
e
n
"
,
 
T
I
C
K
E
R
)
]
.
v
a
l
u
e
s

 
 
 
 
v
o
l
u
m
e
 
=
 
s
t
o
c
k
_
d
a
t
a
[
(
"
V
o
l
u
m
e
"
,
 
T
I
C
K
E
R
)
]
.
v
a
l
u
e
s

e
l
s
e
:

 
 
 
 
c
l
o
s
e
 
=
 
s
t
o
c
k
_
d
a
t
a
[
"
C
l
o
s
e
"
]
.
v
a
l
u
e
s

 
 
 
 
h
i
g
h
 
=
 
s
t
o
c
k
_
d
a
t
a
[
"
H
i
g
h
"
]
.
v
a
l
u
e
s

 
 
 
 
l
o
w
 
=
 
s
t
o
c
k
_
d
a
t
a
[
"
L
o
w
"
]
.
v
a
l
u
e
s

 
 
 
 
o
p
e
n
_
 
=
 
s
t
o
c
k
_
d
a
t
a
[
"
O
p
e
n
"
]
.
v
a
l
u
e
s

 
 
 
 
v
o
l
u
m
e
 
=
 
s
t
o
c
k
_
d
a
t
a
[
"
V
o
l
u
m
e
"
]
.
v
a
l
u
e
s


p
r
i
n
t
(
f
"
C
a
l
c
u
l
a
t
i
n
g
 
t
e
c
h
n
i
c
a
l
 
i
n
d
i
c
a
t
o
r
s
 
f
o
r
 
{
T
I
C
K
E
R
}
.
.
.
"
)


s
m
a
_
2
0
 
=
 
t
a
l
i
b
.
S
M
A
(
c
l
o
s
e
,
 
t
i
m
e
p
e
r
i
o
d
=
2
0
)

s
m
a
_
5
0
 
=
 
t
a
l
i
b
.
S
M
A
(
c
l
o
s
e
,
 
t
i
m
e
p
e
r
i
o
d
=
5
0
)

e
m
a
_
1
2
 
=
 
t
a
l
i
b
.
E
M
A
(
c
l
o
s
e
,
 
t
i
m
e
p
e
r
i
o
d
=
1
2
)

e
m
a
_
2
6
 
=
 
t
a
l
i
b
.
E
M
A
(
c
l
o
s
e
,
 
t
i
m
e
p
e
r
i
o
d
=
2
6
)

m
a
c
d
,
 
m
a
c
d
s
i
g
n
a
l
,
 
m
a
c
d
h
i
s
t
 
=
 
t
a
l
i
b
.
M
A
C
D
(
c
l
o
s
e
,
 
f
a
s
t
p
e
r
i
o
d
=
1
2
,
 
s
l
o
w
p
e
r
i
o
d
=
2
6
,
 
s
i
g
n
a
l
p
e
r
i
o
d
=
9
)

r
s
i
 
=
 
t
a
l
i
b
.
R
S
I
(
c
l
o
s
e
,
 
t
i
m
e
p
e
r
i
o
d
=
1
4
)

s
t
o
c
h
r
s
i
_
k
,
 
s
t
o
c
h
r
s
i
_
d
 
=
 
t
a
l
i
b
.
S
T
O
C
H
R
S
I
(
c
l
o
s
e
,
 
t
i
m
e
p
e
r
i
o
d
=
1
4
,
 
f
a
s
t
k
_
p
e
r
i
o
d
=
3
,
 
f
a
s
t
d
_
p
e
r
i
o
d
=
3
,
 
f
a
s
t
d
_
m
a
t
y
p
e
=
0
)


t
y
p
i
c
a
l
_
p
r
i
c
e
 
=
 
(
h
i
g
h
 
+
 
l
o
w
 
+
 
c
l
o
s
e
)
 
/
 
3

p
r
i
c
e
_
v
o
l
u
m
e
 
=
 
t
y
p
i
c
a
l
_
p
r
i
c
e
 
*
 
v
o
l
u
m
e

c
u
m
u
l
a
t
i
v
e
_
p
r
i
c
e
_
v
o
l
u
m
e
 
=
 
n
p
.
c
u
m
s
u
m
(
p
r
i
c
e
_
v
o
l
u
m
e
)

c
u
m
u
l
a
t
i
v
e
_
v
o
l
u
m
e
 
=
 
n
p
.
c
u
m
s
u
m
(
v
o
l
u
m
e
)

v
w
a
p
 
=
 
c
u
m
u
l
a
t
i
v
e
_
p
r
i
c
e
_
v
o
l
u
m
e
 
/
 
c
u
m
u
l
a
t
i
v
e
_
v
o
l
u
m
e


c
y
c
l
e
_
p
e
r
i
o
d
 
=
 
1
0

m
a
c
d
_
c
y
c
l
e
 
=
 
t
a
l
i
b
.
E
M
A
(
m
a
c
d
,
 
t
i
m
e
p
e
r
i
o
d
=
c
y
c
l
e
_
p
e
r
i
o
d
)

m
a
c
d
_
s
m
o
o
t
h
 
=
 
t
a
l
i
b
.
E
M
A
(
m
a
c
d
_
c
y
c
l
e
,
 
t
i
m
e
p
e
r
i
o
d
=
c
y
c
l
e
_
p
e
r
i
o
d
)

h
i
g
h
e
s
t
_
m
a
c
d
 
=
 
t
a
l
i
b
.
M
A
X
(
m
a
c
d
_
s
m
o
o
t
h
,
 
t
i
m
e
p
e
r
i
o
d
=
c
y
c
l
e
_
p
e
r
i
o
d
)

l
o
w
e
s
t
_
m
a
c
d
 
=
 
t
a
l
i
b
.
M
I
N
(
m
a
c
d
_
s
m
o
o
t
h
,
 
t
i
m
e
p
e
r
i
o
d
=
c
y
c
l
e
_
p
e
r
i
o
d
)

s
t
c
_
k
 
=
 
1
0
0
 
*
 
(
(
m
a
c
d
_
s
m
o
o
t
h
 
-
 
l
o
w
e
s
t
_
m
a
c
d
)
 
/
 
(
h
i
g
h
e
s
t
_
m
a
c
d
 
-
 
l
o
w
e
s
t
_
m
a
c
d
)
)

s
t
c
_
d
 
=
 
t
a
l
i
b
.
S
M
A
(
s
t
c
_
k
,
 
t
i
m
e
p
e
r
i
o
d
=
3
)


i
n
d
i
c
a
t
o
r
s
_
d
f
 
=
 
p
d
.
D
a
t
a
F
r
a
m
e
(
{

 
 
 
 
"
D
a
t
e
"
:
 
s
t
o
c
k
_
d
a
t
a
.
i
n
d
e
x
,
 
"
C
l
o
s
e
"
:
 
c
l
o
s
e
,
 
"
S
M
A
_
2
0
"
:
 
s
m
a
_
2
0
,
 
"
S
M
A
_
5
0
"
:
 
s
m
a
_
5
0
,

 
 
 
 
"
E
M
A
_
1
2
"
:
 
e
m
a
_
1
2
,
 
"
E
M
A
_
2
6
"
:
 
e
m
a
_
2
6
,
 
"
M
A
C
D
"
:
 
m
a
c
d
,
 
"
M
A
C
D
_
S
i
g
n
a
l
"
:
 
m
a
c
d
s
i
g
n
a
l
,

 
 
 
 
"
M
A
C
D
_
H
i
s
t
"
:
 
m
a
c
d
h
i
s
t
,
 
"
R
S
I
"
:
 
r
s
i
,
 
"
S
t
o
c
h
R
S
I
_
K
"
:
 
s
t
o
c
h
r
s
i
_
k
,
 
"
S
t
o
c
h
R
S
I
_
D
"
:
 
s
t
o
c
h
r
s
i
_
d
,

 
 
 
 
"
V
W
A
P
"
:
 
v
w
a
p
,
 
"
S
T
C
_
K
"
:
 
s
t
c
_
k
,
 
"
S
T
C
_
D
"
:
 
s
t
c
_
d

}
)


p
r
i
n
t
(
"
A
l
l
 
t
e
c
h
n
i
c
a
l
 
i
n
d
i
c
a
t
o
r
s
 
c
a
l
c
u
l
a
t
e
d
!
"
)

p
r
i
n
t
(
f
"
D
a
t
a
 
s
h
a
p
e
:
 
{
i
n
d
i
c
a
t
o
r
s
_
d
f
.
s
h
a
p
e
}
"
)

i
n
d
i
c
a
t
o
r
s
_
d
f
.
t
a
i
l
(
5
)

In [ ]:
#
 
P
R
E
P
A
R
E
 
P
R
I
C
E
 
S
E
R
I
E
S
 
+
 
T
R
A
I
N
/
V
A
L
 
S
P
L
I
T


w
a
r
n
i
n
g
s
.
f
i
l
t
e
r
w
a
r
n
i
n
g
s
(
"
i
g
n
o
r
e
"
,
 
m
e
s
s
a
g
e
=
"
D
e
g
r
e
e
s
 
o
f
 
f
r
e
e
d
o
m
 
<
=
 
0
 
f
o
r
 
s
l
i
c
e
"
,
 
c
a
t
e
g
o
r
y
=
R
u
n
t
i
m
e
W
a
r
n
i
n
g
)

w
a
r
n
i
n
g
s
.
f
i
l
t
e
r
w
a
r
n
i
n
g
s
(
"
i
g
n
o
r
e
"
,
 
m
e
s
s
a
g
e
=
"
i
n
v
a
l
i
d
 
v
a
l
u
e
 
e
n
c
o
u
n
t
e
r
e
d
 
i
n
 
s
c
a
l
a
r
 
d
i
v
i
d
e
"
,
 
c
a
t
e
g
o
r
y
=
R
u
n
t
i
m
e
W
a
r
n
i
n
g
)


d
e
f
 
s
e
l
e
c
t
_
c
l
o
s
e
_
s
e
r
i
e
s
(
d
f
,
 
t
i
c
k
e
r
)
:

 
 
 
 
i
f
 
i
s
i
n
s
t
a
n
c
e
(
d
f
.
c
o
l
u
m
n
s
,
 
p
d
.
M
u
l
t
i
I
n
d
e
x
)
:

 
 
 
 
 
 
 
 
i
f
 
(
'
C
l
o
s
e
'
,
 
t
i
c
k
e
r
)
 
i
n
 
d
f
.
c
o
l
u
m
n
s
:

 
 
 
 
 
 
 
 
 
 
 
 
s
 
=
 
d
f
[
(
'
C
l
o
s
e
'
,
 
t
i
c
k
e
r
)
]

 
 
 
 
 
 
 
 
e
l
s
e
:

 
 
 
 
 
 
 
 
 
 
 
 
c
o
l
s
 
=
 
[
c
 
f
o
r
 
c
 
i
n
 
d
f
.
c
o
l
u
m
n
s
 
i
f
 
'
C
l
o
s
e
'
 
i
n
 
s
t
r
(
c
)
]

 
 
 
 
 
 
 
 
 
 
 
 
i
f
 
n
o
t
 
c
o
l
s
:

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
r
a
i
s
e
 
K
e
y
E
r
r
o
r
(
"
C
l
o
s
e
 
n
o
t
 
f
o
u
n
d
"
)

 
 
 
 
 
 
 
 
 
 
 
 
s
 
=
 
d
f
[
c
o
l
s
[
0
]
]

 
 
 
 
e
l
s
e
:

 
 
 
 
 
 
 
 
s
 
=
 
d
f
[
'
C
l
o
s
e
'
]

 
 
 
 
r
e
t
u
r
n
 
s
.
a
s
t
y
p
e
(
f
l
o
a
t
)
.
s
q
u
e
e
z
e
(
)


d
e
f
 
s
e
l
e
c
t
_
s
e
r
i
e
s
(
d
f
,
 
t
i
c
k
e
r
,
 
c
o
l
_
n
a
m
e
)
:

 
 
 
 
"
"
"
E
x
t
r
a
c
t
 
a
n
y
 
O
H
L
C
V
 
s
e
r
i
e
s
 
h
a
n
d
l
i
n
g
 
M
u
l
t
i
I
n
d
e
x
.
"
"
"

 
 
 
 
i
f
 
i
s
i
n
s
t
a
n
c
e
(
d
f
.
c
o
l
u
m
n
s
,
 
p
d
.
M
u
l
t
i
I
n
d
e
x
)
:

 
 
 
 
 
 
 
 
i
f
 
(
c
o
l
_
n
a
m
e
,
 
t
i
c
k
e
r
)
 
i
n
 
d
f
.
c
o
l
u
m
n
s
:

 
 
 
 
 
 
 
 
 
 
 
 
s
 
=
 
d
f
[
(
c
o
l
_
n
a
m
e
,
 
t
i
c
k
e
r
)
]

 
 
 
 
 
 
 
 
e
l
s
e
:

 
 
 
 
 
 
 
 
 
 
 
 
c
o
l
s
 
=
 
[
c
 
f
o
r
 
c
 
i
n
 
d
f
.
c
o
l
u
m
n
s
 
i
f
 
c
o
l
_
n
a
m
e
 
i
n
 
s
t
r
(
c
)
]

 
 
 
 
 
 
 
 
 
 
 
 
i
f
 
n
o
t
 
c
o
l
s
:

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
r
a
i
s
e
 
K
e
y
E
r
r
o
r
(
f
"
{
c
o
l
_
n
a
m
e
}
 
n
o
t
 
f
o
u
n
d
"
)

 
 
 
 
 
 
 
 
 
 
 
 
s
 
=
 
d
f
[
c
o
l
s
[
0
]
]

 
 
 
 
e
l
s
e
:

 
 
 
 
 
 
 
 
s
 
=
 
d
f
[
c
o
l
_
n
a
m
e
]

 
 
 
 
r
e
t
u
r
n
 
s
.
a
s
t
y
p
e
(
f
l
o
a
t
)
.
s
q
u
e
e
z
e
(
)


c
l
o
s
e
_
s
e
r
i
e
s
 
=
 
s
e
l
e
c
t
_
c
l
o
s
e
_
s
e
r
i
e
s
(
s
t
o
c
k
_
d
a
t
a
,
 
T
I
C
K
E
R
)

c
l
o
s
e
_
s
e
r
i
e
s
.
n
a
m
e
 
=
 
'
p
r
i
c
e
'


l
o
w
_
s
e
r
i
e
s
_
f
u
l
l
 
=
 
s
e
l
e
c
t
_
s
e
r
i
e
s
(
s
t
o
c
k
_
d
a
t
a
,
 
T
I
C
K
E
R
,
 
'
L
o
w
'
)

l
o
w
_
s
e
r
i
e
s
_
f
u
l
l
.
n
a
m
e
 
=
 
'
l
o
w
'


h
i
g
h
_
s
e
r
i
e
s
_
f
u
l
l
 
=
 
s
e
l
e
c
t
_
s
e
r
i
e
s
(
s
t
o
c
k
_
d
a
t
a
,
 
T
I
C
K
E
R
,
 
'
H
i
g
h
'
)

h
i
g
h
_
s
e
r
i
e
s
_
f
u
l
l
.
n
a
m
e
 
=
 
'
h
i
g
h
'


T
R
A
I
N
_
R
A
T
I
O
 
=
 
0
.
6
0

s
p
l
i
t
_
i
d
x
 
=
 
i
n
t
(
l
e
n
(
c
l
o
s
e
_
s
e
r
i
e
s
)
 
*
 
T
R
A
I
N
_
R
A
T
I
O
)


t
r
a
i
n
_
c
l
o
s
e
 
=
 
c
l
o
s
e
_
s
e
r
i
e
s
.
i
l
o
c
[
:
s
p
l
i
t
_
i
d
x
]
.
c
o
p
y
(
)

v
a
l
_
c
l
o
s
e
 
 
 
=
 
c
l
o
s
e
_
s
e
r
i
e
s
.
i
l
o
c
[
s
p
l
i
t
_
i
d
x
:
]
.
c
o
p
y
(
)

t
r
a
i
n
_
l
o
w
 
 
 
=
 
l
o
w
_
s
e
r
i
e
s
_
f
u
l
l
.
i
l
o
c
[
:
s
p
l
i
t
_
i
d
x
]
.
c
o
p
y
(
)

v
a
l
_
l
o
w
 
 
 
 
 
=
 
l
o
w
_
s
e
r
i
e
s
_
f
u
l
l
.
i
l
o
c
[
s
p
l
i
t
_
i
d
x
:
]
.
c
o
p
y
(
)


p
r
i
n
t
(
f
"
D
a
t
a
 
r
e
a
d
y
:
 
t
r
a
i
n
=
{
t
r
a
i
n
_
c
l
o
s
e
.
i
n
d
e
x
[
0
]
.
d
a
t
e
(
)
}
 
-
>
 
{
t
r
a
i
n
_
c
l
o
s
e
.
i
n
d
e
x
[
-
1
]
.
d
a
t
e
(
)
}
 
(
{
l
e
n
(
t
r
a
i
n
_
c
l
o
s
e
)
}
 
b
a
r
s
)
"
)

p
r
i
n
t
(
f
"
 
 
 
 
 
 
 
 
 
 
 
 
v
a
l
 
 
=
{
v
a
l
_
c
l
o
s
e
.
i
n
d
e
x
[
0
]
.
d
a
t
e
(
)
}
 
-
>
 
{
v
a
l
_
c
l
o
s
e
.
i
n
d
e
x
[
-
1
]
.
d
a
t
e
(
)
}
 
(
{
l
e
n
(
v
a
l
_
c
l
o
s
e
)
}
 
b
a
r
s
)
"
)

L
O
S
T
 
M
O
M
E
N
T
U
M
 
S
T
R
A
T
E
G
Y
 
G
R
I
D
 
S
E
A
R
C
H
 
-
 
T
R
A
I
N
I
N
G
 
S
E
T

-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-


T
h
i
s
 
s
e
c
t
i
o
n
 
p
e
r
f
o
r
m
s
 
a
 
c
o
m
p
r
e
h
e
n
s
i
v
e
 
g
r
i
d
 
s
e
a
r
c
h
 
o
p
t
i
m
i
z
a
t
i
o
n
 
f
o
r
 
*
*
T
o
m
 
H
o
u
g
a
a
r
d
'
s
 
L
o
s
t
 
M
o
m
e
n
t
u
m
 
S
t
r
a
t
e
g
y
*
*
 
u
s
i
n
g
 
o
n
l
y
 
t
h
e
 
*
*
t
r
a
i
n
i
n
g
 
d
a
t
a
*
*
.


*
*
S
t
r
a
t
e
g
y
 
L
o
g
i
c
 
(
L
o
n
g
 
O
n
l
y
)
*
*
:

1
.
 
C
o
m
p
u
t
e
 
a
 
m
o
v
i
n
g
 
a
v
e
r
a
g
e
 
(
S
M
A
 
o
r
 
E
M
A
)
 
a
s
 
t
r
e
n
d
 
f
i
l
t
e
r

2
.
 
*
*
T
r
e
n
d
 
U
p
*
*
:
 
C
l
o
s
e
 
>
 
M
A
 
A
N
D
 
M
A
 
i
s
 
s
l
o
p
i
n
g
 
u
p
w
a
r
d
 
(
M
A
 
t
o
d
a
y
 
>
 
M
A
 
N
 
b
a
r
s
 
a
g
o
)

3
.
 
*
*
L
o
s
t
 
M
o
m
e
n
t
u
m
*
*
:
 
P
r
i
c
e
 
t
a
k
e
s
 
o
u
t
 
(
b
r
e
a
k
s
 
b
e
l
o
w
)
 
a
 
p
r
e
v
i
o
u
s
 
s
w
i
n
g
 
l
o
w
 
t
h
a
t
 
i
s
 
a
t
 
l
e
a
s
t
 
`
s
w
i
n
g
_
l
o
o
k
b
a
c
k
`
 
b
a
r
s
 
o
l
d

4
.
 
*
*
R
e
c
o
v
e
r
y
*
*
:
 
P
r
i
c
e
 
c
l
o
s
e
s
 
b
a
c
k
 
A
B
O
V
E
 
t
h
e
 
b
r
o
k
e
n
 
s
w
i
n
g
 
l
o
w
 
l
e
v
e
l
 
-
>
 
B
U
Y
 
s
i
g
n
a
l

5
.
 
*
*
E
x
i
t
*
*
:
 
C
l
o
s
e
 
<
 
M
A
 
(
t
r
e
n
d
 
i
s
 
b
r
o
k
e
n
)


A
l
l
 
s
i
g
n
a
l
s
 
a
r
e
 
s
h
i
f
t
e
d
 
b
y
 
1
 
b
a
r
 
t
o
 
p
r
e
v
e
n
t
 
l
o
o
k
a
h
e
a
d
 
b
i
a
s
.
 
T
h
e
 
i
d
e
a
 
i
s
 
t
h
a
t
 
i
n
 
a
n
 
u
p
t
r
e
n
d
,
 
w
h
e
n
 
p
r
i
c
e
 
b
r
i
e
f
l
y
 
l
o
s
e
s
 
m
o
m
e
n
t
u
m
 
b
y
 
b
r
e
a
k
i
n
g
 
a
 
s
w
i
n
g
 
l
o
w
 
b
u
t
 
q
u
i
c
k
l
y
 
r
e
c
o
v
e
r
s
,
 
i
t
 
r
e
p
r
e
s
e
n
t
s
 
a
 
h
i
g
h
-
p
r
o
b
a
b
i
l
i
t
y
 
l
o
n
g
 
e
n
t
r
y
.


*
*
G
r
i
d
 
S
e
a
r
c
h
 
P
a
r
a
m
e
t
e
r
s
*
*
:

-
 
`
m
a
_
p
e
r
i
o
d
`
:
 
[
2
0
,
 
3
0
,
 
4
0
,
 
5
0
,
 
6
2
,
 
8
0
,
 
1
0
0
]
 
(
6
2
 
i
s
 
H
o
u
g
a
a
r
d
'
s
 
d
e
f
a
u
l
t
)

-
 
`
m
a
_
t
y
p
e
`
:
 
[
'
S
M
A
'
,
 
'
E
M
A
'
]

-
 
`
s
l
o
p
e
_
l
o
o
k
b
a
c
k
`
:
 
[
3
,
 
5
,
 
8
,
 
1
0
]
 
(
b
a
r
s
 
t
o
 
m
e
a
s
u
r
e
 
M
A
 
s
l
o
p
e
)

-
 
`
s
w
i
n
g
_
l
o
o
k
b
a
c
k
`
:
 
[
8
,
 
1
0
,
 
1
5
,
 
2
0
]
 
(
m
i
n
i
m
u
m
 
a
g
e
 
o
f
 
t
h
e
 
b
r
o
k
e
n
 
s
w
i
n
g
 
l
o
w
)


-
-
-

In [ ]:
#
 
D
e
f
i
n
e
 
P
a
r
a
m
e
t
e
r
 
R
a
n
g
e
s
 
f
o
r
 
L
o
s
t
 
M
o
m
e
n
t
u
m
 
S
t
r
a
t
e
g
y


m
a
_
p
e
r
i
o
d
_
r
a
n
g
e
 
 
 
 
=
 
[
2
0
,
 
3
0
,
 
4
0
,
 
5
0
,
 
6
2
,
 
8
0
,
 
1
0
0
]
 
 
 
#
 
7
 
v
a
l
u
e
s
 
(
6
2
 
=
 
H
o
u
g
a
a
r
d
 
d
e
f
a
u
l
t
)

m
a
_
t
y
p
e
_
r
a
n
g
e
 
 
 
 
 
 
=
 
[
'
S
M
A
'
,
 
'
E
M
A
'
]
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
#
 
2
 
v
a
l
u
e
s

s
l
o
p
e
_
l
o
o
k
b
a
c
k
_
r
a
n
g
e
 
=
 
[
3
,
 
5
,
 
8
,
 
1
0
]
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
#
 
4
 
v
a
l
u
e
s

s
w
i
n
g
_
l
o
o
k
b
a
c
k
_
r
a
n
g
e
 
=
 
[
8
,
 
1
0
,
 
1
5
,
 
2
0
]
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
#
 
4
 
v
a
l
u
e
s


p
r
i
n
t
(
"
M
A
 
P
e
r
i
o
d
s
 
(
t
r
e
n
d
 
f
i
l
t
e
r
)
:
"
)

f
o
r
 
i
,
 
p
e
r
i
o
d
 
i
n
 
e
n
u
m
e
r
a
t
e
(
m
a
_
p
e
r
i
o
d
_
r
a
n
g
e
,
 
1
)
:

 
 
 
 
p
r
i
n
t
(
f
"
 
 
{
i
}
.
 
{
p
e
r
i
o
d
}
 
p
e
r
i
o
d
s
"
)


p
r
i
n
t
(
"
\
n
M
A
 
T
y
p
e
s
:
"
)

f
o
r
 
i
,
 
m
t
 
i
n
 
e
n
u
m
e
r
a
t
e
(
m
a
_
t
y
p
e
_
r
a
n
g
e
,
 
1
)
:

 
 
 
 
p
r
i
n
t
(
f
"
 
 
{
i
}
.
 
{
m
t
}
"
)


p
r
i
n
t
(
"
\
n
S
l
o
p
e
 
L
o
o
k
b
a
c
k
 
(
b
a
r
s
 
t
o
 
m
e
a
s
u
r
e
 
M
A
 
s
l
o
p
e
)
:
"
)

f
o
r
 
i
,
 
s
l
 
i
n
 
e
n
u
m
e
r
a
t
e
(
s
l
o
p
e
_
l
o
o
k
b
a
c
k
_
r
a
n
g
e
,
 
1
)
:

 
 
 
 
p
r
i
n
t
(
f
"
 
 
{
i
}
.
 
{
s
l
}
 
b
a
r
s
"
)


p
r
i
n
t
(
"
\
n
S
w
i
n
g
 
L
o
o
k
b
a
c
k
 
(
m
i
n
i
m
u
m
 
a
g
e
 
o
f
 
b
r
o
k
e
n
 
s
w
i
n
g
 
l
o
w
)
:
"
)

f
o
r
 
i
,
 
s
w
 
i
n
 
e
n
u
m
e
r
a
t
e
(
s
w
i
n
g
_
l
o
o
k
b
a
c
k
_
r
a
n
g
e
,
 
1
)
:

 
 
 
 
p
r
i
n
t
(
f
"
 
 
{
i
}
.
 
{
s
w
}
 
b
a
r
s
"
)


#
 
G
e
n
e
r
a
t
e
 
a
l
l
 
c
o
m
b
i
n
a
t
i
o
n
s

f
r
o
m
 
i
t
e
r
t
o
o
l
s
 
i
m
p
o
r
t
 
p
r
o
d
u
c
t

l
m
_
c
o
m
b
i
n
a
t
i
o
n
s
 
=
 
l
i
s
t
(
p
r
o
d
u
c
t
(
m
a
_
p
e
r
i
o
d
_
r
a
n
g
e
,
 
m
a
_
t
y
p
e
_
r
a
n
g
e
,
 
s
l
o
p
e
_
l
o
o
k
b
a
c
k
_
r
a
n
g
e
,
 
s
w
i
n
g
_
l
o
o
k
b
a
c
k
_
r
a
n
g
e
)
)


p
r
i
n
t
(
f
"
\
n
G
e
n
e
r
a
t
e
d
 
{
l
e
n
(
l
m
_
c
o
m
b
i
n
a
t
i
o
n
s
)
}
 
L
o
s
t
 
M
o
m
e
n
t
u
m
 
c
o
m
b
i
n
a
t
i
o
n
s
"
)

p
r
i
n
t
(
f
"
 
 
{
l
e
n
(
m
a
_
p
e
r
i
o
d
_
r
a
n
g
e
)
}
 
x
 
{
l
e
n
(
m
a
_
t
y
p
e
_
r
a
n
g
e
)
}
 
x
 
{
l
e
n
(
s
l
o
p
e
_
l
o
o
k
b
a
c
k
_
r
a
n
g
e
)
}
 
x
 
{
l
e
n
(
s
w
i
n
g
_
l
o
o
k
b
a
c
k
_
r
a
n
g
e
)
}
 
=
 
{
l
e
n
(
l
m
_
c
o
m
b
i
n
a
t
i
o
n
s
)
}
"
)


p
r
i
n
t
(
"
\
n
F
i
r
s
t
 
1
0
 
c
o
m
b
i
n
a
t
i
o
n
s
 
p
r
e
v
i
e
w
:
"
)

f
o
r
 
i
,
 
(
m
a
_
p
,
 
m
a
_
t
,
 
s
l
,
 
s
w
)
 
i
n
 
e
n
u
m
e
r
a
t
e
(
l
m
_
c
o
m
b
i
n
a
t
i
o
n
s
[
:
1
0
]
,
 
1
)
:

 
 
 
 
p
r
i
n
t
(
f
"
 
 
{
i
:
2
d
}
.
 
M
A
:
 
{
m
a
_
t
}
(
{
m
a
_
p
}
)
 
|
 
S
l
o
p
e
 
L
B
:
 
{
s
l
}
 
|
 
S
w
i
n
g
 
L
B
:
 
{
s
w
}
"
)

i
f
 
l
e
n
(
l
m
_
c
o
m
b
i
n
a
t
i
o
n
s
)
 
>
 
1
0
:

 
 
 
 
p
r
i
n
t
(
f
"
 
 
 
.
.
.
 
a
n
d
 
{
l
e
n
(
l
m
_
c
o
m
b
i
n
a
t
i
o
n
s
)
 
-
 
1
0
}
 
m
o
r
e
 
c
o
m
b
i
n
a
t
i
o
n
s
"
)


p
r
i
n
t
(
"
\
n
R
e
a
d
y
 
t
o
 
t
e
s
t
 
a
l
l
 
c
o
m
b
i
n
a
t
i
o
n
s
 
o
n
 
t
r
a
i
n
i
n
g
 
d
a
t
a
!
"
)

In [ ]:
#
 
I
n
i
t
i
a
l
i
z
e
 
L
o
s
t
 
M
o
m
e
n
t
u
m
 
R
e
s
u
l
t
s
 
C
o
l
l
e
c
t
i
o
n
 
S
y
s
t
e
m


g
r
i
d
_
s
e
a
r
c
h
_
r
e
s
u
l
t
s
 
=
 
[
]


p
r
i
n
t
(
"
L
o
s
t
 
M
o
m
e
n
t
u
m
 
R
e
s
u
l
t
s
 
C
o
l
l
e
c
t
i
o
n
 
S
y
s
t
e
m
 
I
n
i
t
i
a
l
i
z
e
d
"
)

p
r
i
n
t
(
f
"
 
 
 
-
 
W
i
l
l
 
t
e
s
t
 
{
l
e
n
(
l
m
_
c
o
m
b
i
n
a
t
i
o
n
s
)
}
 
p
a
r
a
m
e
t
e
r
 
c
o
m
b
i
n
a
t
i
o
n
s
"
)

p
r
i
n
t
(
"
 
 
 
-
 
R
e
s
u
l
t
s
 
w
i
l
l
 
b
e
 
s
t
o
r
e
d
 
i
n
 
'
g
r
i
d
_
s
e
a
r
c
h
_
r
e
s
u
l
t
s
'
 
l
i
s
t
"
)


m
e
t
r
i
c
s
_
t
o
_
c
o
l
l
e
c
t
 
=
 
[

 
 
 
 
"
m
a
_
p
e
r
i
o
d
"
,
 
"
m
a
_
t
y
p
e
"
,
 
"
s
l
o
p
e
_
l
o
o
k
b
a
c
k
"
,
 
"
s
w
i
n
g
_
l
o
o
k
b
a
c
k
"
,

 
 
 
 
"
t
o
t
a
l
_
r
e
t
u
r
n
"
,
 
"
a
n
n
u
a
l
i
z
e
d
_
r
e
t
u
r
n
"
,

 
 
 
 
"
s
h
a
r
p
e
_
r
a
t
i
o
"
,
 
"
s
o
r
t
i
n
o
_
r
a
t
i
o
"
,
 
"
c
a
l
m
a
r
_
r
a
t
i
o
"
,
 
"
o
m
e
g
a
_
r
a
t
i
o
"
,

 
 
 
 
"
i
n
f
o
r
m
a
t
i
o
n
_
r
a
t
i
o
"
,
 
"
t
a
i
l
_
r
a
t
i
o
"
,
 
"
d
e
f
l
a
t
e
d
_
s
h
a
r
p
e
_
r
a
t
i
o
"
,

 
 
 
 
"
m
a
x
_
d
r
a
w
d
o
w
n
"
,
 
"
v
o
l
a
t
i
l
i
t
y
"
,
 
"
u
l
c
e
r
_
i
n
d
e
x
"
,

 
 
 
 
"
w
i
n
_
r
a
t
e
"
,
 
"
t
o
t
a
l
_
t
r
a
d
e
s
"
,
 
"
a
v
g
_
t
r
a
d
e
_
d
u
r
a
t
i
o
n
"
,
 
"
e
x
p
e
c
t
a
n
c
y
"
,

 
 
 
 
"
p
r
o
f
i
t
_
f
a
c
t
o
r
"
,
 
"
s
q
n
"
,

 
 
 
 
"
p
a
y
o
f
f
_
r
a
t
i
o
"
,
 
"
l
a
r
g
e
s
t
_
w
i
n
"
,
 
"
l
a
r
g
e
s
t
_
l
o
s
s
"
,

 
 
 
 
"
a
v
g
_
w
i
n
_
a
m
o
u
n
t
"
,
 
"
a
v
g
_
l
o
s
s
_
a
m
o
u
n
t
"
,
 
"
w
i
n
n
i
n
g
_
s
t
r
e
a
k
"
,
 
"
l
o
s
i
n
g
_
s
t
r
e
a
k
"
,

 
 
 
 
"
r
e
c
o
v
e
r
y
_
f
a
c
t
o
r
"
,
 
"
g
a
i
n
_
t
o
_
p
a
i
n
_
r
a
t
i
o
"
,
 
"
s
e
r
e
n
i
t
y
_
i
n
d
e
x
"

]


p
r
i
n
t
(
"
\
n
M
e
t
r
i
c
s
 
t
o
 
c
o
l
l
e
c
t
 
f
o
r
 
e
a
c
h
 
c
o
m
b
i
n
a
t
i
o
n
:
"
)

f
o
r
 
i
,
 
m
e
t
r
i
c
 
i
n
 
e
n
u
m
e
r
a
t
e
(
m
e
t
r
i
c
s
_
t
o
_
c
o
l
l
e
c
t
,
 
1
)
:

 
 
 
 
p
r
i
n
t
(
f
"
 
 
{
i
}
.
 
{
m
e
t
r
i
c
.
r
e
p
l
a
c
e
(
'
_
'
,
 
'
 
'
)
.
t
i
t
l
e
(
)
}
"
)


p
r
i
n
t
(
f
"
\
n
T
o
t
a
l
 
m
e
t
r
i
c
s
 
p
e
r
 
c
o
m
b
i
n
a
t
i
o
n
:
 
{
l
e
n
(
m
e
t
r
i
c
s
_
t
o
_
c
o
l
l
e
c
t
)
}
"
)

p
r
i
n
t
(
"
R
e
a
d
y
 
t
o
 
s
t
a
r
t
 
t
h
e
 
L
o
s
t
 
M
o
m
e
n
t
u
m
 
g
r
i
d
 
s
e
a
r
c
h
!
"
)

In [ ]:
#
 
L
O
S
T
 
M
O
M
E
N
T
U
M
 
G
R
I
D
 
S
E
A
R
C
H
 
O
N
 
T
R
A
I
N
I
N
G
 
D
A
T
A


p
r
i
n
t
(
"
I
N
I
T
I
A
T
I
N
G
 
L
O
S
T
 
M
O
M
E
N
T
U
M
 
G
R
I
D
 
S
E
A
R
C
H
 
O
P
T
I
M
I
Z
A
T
I
O
N
"
)

p
r
i
n
t
(
"
=
"
 
*
 
7
0
)

p
r
i
n
t
(
f
"
T
e
s
t
i
n
g
 
S
t
r
a
t
e
g
y
:
 
L
o
s
t
 
M
o
m
e
n
t
u
m
 
(
H
o
u
g
a
a
r
d
)
"
)

p
r
i
n
t
(
f
"
T
r
a
i
n
i
n
g
 
P
e
r
i
o
d
:
 
{
t
r
a
i
n
_
c
l
o
s
e
.
i
n
d
e
x
[
0
]
.
d
a
t
e
(
)
}
 
-
>
 
{
t
r
a
i
n
_
c
l
o
s
e
.
i
n
d
e
x
[
-
1
]
.
d
a
t
e
(
)
}
"
)

p
r
i
n
t
(
f
"
I
n
i
t
i
a
l
 
C
a
p
i
t
a
l
:
 
$
1
0
0
,
0
0
0
"
)

p
r
i
n
t
(
f
"
T
r
a
n
s
a
c
t
i
o
n
 
C
o
s
t
s
:
 
0
.
0
5
%
 
p
e
r
 
t
r
a
d
e
 
(
f
e
e
s
 
+
 
s
l
i
p
p
a
g
e
)
"
)

p
r
i
n
t
(
f
"
O
p
t
i
m
i
z
a
t
i
o
n
 
M
e
t
r
i
c
:
 
S
h
a
r
p
e
 
R
a
t
i
o
 
(
r
i
s
k
-
a
d
j
u
s
t
e
d
 
r
e
t
u
r
n
s
)
"
)

p
r
i
n
t
(
"
=
"
 
*
 
7
0
)


t
o
t
a
l
_
c
o
m
b
i
n
a
t
i
o
n
s
 
=
 
l
e
n
(
l
m
_
c
o
m
b
i
n
a
t
i
o
n
s
)

p
r
i
n
t
(
f
"
T
o
t
a
l
 
c
o
m
b
i
n
a
t
i
o
n
s
 
t
o
 
t
e
s
t
:
 
{
t
o
t
a
l
_
c
o
m
b
i
n
a
t
i
o
n
s
}
"
)

p
r
i
n
t
(
f
"
P
r
o
c
e
s
s
i
n
g
 
s
e
q
u
e
n
t
i
a
l
l
y
 
w
i
t
h
 
p
r
o
g
r
e
s
s
 
e
v
e
r
y
 
1
0
0
 
c
o
m
b
o
s
\
n
"
)


g
r
i
d
_
s
e
a
r
c
h
_
r
e
s
u
l
t
s
 
=
 
[
]

s
u
c
c
e
s
s
f
u
l
_
t
e
s
t
s
 
=
 
0

f
a
i
l
e
d
_
t
e
s
t
s
 
=
 
0

s
k
i
p
p
e
d
_
l
o
w
_
t
r
a
d
e
s
 
=
 
0


t
r
a
i
n
_
c
l
o
s
e
_
v
a
l
s
 
=
 
t
r
a
i
n
_
c
l
o
s
e
.
v
a
l
u
e
s
.
a
s
t
y
p
e
(
f
l
o
a
t
)

t
r
a
i
n
_
l
o
w
_
v
a
l
s
 
=
 
t
r
a
i
n
_
l
o
w
.
v
a
l
u
e
s
.
a
s
t
y
p
e
(
f
l
o
a
t
)

t
r
a
i
n
_
i
d
x
 
=
 
t
r
a
i
n
_
c
l
o
s
e
.
i
n
d
e
x

y
e
a
r
s
 
=
 
m
a
x
(
(
t
r
a
i
n
_
i
d
x
[
-
1
]
 
-
 
t
r
a
i
n
_
i
d
x
[
0
]
)
.
d
a
y
s
 
/
 
3
6
5
.
2
5
,
 
1
e
-
9
)


d
e
f
 
_
l
o
s
t
_
m
o
m
e
n
t
u
m
_
s
i
g
n
a
l
s
(
c
l
o
s
e
_
s
,
 
l
o
w
_
s
,
 
m
a
_
p
e
r
i
o
d
,
 
m
a
_
t
y
p
e
,
 
s
l
o
p
e
_
l
o
o
k
b
a
c
k
,
 
s
w
i
n
g
_
l
o
o
k
b
a
c
k
)
:

 
 
 
 
"
"
"
G
e
n
e
r
a
t
e
 
L
o
s
t
 
M
o
m
e
n
t
u
m
 
e
n
t
r
y
/
e
x
i
t
 
s
i
g
n
a
l
s
 
w
i
t
h
 
1
-
b
a
r
 
e
x
e
c
u
t
i
o
n
 
d
e
l
a
y
.
"
"
"

 
 
 
 
i
d
x
 
=
 
c
l
o
s
e
_
s
.
i
n
d
e
x

 
 
 
 
c
_
v
a
l
s
 
=
 
c
l
o
s
e
_
s
.
v
a
l
u
e
s
.
a
s
t
y
p
e
(
f
l
o
a
t
)

 
 
 
 
l
_
v
a
l
s
 
=
 
l
o
w
_
s
.
v
a
l
u
e
s
.
a
s
t
y
p
e
(
f
l
o
a
t
)


 
 
 
 
#
 
C
o
m
p
u
t
e
 
M
A

 
 
 
 
i
f
 
m
a
_
t
y
p
e
 
=
=
 
'
S
M
A
'
:

 
 
 
 
 
 
 
 
m
a
_
v
a
l
s
 
=
 
t
a
l
i
b
.
S
M
A
(
c
_
v
a
l
s
,
 
t
i
m
e
p
e
r
i
o
d
=
m
a
_
p
e
r
i
o
d
)

 
 
 
 
e
l
s
e
:

 
 
 
 
 
 
 
 
m
a
_
v
a
l
s
 
=
 
t
a
l
i
b
.
E
M
A
(
c
_
v
a
l
s
,
 
t
i
m
e
p
e
r
i
o
d
=
m
a
_
p
e
r
i
o
d
)


 
 
 
 
m
a
 
=
 
p
d
.
S
e
r
i
e
s
(
m
a
_
v
a
l
s
,
 
i
n
d
e
x
=
i
d
x
)

 
 
 
 
c
l
o
s
e
_
p
d
 
=
 
p
d
.
S
e
r
i
e
s
(
c
_
v
a
l
s
,
 
i
n
d
e
x
=
i
d
x
)

 
 
 
 
l
o
w
_
p
d
 
=
 
p
d
.
S
e
r
i
e
s
(
l
_
v
a
l
s
,
 
i
n
d
e
x
=
i
d
x
)


 
 
 
 
#
 
T
r
e
n
d
 
c
o
n
d
i
t
i
o
n
s

 
 
 
 
t
r
e
n
d
_
u
p
 
=
 
c
l
o
s
e
_
p
d
 
>
 
m
a

 
 
 
 
s
l
o
p
e
_
u
p
 
=
 
m
a
 
>
 
m
a
.
s
h
i
f
t
(
s
l
o
p
e
_
l
o
o
k
b
a
c
k
)


 
 
 
 
#
 
S
w
i
n
g
 
l
o
w
 
d
e
t
e
c
t
i
o
n
:
 
l
o
w
e
s
t
 
l
o
w
 
i
n
 
p
a
s
t
 
s
w
i
n
g
_
l
o
o
k
b
a
c
k
 
b
a
r
s
 
(
s
h
i
f
t
e
d
 
b
y
 
1
 
t
o
 
u
s
e
 
p
r
e
v
i
o
u
s
 
b
a
r
'
s
 
s
w
i
n
g
 
l
o
w
)

 
 
 
 
s
w
i
n
g
_
l
o
w
_
r
a
w
 
=
 
p
d
.
S
e
r
i
e
s
(
t
a
l
i
b
.
M
I
N
(
l
_
v
a
l
s
,
 
t
i
m
e
p
e
r
i
o
d
=
s
w
i
n
g
_
l
o
o
k
b
a
c
k
)
,
 
i
n
d
e
x
=
i
d
x
)

 
 
 
 
s
w
i
n
g
_
l
o
w
 
=
 
s
w
i
n
g
_
l
o
w
_
r
a
w
.
s
h
i
f
t
(
1
)
 
 
#
 
p
r
e
v
i
o
u
s
 
b
a
r
'
s
 
s
w
i
n
g
 
l
o
w
 
l
e
v
e
l


 
 
 
 
#
 
L
o
s
t
 
m
o
m
e
n
t
u
m
:
 
y
e
s
t
e
r
d
a
y
'
s
 
l
o
w
 
b
r
o
k
e
 
b
e
l
o
w
 
t
h
e
 
s
w
i
n
g
 
l
o
w
 
l
e
v
e
l

 
 
 
 
b
r
o
k
e
_
l
o
w
 
=
 
(
l
o
w
_
p
d
.
s
h
i
f
t
(
1
)
 
<
=
 
s
w
i
n
g
_
l
o
w
.
s
h
i
f
t
(
1
)
)


 
 
 
 
#
 
R
e
c
o
v
e
r
y
:
 
t
o
d
a
y
'
s
 
c
l
o
s
e
 
i
s
 
b
a
c
k
 
a
b
o
v
e
 
t
h
e
 
s
w
i
n
g
 
l
o
w
 
l
e
v
e
l

 
 
 
 
r
e
c
o
v
e
r
e
d
 
=
 
c
l
o
s
e
_
p
d
 
>
 
s
w
i
n
g
_
l
o
w


 
 
 
 
#
 
C
o
m
b
i
n
e
d
 
e
n
t
r
y
:
 
b
r
o
k
e
 
l
o
w
 
+
 
r
e
c
o
v
e
r
e
d
 
+
 
t
r
e
n
d
 
u
p
 
+
 
s
l
o
p
e
 
u
p

 
 
 
 
e
n
t
r
i
e
s
_
r
a
w
 
=
 
b
r
o
k
e
_
l
o
w
 
&
 
r
e
c
o
v
e
r
e
d
 
&
 
t
r
e
n
d
_
u
p
 
&
 
s
l
o
p
e
_
u
p


 
 
 
 
#
 
E
x
i
t
:
 
c
l
o
s
e
 
b
e
l
o
w
 
M
A

 
 
 
 
e
x
i
t
s
_
r
a
w
 
=
 
c
l
o
s
e
_
p
d
 
<
 
m
a


 
 
 
 
#
 
1
-
b
a
r
 
e
x
e
c
u
t
i
o
n
 
d
e
l
a
y

 
 
 
 
e
n
t
r
i
e
s
 
=
 
p
d
.
S
e
r
i
e
s
(
n
p
.
w
h
e
r
e
(
e
n
t
r
i
e
s
_
r
a
w
.
s
h
i
f
t
(
1
)
.
i
s
n
a
(
)
,
 
F
a
l
s
e
,
 
e
n
t
r
i
e
s
_
r
a
w
.
s
h
i
f
t
(
1
)
)
,
 
i
n
d
e
x
=
i
d
x
,
 
d
t
y
p
e
=
b
o
o
l
)

 
 
 
 
e
x
i
t
s
 
=
 
p
d
.
S
e
r
i
e
s
(
n
p
.
w
h
e
r
e
(
e
x
i
t
s
_
r
a
w
.
s
h
i
f
t
(
1
)
.
i
s
n
a
(
)
,
 
F
a
l
s
e
,
 
e
x
i
t
s
_
r
a
w
.
s
h
i
f
t
(
1
)
)
,
 
i
n
d
e
x
=
i
d
x
,
 
d
t
y
p
e
=
b
o
o
l
)


 
 
 
 
r
e
t
u
r
n
 
e
n
t
r
i
e
s
,
 
e
x
i
t
s


f
o
r
 
c
o
m
b
o
_
n
u
m
,
 
(
m
a
_
p
e
r
i
o
d
,
 
m
a
_
t
y
p
e
,
 
s
l
o
p
e
_
l
b
,
 
s
w
i
n
g
_
l
b
)
 
i
n
 
e
n
u
m
e
r
a
t
e
(
l
m
_
c
o
m
b
i
n
a
t
i
o
n
s
,
 
1
)
:

 
 
 
 
t
r
y
:

 
 
 
 
 
 
 
 
e
n
t
r
i
e
s
,
 
e
x
i
t
s
 
=
 
_
l
o
s
t
_
m
o
m
e
n
t
u
m
_
s
i
g
n
a
l
s
(
t
r
a
i
n
_
c
l
o
s
e
,
 
t
r
a
i
n
_
l
o
w
,
 
m
a
_
p
e
r
i
o
d
,
 
m
a
_
t
y
p
e
,
 
s
l
o
p
e
_
l
b
,
 
s
w
i
n
g
_
l
b
)


 
 
 
 
 
 
 
 
p
f
 
=
 
v
b
t
.
P
o
r
t
f
o
l
i
o
.
f
r
o
m
_
s
i
g
n
a
l
s
(
c
l
o
s
e
=
t
r
a
i
n
_
c
l
o
s
e
,
 
e
n
t
r
i
e
s
=
e
n
t
r
i
e
s
,
 
e
x
i
t
s
=
e
x
i
t
s
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
i
n
i
t
_
c
a
s
h
=
1
0
0
_
0
0
0
,
 
f
e
e
s
=
0
.
0
0
0
5
,
 
s
l
i
p
p
a
g
e
=
0
.
0
0
0
5
,
 
f
r
e
q
=
'
D
'
)

 
 
 
 
 
 
 
 
t
r
a
d
e
s
 
=
 
p
f
.
t
r
a
d
e
s

 
 
 
 
 
 
 
 
t
o
t
a
l
_
t
r
a
d
e
s
 
=
 
l
e
n
(
t
r
a
d
e
s
)


 
 
 
 
 
 
 
 
i
f
 
t
o
t
a
l
_
t
r
a
d
e
s
 
<
 
1
0
:

 
 
 
 
 
 
 
 
 
 
 
 
s
k
i
p
p
e
d
_
l
o
w
_
t
r
a
d
e
s
 
+
=
 
1

 
 
 
 
 
 
 
 
 
 
 
 
c
o
n
t
i
n
u
e


 
 
 
 
 
 
 
 
t
r
a
d
e
s
_
p
e
r
_
y
e
a
r
 
=
 
t
o
t
a
l
_
t
r
a
d
e
s
 
/
 
y
e
a
r
s

 
 
 
 
 
 
 
 
t
o
t
a
l
_
r
e
t
u
r
n
 
=
 
f
l
o
a
t
(
p
f
.
t
o
t
a
l
_
r
e
t
u
r
n
(
)
)

 
 
 
 
 
 
 
 
a
n
n
u
a
l
i
z
e
d
_
r
e
t
u
r
n
 
=
 
f
l
o
a
t
(
p
f
.
a
n
n
u
a
l
i
z
e
d
_
r
e
t
u
r
n
(
f
r
e
q
=
'
D
'
)
)

 
 
 
 
 
 
 
 
m
a
x
_
d
r
a
w
d
o
w
n
 
=
 
f
l
o
a
t
(
p
f
.
m
a
x
_
d
r
a
w
d
o
w
n
(
)
)

 
 
 
 
 
 
 
 
v
o
l
a
t
i
l
i
t
y
 
=
 
f
l
o
a
t
(
p
f
.
a
n
n
u
a
l
i
z
e
d
_
v
o
l
a
t
i
l
i
t
y
(
f
r
e
q
=
'
D
'
)
)

 
 
 
 
 
 
 
 
s
h
a
r
p
e
_
r
a
t
i
o
 
=
 
f
l
o
a
t
(
p
f
.
s
h
a
r
p
e
_
r
a
t
i
o
(
f
r
e
q
=
'
D
'
)
)

 
 
 
 
 
 
 
 
s
o
r
t
i
n
o
_
r
a
t
i
o
 
=
 
f
l
o
a
t
(
p
f
.
s
o
r
t
i
n
o
_
r
a
t
i
o
(
f
r
e
q
=
'
D
'
)
)

 
 
 
 
 
 
 
 
c
a
l
m
a
r
_
r
a
t
i
o
 
=
 
a
n
n
u
a
l
i
z
e
d
_
r
e
t
u
r
n
 
/
 
a
b
s
(
m
a
x
_
d
r
a
w
d
o
w
n
)
 
i
f
 
a
b
s
(
m
a
x
_
d
r
a
w
d
o
w
n
)
 
>
 
1
e
-
9
 
e
l
s
e
 
n
p
.
n
a
n


 
 
 
 
 
 
 
 
w
i
n
_
r
a
t
e
_
p
c
t
 
=
 
n
p
.
n
a
n
;
 
p
r
o
f
i
t
_
f
a
c
t
o
r
 
=
 
n
p
.
n
a
n
;
 
e
x
p
e
c
t
a
n
c
y
 
=
 
0
.
0

 
 
 
 
 
 
 
 
a
v
g
_
w
i
n
_
a
m
o
u
n
t
 
=
 
0
.
0
;
 
a
v
g
_
l
o
s
s
_
a
m
o
u
n
t
 
=
 
0
.
0
;
 
l
a
r
g
e
s
t
_
w
i
n
 
=
 
0
.
0
;
 
l
a
r
g
e
s
t
_
l
o
s
s
 
=
 
0
.
0

 
 
 
 
 
 
 
 
w
i
n
n
i
n
g
_
s
t
r
e
a
k
 
=
 
n
p
.
n
a
n
;
 
l
o
s
i
n
g
_
s
t
r
e
a
k
 
=
 
n
p
.
n
a
n
;
 
a
v
g
_
t
r
a
d
e
_
d
u
r
a
t
i
o
n
 
=
 
n
p
.
n
a
n
;
 
s
q
n
 
=
 
n
p
.
n
a
n


 
 
 
 
 
 
 
 
t
r
 
=
 
t
r
a
d
e
s
.
r
e
t
u
r
n
s
.
v
a
l
u
e
s
 
i
f
 
h
a
s
a
t
t
r
(
t
r
a
d
e
s
.
r
e
t
u
r
n
s
,
 
'
v
a
l
u
e
s
'
)
 
e
l
s
e
 
n
p
.
a
r
r
a
y
(
t
r
a
d
e
s
.
r
e
t
u
r
n
s
)

 
 
 
 
 
 
 
 
i
f
 
t
r
.
s
i
z
e
 
>
 
0
:

 
 
 
 
 
 
 
 
 
 
 
 
p
o
s
 
=
 
t
r
[
t
r
 
>
 
0
]
;
 
n
e
g
 
=
 
t
r
[
t
r
 
<
 
0
]

 
 
 
 
 
 
 
 
 
 
 
 
w
i
n
_
r
a
t
e
_
p
c
t
 
=
 
(
l
e
n
(
p
o
s
)
 
/
 
l
e
n
(
t
r
)
)
 
*
 
1
0
0
.
0
 
i
f
 
l
e
n
(
t
r
)
 
>
 
0
 
e
l
s
e
 
n
p
.
n
a
n

 
 
 
 
 
 
 
 
 
 
 
 
g
a
i
n
s
 
=
 
p
o
s
.
s
u
m
(
)
 
i
f
 
l
e
n
(
p
o
s
)
 
e
l
s
e
 
0
.
0

 
 
 
 
 
 
 
 
 
 
 
 
l
o
s
s
e
s
 
=
 
a
b
s
(
n
e
g
.
s
u
m
(
)
)
 
i
f
 
l
e
n
(
n
e
g
)
 
e
l
s
e
 
0
.
0

 
 
 
 
 
 
 
 
 
 
 
 
p
r
o
f
i
t
_
f
a
c
t
o
r
 
=
 
g
a
i
n
s
 
/
 
l
o
s
s
e
s
 
i
f
 
l
o
s
s
e
s
 
>
 
0
 
e
l
s
e
 
n
p
.
i
n
f

 
 
 
 
 
 
 
 
 
 
 
 
e
x
p
e
c
t
a
n
c
y
 
=
 
f
l
o
a
t
(
t
r
.
m
e
a
n
(
)
)

 
 
 
 
 
 
 
 
 
 
 
 
a
v
g
_
w
i
n
_
a
m
o
u
n
t
 
=
 
f
l
o
a
t
(
p
o
s
.
m
e
a
n
(
)
)
 
i
f
 
l
e
n
(
p
o
s
)
 
e
l
s
e
 
0
.
0

 
 
 
 
 
 
 
 
 
 
 
 
a
v
g
_
l
o
s
s
_
a
m
o
u
n
t
 
=
 
f
l
o
a
t
(
a
b
s
(
n
e
g
.
m
e
a
n
(
)
)
)
 
i
f
 
l
e
n
(
n
e
g
)
 
e
l
s
e
 
0
.
0

 
 
 
 
 
 
 
 
 
 
 
 
l
a
r
g
e
s
t
_
w
i
n
 
=
 
f
l
o
a
t
(
p
o
s
.
m
a
x
(
)
)
 
i
f
 
l
e
n
(
p
o
s
)
 
e
l
s
e
 
0
.
0

 
 
 
 
 
 
 
 
 
 
 
 
l
a
r
g
e
s
t
_
l
o
s
s
 
=
 
f
l
o
a
t
(
n
e
g
.
m
i
n
(
)
)
 
i
f
 
l
e
n
(
n
e
g
)
 
e
l
s
e
 
0
.
0

 
 
 
 
 
 
 
 
 
 
 
 
s
q
n
 
=
 
f
l
o
a
t
(
t
r
.
m
e
a
n
(
)
 
/
 
t
r
.
s
t
d
(
)
 
*
 
n
p
.
s
q
r
t
(
l
e
n
(
t
r
)
)
)
 
i
f
 
t
r
.
s
t
d
(
)
 
>
 
0
 
e
l
s
e
 
n
p
.
n
a
n

 
 
 
 
 
 
 
 
 
 
 
 
t
r
y
:

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
w
i
n
n
i
n
g
_
s
t
r
e
a
k
 
=
 
i
n
t
(
t
r
a
d
e
s
.
w
i
n
n
i
n
g
_
s
t
r
e
a
k
(
)
)

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
l
o
s
i
n
g
_
s
t
r
e
a
k
 
=
 
i
n
t
(
t
r
a
d
e
s
.
l
o
s
i
n
g
_
s
t
r
e
a
k
(
)
)

 
 
 
 
 
 
 
 
 
 
 
 
e
x
c
e
p
t
:
 
p
a
s
s


 
 
 
 
 
 
 
 
r
e
t
u
r
n
s
 
=
 
p
f
.
r
e
t
u
r
n
s
(
)

 
 
 
 
 
 
 
 
c
u
m
 
=
 
(
1
 
+
 
r
e
t
u
r
n
s
)
.
c
u
m
p
r
o
d
(
)
;
 
p
e
a
k
 
=
 
c
u
m
.
c
u
m
m
a
x
(
)
;
 
d
d
 
=
 
(
c
u
m
 
-
 
p
e
a
k
)
 
/
 
p
e
a
k

 
 
 
 
 
 
 
 
u
l
c
e
r
_
i
n
d
e
x
 
=
 
f
l
o
a
t
(
n
p
.
s
q
r
t
(
(
d
d
.
p
o
w
(
2
)
)
.
m
e
a
n
(
)
)
)
 
i
f
 
l
e
n
(
d
d
)
 
>
 
0
 
e
l
s
e
 
n
p
.
n
a
n

 
 
 
 
 
 
 
 
p
a
y
o
f
f
_
r
a
t
i
o
 
=
 
(
a
v
g
_
w
i
n
_
a
m
o
u
n
t
 
/
 
a
v
g
_
l
o
s
s
_
a
m
o
u
n
t
)
 
i
f
 
a
v
g
_
l
o
s
s
_
a
m
o
u
n
t
 
>
 
0
 
e
l
s
e
 
n
p
.
i
n
f

 
 
 
 
 
 
 
 
r
e
t
s
 
=
 
r
e
t
u
r
n
s
.
v
a
l
u
e
s

 
 
 
 
 
 
 
 
o
m
e
g
a
_
r
a
t
i
o
 
=
 
f
l
o
a
t
(
r
e
t
s
[
r
e
t
s
 
>
 
0
]
.
s
u
m
(
)
 
/
 
a
b
s
(
r
e
t
s
[
r
e
t
s
 
<
 
0
]
.
s
u
m
(
)
)
)
 
i
f
 
a
b
s
(
r
e
t
s
[
r
e
t
s
 
<
 
0
]
.
s
u
m
(
)
)
 
>
 
0
 
e
l
s
e
 
n
p
.
i
n
f

 
 
 
 
 
 
 
 
r
e
c
o
v
e
r
y
_
f
a
c
t
o
r
 
=
 
t
o
t
a
l
_
r
e
t
u
r
n
 
/
 
a
b
s
(
m
a
x
_
d
r
a
w
d
o
w
n
)
 
i
f
 
a
b
s
(
m
a
x
_
d
r
a
w
d
o
w
n
)
 
>
 
1
e
-
9
 
e
l
s
e
 
n
p
.
n
a
n

 
 
 
 
 
 
 
 
g
a
i
n
_
t
o
_
p
a
i
n
 
=
 
f
l
o
a
t
(
r
e
t
s
.
s
u
m
(
)
 
/
 
a
b
s
(
r
e
t
s
[
r
e
t
s
 
<
 
0
]
.
s
u
m
(
)
)
)
 
i
f
 
a
b
s
(
r
e
t
s
[
r
e
t
s
 
<
 
0
]
.
s
u
m
(
)
)
 
>
 
0
 
e
l
s
e
 
n
p
.
i
n
f

 
 
 
 
 
 
 
 
s
e
r
e
n
i
t
y
_
i
n
d
e
x
 
=
 
r
e
c
o
v
e
r
y
_
f
a
c
t
o
r
 
/
 
u
l
c
e
r
_
i
n
d
e
x
 
i
f
 
u
l
c
e
r
_
i
n
d
e
x
 
a
n
d
 
u
l
c
e
r
_
i
n
d
e
x
 
>
 
0
 
e
l
s
e
 
n
p
.
n
a
n


 
 
 
 
 
 
 
 
g
r
i
d
_
s
e
a
r
c
h
_
r
e
s
u
l
t
s
.
a
p
p
e
n
d
(
{

 
 
 
 
 
 
 
 
 
 
 
 
"
m
a
_
p
e
r
i
o
d
"
:
 
m
a
_
p
e
r
i
o
d
,
 
"
m
a
_
t
y
p
e
"
:
 
m
a
_
t
y
p
e
,
 
"
s
l
o
p
e
_
l
o
o
k
b
a
c
k
"
:
 
s
l
o
p
e
_
l
b
,
 
"
s
w
i
n
g
_
l
o
o
k
b
a
c
k
"
:
 
s
w
i
n
g
_
l
b
,

 
 
 
 
 
 
 
 
 
 
 
 
"
t
o
t
a
l
_
r
e
t
u
r
n
"
:
 
t
o
t
a
l
_
r
e
t
u
r
n
,
 
"
a
n
n
u
a
l
i
z
e
d
_
r
e
t
u
r
n
"
:
 
a
n
n
u
a
l
i
z
e
d
_
r
e
t
u
r
n
,

 
 
 
 
 
 
 
 
 
 
 
 
"
s
h
a
r
p
e
_
r
a
t
i
o
"
:
 
s
h
a
r
p
e
_
r
a
t
i
o
,
 
"
s
o
r
t
i
n
o
_
r
a
t
i
o
"
:
 
s
o
r
t
i
n
o
_
r
a
t
i
o
,
 
"
c
a
l
m
a
r
_
r
a
t
i
o
"
:
 
c
a
l
m
a
r
_
r
a
t
i
o
,

 
 
 
 
 
 
 
 
 
 
 
 
"
o
m
e
g
a
_
r
a
t
i
o
"
:
 
o
m
e
g
a
_
r
a
t
i
o
,
 
"
i
n
f
o
r
m
a
t
i
o
n
_
r
a
t
i
o
"
:
 
n
p
.
n
a
n
,
 
"
t
a
i
l
_
r
a
t
i
o
"
:
 
n
p
.
n
a
n
,

 
 
 
 
 
 
 
 
 
 
 
 
"
d
e
f
l
a
t
e
d
_
s
h
a
r
p
e
_
r
a
t
i
o
"
:
 
n
p
.
n
a
n
,
 
"
m
a
x
_
d
r
a
w
d
o
w
n
"
:
 
m
a
x
_
d
r
a
w
d
o
w
n
,
 
"
v
o
l
a
t
i
l
i
t
y
"
:
 
v
o
l
a
t
i
l
i
t
y
,

 
 
 
 
 
 
 
 
 
 
 
 
"
u
l
c
e
r
_
i
n
d
e
x
"
:
 
u
l
c
e
r
_
i
n
d
e
x
,
 
"
w
i
n
_
r
a
t
e
"
:
 
w
i
n
_
r
a
t
e
_
p
c
t
,
 
"
t
o
t
a
l
_
t
r
a
d
e
s
"
:
 
t
o
t
a
l
_
t
r
a
d
e
s
,

 
 
 
 
 
 
 
 
 
 
 
 
"
a
v
g
_
t
r
a
d
e
_
d
u
r
a
t
i
o
n
"
:
 
a
v
g
_
t
r
a
d
e
_
d
u
r
a
t
i
o
n
,
 
"
e
x
p
e
c
t
a
n
c
y
"
:
 
e
x
p
e
c
t
a
n
c
y
,

 
 
 
 
 
 
 
 
 
 
 
 
"
p
r
o
f
i
t
_
f
a
c
t
o
r
"
:
 
p
r
o
f
i
t
_
f
a
c
t
o
r
,
 
"
s
q
n
"
:
 
s
q
n
,
 
"
p
a
y
o
f
f
_
r
a
t
i
o
"
:
 
p
a
y
o
f
f
_
r
a
t
i
o
,

 
 
 
 
 
 
 
 
 
 
 
 
"
l
a
r
g
e
s
t
_
w
i
n
"
:
 
l
a
r
g
e
s
t
_
w
i
n
,
 
"
l
a
r
g
e
s
t
_
l
o
s
s
"
:
 
l
a
r
g
e
s
t
_
l
o
s
s
,

 
 
 
 
 
 
 
 
 
 
 
 
"
a
v
g
_
w
i
n
_
a
m
o
u
n
t
"
:
 
a
v
g
_
w
i
n
_
a
m
o
u
n
t
,
 
"
a
v
g
_
l
o
s
s
_
a
m
o
u
n
t
"
:
 
a
v
g
_
l
o
s
s
_
a
m
o
u
n
t
,

 
 
 
 
 
 
 
 
 
 
 
 
"
w
i
n
n
i
n
g
_
s
t
r
e
a
k
"
:
 
w
i
n
n
i
n
g
_
s
t
r
e
a
k
,
 
"
l
o
s
i
n
g
_
s
t
r
e
a
k
"
:
 
l
o
s
i
n
g
_
s
t
r
e
a
k
,

 
 
 
 
 
 
 
 
 
 
 
 
"
r
e
c
o
v
e
r
y
_
f
a
c
t
o
r
"
:
 
r
e
c
o
v
e
r
y
_
f
a
c
t
o
r
,
 
"
g
a
i
n
_
t
o
_
p
a
i
n
_
r
a
t
i
o
"
:
 
g
a
i
n
_
t
o
_
p
a
i
n
,

 
 
 
 
 
 
 
 
 
 
 
 
"
s
e
r
e
n
i
t
y
_
i
n
d
e
x
"
:
 
s
e
r
e
n
i
t
y
_
i
n
d
e
x
,
 
"
t
r
a
d
e
s
_
p
e
r
_
y
e
a
r
"
:
 
t
r
a
d
e
s
_
p
e
r
_
y
e
a
r

 
 
 
 
 
 
 
 
}
)

 
 
 
 
 
 
 
 
s
u
c
c
e
s
s
f
u
l
_
t
e
s
t
s
 
+
=
 
1

 
 
 
 
e
x
c
e
p
t
 
E
x
c
e
p
t
i
o
n
 
a
s
 
e
:

 
 
 
 
 
 
 
 
f
a
i
l
e
d
_
t
e
s
t
s
 
+
=
 
1


 
 
 
 
i
f
 
c
o
m
b
o
_
n
u
m
 
%
 
1
0
0
 
=
=
 
0
:

 
 
 
 
 
 
 
 
p
r
o
g
r
e
s
s
_
p
c
t
 
=
 
(
c
o
m
b
o
_
n
u
m
 
/
 
t
o
t
a
l
_
c
o
m
b
i
n
a
t
i
o
n
s
)
 
*
 
1
0
0

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
\
U
0
0
0
1
f
5
0
4
 
P
r
o
g
r
e
s
s
:
 
{
c
o
m
b
o
_
n
u
m
}
/
{
t
o
t
a
l
_
c
o
m
b
i
n
a
t
i
o
n
s
}
 
(
{
p
r
o
g
r
e
s
s
_
p
c
t
:
.
1
f
}
%
)
 
|
 
\
u
2
7
1
4
 
{
s
u
c
c
e
s
s
f
u
l
_
t
e
s
t
s
}
 
s
u
c
c
e
s
s
f
u
l
 
|
 
S
k
i
p
p
e
d
:
 
{
s
k
i
p
p
e
d
_
l
o
w
_
t
r
a
d
e
s
}
"
)


p
r
i
n
t
(
"
\
n
"
 
+
 
"
=
"
 
*
 
7
0
)

p
r
i
n
t
(
"
L
O
S
T
 
M
O
M
E
N
T
U
M
 
G
R
I
D
 
S
E
A
R
C
H
 
C
O
M
P
L
E
T
E
D
!
"
)

p
r
i
n
t
(
"
=
"
 
*
 
7
0
)

p
r
i
n
t
(
f
"
T
o
t
a
l
 
c
o
m
b
i
n
a
t
i
o
n
s
 
a
t
t
e
m
p
t
e
d
:
 
{
t
o
t
a
l
_
c
o
m
b
i
n
a
t
i
o
n
s
}
"
)

p
r
i
n
t
(
f
"
S
u
c
c
e
s
s
f
u
l
l
y
 
c
o
m
p
l
e
t
e
d
:
 
{
s
u
c
c
e
s
s
f
u
l
_
t
e
s
t
s
}
"
)

p
r
i
n
t
(
f
"
S
k
i
p
p
e
d
 
(
<
 
1
0
 
t
r
a
d
e
s
)
:
 
{
s
k
i
p
p
e
d
_
l
o
w
_
t
r
a
d
e
s
}
"
)

p
r
i
n
t
(
f
"
F
a
i
l
e
d
:
 
{
f
a
i
l
e
d
_
t
e
s
t
s
}
"
)

p
r
i
n
t
(
f
"
S
u
c
c
e
s
s
 
r
a
t
e
:
 
{
(
s
u
c
c
e
s
s
f
u
l
_
t
e
s
t
s
/
t
o
t
a
l
_
c
o
m
b
i
n
a
t
i
o
n
s
)
*
1
0
0
:
.
1
f
}
%
"
)

p
r
i
n
t
(
f
"
\
n
\
u
2
7
1
4
 
R
e
s
u
l
t
s
 
s
t
o
r
e
d
 
i
n
 
'
g
r
i
d
_
s
e
a
r
c
h
_
r
e
s
u
l
t
s
'
 
(
{
l
e
n
(
g
r
i
d
_
s
e
a
r
c
h
_
r
e
s
u
l
t
s
)
}
 
e
n
t
r
i
e
s
)
"
)


i
f
 
s
u
c
c
e
s
s
f
u
l
_
t
e
s
t
s
 
>
 
0
:

 
 
 
 
r
e
s
u
l
t
s
_
d
f
 
=
 
p
d
.
D
a
t
a
F
r
a
m
e
(
g
r
i
d
_
s
e
a
r
c
h
_
r
e
s
u
l
t
s
)

 
 
 
 
p
r
i
n
t
(
"
\
n
"
 
+
 
"
=
"
 
*
 
7
0
)

 
 
 
 
p
r
i
n
t
(
"
\
U
0
0
0
1
f
3
c
6
 
T
O
P
 
1
0
 
C
O
M
B
I
N
A
T
I
O
N
S
 
(
b
y
 
I
n
-
S
a
m
p
l
e
 
S
h
a
r
p
e
 
R
a
t
i
o
)
"
)

 
 
 
 
p
r
i
n
t
(
"
=
"
 
*
 
7
0
)

 
 
 
 
t
o
p
_
1
0
 
=
 
r
e
s
u
l
t
s
_
d
f
.
n
l
a
r
g
e
s
t
(
1
0
,
 
'
s
h
a
r
p
e
_
r
a
t
i
o
'
)

 
 
 
 
f
o
r
 
r
a
n
k
,
 
(
i
d
x
,
 
r
o
w
)
 
i
n
 
e
n
u
m
e
r
a
t
e
(
t
o
p
_
1
0
.
i
t
e
r
r
o
w
s
(
)
,
 
1
)
:

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
\
n
#
{
r
a
n
k
}
 
-
 
{
r
o
w
[
'
m
a
_
t
y
p
e
'
]
}
(
{
i
n
t
(
r
o
w
[
'
m
a
_
p
e
r
i
o
d
'
]
)
}
)
,
 
s
l
o
p
e
_
l
b
=
{
i
n
t
(
r
o
w
[
'
s
l
o
p
e
_
l
o
o
k
b
a
c
k
'
]
)
}
,
 
s
w
i
n
g
_
l
b
=
{
i
n
t
(
r
o
w
[
'
s
w
i
n
g
_
l
o
o
k
b
a
c
k
'
]
)
}
"
)

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
 
 
 
S
h
a
r
p
e
 
R
a
t
i
o
:
 
 
 
 
 
 
{
r
o
w
[
'
s
h
a
r
p
e
_
r
a
t
i
o
'
]
:
.
3
f
}
"
)

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
 
 
 
T
o
t
a
l
 
R
e
t
u
r
n
:
 
 
 
 
 
 
{
r
o
w
[
'
t
o
t
a
l
_
r
e
t
u
r
n
'
]
:
.
2
%
}
"
)

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
 
 
 
A
n
n
u
a
l
i
z
e
d
 
R
e
t
u
r
n
:
 
{
r
o
w
[
'
a
n
n
u
a
l
i
z
e
d
_
r
e
t
u
r
n
'
]
:
.
2
%
}
"
)

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
 
 
 
M
a
x
 
D
r
a
w
d
o
w
n
:
 
 
 
 
 
 
{
r
o
w
[
'
m
a
x
_
d
r
a
w
d
o
w
n
'
]
:
.
2
%
}
"
)

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
 
 
 
W
i
n
 
R
a
t
e
:
 
 
 
 
 
 
 
 
 
 
{
r
o
w
[
'
w
i
n
_
r
a
t
e
'
]
:
.
1
f
}
%
"
)

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
 
 
 
P
r
o
f
i
t
 
F
a
c
t
o
r
:
 
 
 
 
 
{
r
o
w
[
'
p
r
o
f
i
t
_
f
a
c
t
o
r
'
]
:
.
2
f
}
"
)

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
 
 
 
T
o
t
a
l
 
T
r
a
d
e
s
:
 
 
 
 
 
 
{
i
n
t
(
r
o
w
[
'
t
o
t
a
l
_
t
r
a
d
e
s
'
]
)
}
 
(
{
r
o
w
[
'
t
r
a
d
e
s
_
p
e
r
_
y
e
a
r
'
]
:
.
1
f
}
/
y
e
a
r
)
"
)

 
 
 
 
p
r
i
n
t
(
"
\
n
"
 
+
 
"
=
"
 
*
 
7
0
)

In [ ]:
#
 
T
O
P
 
5
 
O
U
T
-
O
F
-
S
A
M
P
L
E
 
V
A
L
I
D
A
T
I
O
N
 
&
 
C
O
M
P
A
R
I
S
O
N
 
T
A
B
L
E


F
R
E
Q
 
=
 
"
1
D
"

r
e
s
u
l
t
s
_
d
f
 
=
 
p
d
.
D
a
t
a
F
r
a
m
e
(
g
r
i
d
_
s
e
a
r
c
h
_
r
e
s
u
l
t
s
)


i
f
 
r
e
s
u
l
t
s
_
d
f
.
e
m
p
t
y
:

 
 
 
 
p
r
i
n
t
(
"
N
o
 
r
e
s
u
l
t
s
 
t
o
 
v
a
l
i
d
a
t
e
.
"
)

e
l
s
e
:

 
 
 
 
p
r
i
n
t
(
"
=
"
 
*
 
9
0
)

 
 
 
 
p
r
i
n
t
(
"
\
U
0
0
0
1
f
3
c
6
 
T
O
P
 
5
 
S
T
R
A
T
E
G
I
E
S
 
-
 
O
U
T
-
O
F
-
S
A
M
P
L
E
 
V
A
L
I
D
A
T
I
O
N
"
)

 
 
 
 
p
r
i
n
t
(
"
=
"
 
*
 
9
0
)

 
 
 
 
p
r
i
n
t
(
f
"
T
r
a
i
n
i
n
g
 
P
e
r
i
o
d
:
 
{
t
r
a
i
n
_
c
l
o
s
e
.
i
n
d
e
x
[
0
]
.
d
a
t
e
(
)
}
 
-
>
 
{
t
r
a
i
n
_
c
l
o
s
e
.
i
n
d
e
x
[
-
1
]
.
d
a
t
e
(
)
}
"
)

 
 
 
 
p
r
i
n
t
(
f
"
V
a
l
i
d
a
t
i
o
n
 
P
e
r
i
o
d
:
 
{
v
a
l
_
c
l
o
s
e
.
i
n
d
e
x
[
0
]
.
d
a
t
e
(
)
}
 
-
>
 
{
v
a
l
_
c
l
o
s
e
.
i
n
d
e
x
[
-
1
]
.
d
a
t
e
(
)
}
"
)

 
 
 
 
p
r
i
n
t
(
"
=
"
 
*
 
9
0
)


 
 
 
 
t
o
p
_
5
 
=
 
r
e
s
u
l
t
s
_
d
f
.
n
l
a
r
g
e
s
t
(
5
,
 
'
s
h
a
r
p
e
_
r
a
t
i
o
'
)

 
 
 
 
o
o
s
_
r
e
s
u
l
t
s
 
=
 
[
]


 
 
 
 
f
o
r
 
_
,
 
s
t
r
a
t
e
g
y
 
i
n
 
t
o
p
_
5
.
i
t
e
r
r
o
w
s
(
)
:

 
 
 
 
 
 
 
 
m
p
 
=
 
i
n
t
(
s
t
r
a
t
e
g
y
[
'
m
a
_
p
e
r
i
o
d
'
]
)
;
 
m
t
 
=
 
s
t
r
a
t
e
g
y
[
'
m
a
_
t
y
p
e
'
]

 
 
 
 
 
 
 
 
s
l
 
=
 
i
n
t
(
s
t
r
a
t
e
g
y
[
'
s
l
o
p
e
_
l
o
o
k
b
a
c
k
'
]
)
;
 
s
w
 
=
 
i
n
t
(
s
t
r
a
t
e
g
y
[
'
s
w
i
n
g
_
l
o
o
k
b
a
c
k
'
]
)


 
 
 
 
 
 
 
 
e
,
 
x
 
=
 
_
l
o
s
t
_
m
o
m
e
n
t
u
m
_
s
i
g
n
a
l
s
(
v
a
l
_
c
l
o
s
e
,
 
v
a
l
_
l
o
w
,
 
m
p
,
 
m
t
,
 
s
l
,
 
s
w
)

 
 
 
 
 
 
 
 
p
f
_
v
a
l
 
=
 
v
b
t
.
P
o
r
t
f
o
l
i
o
.
f
r
o
m
_
s
i
g
n
a
l
s
(
c
l
o
s
e
=
v
a
l
_
c
l
o
s
e
,
 
e
n
t
r
i
e
s
=
e
,
 
e
x
i
t
s
=
x
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
i
n
i
t
_
c
a
s
h
=
1
0
0
_
0
0
0
,
 
f
e
e
s
=
0
.
0
0
0
5
,
 
s
l
i
p
p
a
g
e
=
0
.
0
0
0
5
,
 
f
r
e
q
=
F
R
E
Q
)

 
 
 
 
 
 
 
 
o
o
s
_
s
h
a
r
p
e
 
=
 
f
l
o
a
t
(
p
f
_
v
a
l
.
s
h
a
r
p
e
_
r
a
t
i
o
(
f
r
e
q
=
F
R
E
Q
)
)

 
 
 
 
 
 
 
 
o
o
s
_
r
e
t
 
=
 
f
l
o
a
t
(
p
f
_
v
a
l
.
t
o
t
a
l
_
r
e
t
u
r
n
(
)
)

 
 
 
 
 
 
 
 
o
o
s
_
d
d
 
=
 
f
l
o
a
t
(
p
f
_
v
a
l
.
m
a
x
_
d
r
a
w
d
o
w
n
(
)
)

 
 
 
 
 
 
 
 
t
r
a
d
e
s
 
=
 
p
f
_
v
a
l
.
t
r
a
d
e
s
;
 
o
o
s
_
t
r
a
d
e
s
 
=
 
l
e
n
(
t
r
a
d
e
s
)

 
 
 
 
 
 
 
 
o
o
s
_
w
r
 
=
 
n
p
.
n
a
n
;
 
o
o
s
_
p
f
 
=
 
n
p
.
n
a
n

 
 
 
 
 
 
 
 
i
f
 
o
o
s
_
t
r
a
d
e
s
 
>
 
0
:

 
 
 
 
 
 
 
 
 
 
 
 
t
r
 
=
 
t
r
a
d
e
s
.
r
e
t
u
r
n
s
.
v
a
l
u
e
s
 
i
f
 
h
a
s
a
t
t
r
(
t
r
a
d
e
s
.
r
e
t
u
r
n
s
,
 
'
v
a
l
u
e
s
'
)
 
e
l
s
e
 
n
p
.
a
r
r
a
y
(
t
r
a
d
e
s
.
r
e
t
u
r
n
s
)

 
 
 
 
 
 
 
 
 
 
 
 
i
f
 
t
r
.
s
i
z
e
 
>
 
0
:

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
p
 
=
 
t
r
[
t
r
 
>
 
0
]
;
 
n
 
=
 
t
r
[
t
r
 
<
 
0
]

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
o
o
s
_
w
r
 
=
 
(
l
e
n
(
p
)
/
l
e
n
(
t
r
)
)
*
1
0
0
 
i
f
 
l
e
n
(
t
r
)
 
>
 
0
 
e
l
s
e
 
n
p
.
n
a
n

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
o
o
s
_
p
f
 
=
 
p
.
s
u
m
(
)
/
a
b
s
(
n
.
s
u
m
(
)
)
 
i
f
 
l
e
n
(
n
)
 
>
 
0
 
a
n
d
 
a
b
s
(
n
.
s
u
m
(
)
)
 
>
 
0
 
e
l
s
e
 
n
p
.
i
n
f


 
 
 
 
 
 
 
 
o
o
s
_
r
e
s
u
l
t
s
.
a
p
p
e
n
d
(
{

 
 
 
 
 
 
 
 
 
 
 
 
'
R
a
n
k
'
:
 
l
e
n
(
o
o
s
_
r
e
s
u
l
t
s
)
+
1
,
 
'
M
A
_
P
e
r
i
o
d
'
:
 
m
p
,
 
'
M
A
_
T
y
p
e
'
:
 
m
t
,

 
 
 
 
 
 
 
 
 
 
 
 
'
S
l
o
p
e
_
L
B
'
:
 
s
l
,
 
'
S
w
i
n
g
_
L
B
'
:
 
s
w
,

 
 
 
 
 
 
 
 
 
 
 
 
'
I
S
_
S
h
a
r
p
e
'
:
 
f
l
o
a
t
(
s
t
r
a
t
e
g
y
[
'
s
h
a
r
p
e
_
r
a
t
i
o
'
]
)
,
 
'
I
S
_
R
e
t
u
r
n
'
:
 
f
l
o
a
t
(
s
t
r
a
t
e
g
y
[
'
t
o
t
a
l
_
r
e
t
u
r
n
'
]
)
,

 
 
 
 
 
 
 
 
 
 
 
 
'
I
S
_
M
a
x
D
D
'
:
 
f
l
o
a
t
(
s
t
r
a
t
e
g
y
[
'
m
a
x
_
d
r
a
w
d
o
w
n
'
]
)
,
 
'
I
S
_
W
i
n
R
a
t
e
'
:
 
f
l
o
a
t
(
s
t
r
a
t
e
g
y
[
'
w
i
n
_
r
a
t
e
'
]
)
,

 
 
 
 
 
 
 
 
 
 
 
 
'
O
O
S
_
S
h
a
r
p
e
'
:
 
o
o
s
_
s
h
a
r
p
e
,
 
'
O
O
S
_
R
e
t
u
r
n
'
:
 
o
o
s
_
r
e
t
,
 
'
O
O
S
_
M
a
x
D
D
'
:
 
o
o
s
_
d
d
,

 
 
 
 
 
 
 
 
 
 
 
 
'
O
O
S
_
W
i
n
R
a
t
e
'
:
 
o
o
s
_
w
r
,
 
'
O
O
S
_
T
r
a
d
e
s
'
:
 
o
o
s
_
t
r
a
d
e
s
,
 
'
O
O
S
_
P
r
o
f
i
t
F
a
c
t
o
r
'
:
 
o
o
s
_
p
f
,

 
 
 
 
 
 
 
 
 
 
 
 
'
S
h
a
r
p
e
_
D
e
g
r
a
d
a
t
i
o
n
'
:
 
(
(
o
o
s
_
s
h
a
r
p
e
 
-
 
s
t
r
a
t
e
g
y
[
'
s
h
a
r
p
e
_
r
a
t
i
o
'
]
)
/
a
b
s
(
s
t
r
a
t
e
g
y
[
'
s
h
a
r
p
e
_
r
a
t
i
o
'
]
)
*
1
0
0
)
 
i
f
 
s
t
r
a
t
e
g
y
[
'
s
h
a
r
p
e
_
r
a
t
i
o
'
]
 
!
=
 
0
 
e
l
s
e
 
n
p
.
n
a
n
,

 
 
 
 
 
 
 
 
 
 
 
 
'
R
e
t
u
r
n
_
D
e
g
r
a
d
a
t
i
o
n
'
:
 
(
(
o
o
s
_
r
e
t
 
-
 
s
t
r
a
t
e
g
y
[
'
t
o
t
a
l
_
r
e
t
u
r
n
'
]
)
/
a
b
s
(
s
t
r
a
t
e
g
y
[
'
t
o
t
a
l
_
r
e
t
u
r
n
'
]
)
*
1
0
0
)
 
i
f
 
s
t
r
a
t
e
g
y
[
'
t
o
t
a
l
_
r
e
t
u
r
n
'
]
 
!
=
 
0
 
e
l
s
e
 
n
p
.
n
a
n

 
 
 
 
 
 
 
 
}
)


 
 
 
 
o
o
s
_
d
f
 
=
 
p
d
.
D
a
t
a
F
r
a
m
e
(
o
o
s
_
r
e
s
u
l
t
s
)
.
s
o
r
t
_
v
a
l
u
e
s
(
'
O
O
S
_
S
h
a
r
p
e
'
,
 
a
s
c
e
n
d
i
n
g
=
F
a
l
s
e
)
.
r
e
s
e
t
_
i
n
d
e
x
(
d
r
o
p
=
T
r
u
e
)

 
 
 
 
o
o
s
_
d
f
[
'
O
O
S
_
R
a
n
k
'
]
 
=
 
r
a
n
g
e
(
1
,
 
l
e
n
(
o
o
s
_
d
f
)
+
1
)

 
 
 
 
p
r
i
n
t
(
"
\
n
\
U
0
0
0
1
f
4
c
a
 
C
O
M
P
R
E
H
E
N
S
I
V
E
 
C
O
M
P
A
R
I
S
O
N
 
T
A
B
L
E
 
(
S
o
r
t
e
d
 
b
y
 
O
O
S
 
S
h
a
r
p
e
)
"
)

 
 
 
 
p
r
i
n
t
(
"
=
"
 
*
 
9
0
)

 
 
 
 
d
i
s
p
l
a
y
_
d
f
 
=
 
p
d
.
D
a
t
a
F
r
a
m
e
(
{

 
 
 
 
 
 
 
 
'
I
S
-
>
O
O
S
 
R
a
n
k
'
:
 
o
o
s
_
d
f
[
'
R
a
n
k
'
]
.
a
s
t
y
p
e
(
s
t
r
)
+
'
-
>
'
+
o
o
s
_
d
f
[
'
O
O
S
_
R
a
n
k
'
]
.
a
s
t
y
p
e
(
s
t
r
)
,

 
 
 
 
 
 
 
 
'
S
t
r
a
t
e
g
y
'
:
 
o
o
s
_
d
f
.
a
p
p
l
y
(
l
a
m
b
d
a
 
x
:
 
f
"
{
x
[
'
M
A
_
T
y
p
e
'
]
}
(
{
i
n
t
(
x
[
'
M
A
_
P
e
r
i
o
d
'
]
)
}
)
,
s
l
=
{
i
n
t
(
x
[
'
S
l
o
p
e
_
L
B
'
]
)
}
,
s
w
=
{
i
n
t
(
x
[
'
S
w
i
n
g
_
L
B
'
]
)
}
"
,
 
a
x
i
s
=
1
)
,

 
 
 
 
 
 
 
 
'
I
S
 
S
h
a
r
p
e
'
:
 
o
o
s
_
d
f
[
'
I
S
_
S
h
a
r
p
e
'
]
.
m
a
p
(
'
{
:
.
3
f
}
'
.
f
o
r
m
a
t
)
,

 
 
 
 
 
 
 
 
'
O
O
S
 
S
h
a
r
p
e
'
:
 
o
o
s
_
d
f
[
'
O
O
S
_
S
h
a
r
p
e
'
]
.
m
a
p
(
'
{
:
.
3
f
}
'
.
f
o
r
m
a
t
)
,

 
 
 
 
 
 
 
 
'
S
h
a
r
p
e
 
D
%
'
:
 
o
o
s
_
d
f
[
'
S
h
a
r
p
e
_
D
e
g
r
a
d
a
t
i
o
n
'
]
.
m
a
p
(
'
{
:
+
.
1
f
}
%
'
.
f
o
r
m
a
t
)
,

 
 
 
 
 
 
 
 
'
I
S
 
R
e
t
u
r
n
'
:
 
o
o
s
_
d
f
[
'
I
S
_
R
e
t
u
r
n
'
]
.
m
a
p
(
'
{
:
.
1
%
}
'
.
f
o
r
m
a
t
)
,

 
 
 
 
 
 
 
 
'
O
O
S
 
R
e
t
u
r
n
'
:
 
o
o
s
_
d
f
[
'
O
O
S
_
R
e
t
u
r
n
'
]
.
m
a
p
(
'
{
:
.
1
%
}
'
.
f
o
r
m
a
t
)
,

 
 
 
 
 
 
 
 
'
R
e
t
u
r
n
 
D
%
'
:
 
o
o
s
_
d
f
[
'
R
e
t
u
r
n
_
D
e
g
r
a
d
a
t
i
o
n
'
]
.
m
a
p
(
'
{
:
+
.
1
f
}
%
'
.
f
o
r
m
a
t
)
,

 
 
 
 
 
 
 
 
'
O
O
S
 
T
r
a
d
e
s
'
:
 
o
o
s
_
d
f
[
'
O
O
S
_
T
r
a
d
e
s
'
]
.
a
s
t
y
p
e
(
i
n
t
)
,

 
 
 
 
 
 
 
 
'
O
O
S
 
W
i
n
R
a
t
e
'
:
 
o
o
s
_
d
f
[
'
O
O
S
_
W
i
n
R
a
t
e
'
]
.
m
a
p
(
'
{
:
.
1
f
}
%
'
.
f
o
r
m
a
t
)
,

 
 
 
 
 
 
 
 
'
O
O
S
 
P
F
'
:
 
o
o
s
_
d
f
[
'
O
O
S
_
P
r
o
f
i
t
F
a
c
t
o
r
'
]
.
m
a
p
(
'
{
:
.
2
f
}
'
.
f
o
r
m
a
t
)

 
 
 
 
}
)

 
 
 
 
p
r
i
n
t
(
d
i
s
p
l
a
y
_
d
f
.
t
o
_
s
t
r
i
n
g
(
i
n
d
e
x
=
F
a
l
s
e
)
)

 
 
 
 
b
o
 
=
 
o
o
s
_
d
f
.
i
l
o
c
[
0
]

 
 
 
 
p
r
i
n
t
(
"
\
n
"
+
"
=
"
*
9
0
)

 
 
 
 
p
r
i
n
t
(
f
"
\
u
2
7
2
8
 
B
E
S
T
 
O
U
T
-
O
F
-
S
A
M
P
L
E
 
P
E
R
F
O
R
M
E
R
"
)

 
 
 
 
p
r
i
n
t
(
"
=
"
*
9
0
)

 
 
 
 
p
r
i
n
t
(
f
"
S
t
r
a
t
e
g
y
:
 
{
b
o
[
'
M
A
_
T
y
p
e
'
]
}
(
m
a
=
{
i
n
t
(
b
o
[
'
M
A
_
P
e
r
i
o
d
'
]
)
}
,
 
s
l
o
p
e
_
l
b
=
{
i
n
t
(
b
o
[
'
S
l
o
p
e
_
L
B
'
]
)
}
,
 
s
w
i
n
g
_
l
b
=
{
i
n
t
(
b
o
[
'
S
w
i
n
g
_
L
B
'
]
)
}
)
"
)

 
 
 
 
p
r
i
n
t
(
f
"
I
n
-
S
a
m
p
l
e
 
R
a
n
k
:
 
 
 
 
 
 
 
 
#
{
i
n
t
(
b
o
[
'
R
a
n
k
'
]
)
}
"
)

 
 
 
 
p
r
i
n
t
(
f
"
O
u
t
-
o
f
-
S
a
m
p
l
e
 
R
a
n
k
:
 
 
 
 
#
1
"
)

 
 
 
 
p
r
i
n
t
(
f
"
O
O
S
 
S
h
a
r
p
e
 
R
a
t
i
o
:
 
 
 
 
 
 
{
b
o
[
'
O
O
S
_
S
h
a
r
p
e
'
]
:
.
3
f
}
"
)

 
 
 
 
p
r
i
n
t
(
f
"
O
O
S
 
R
e
t
u
r
n
:
 
 
 
 
 
 
 
 
 
 
 
 
{
b
o
[
'
O
O
S
_
R
e
t
u
r
n
'
]
:
.
2
%
}
"
)

 
 
 
 
p
r
i
n
t
(
f
"
O
O
S
 
M
a
x
 
D
r
a
w
d
o
w
n
:
 
 
 
 
 
 
{
b
o
[
'
O
O
S
_
M
a
x
D
D
'
]
:
.
2
%
}
"
)

 
 
 
 
p
r
i
n
t
(
f
"
O
O
S
 
W
i
n
 
R
a
t
e
:
 
 
 
 
 
 
 
 
 
 
{
b
o
[
'
O
O
S
_
W
i
n
R
a
t
e
'
]
:
.
1
f
}
%
"
)

 
 
 
 
p
r
i
n
t
(
f
"
O
O
S
 
P
r
o
f
i
t
 
F
a
c
t
o
r
:
 
 
 
 
 
{
b
o
[
'
O
O
S
_
P
r
o
f
i
t
F
a
c
t
o
r
'
]
:
.
2
f
}
"
)

 
 
 
 
p
r
i
n
t
(
f
"
O
O
S
 
T
o
t
a
l
 
T
r
a
d
e
s
:
 
 
 
 
 
 
{
i
n
t
(
b
o
[
'
O
O
S
_
T
r
a
d
e
s
'
]
)
}
"
)

 
 
 
 
p
r
i
n
t
(
f
"
S
h
a
r
p
e
 
D
e
g
r
a
d
a
t
i
o
n
:
 
 
 
 
{
b
o
[
'
S
h
a
r
p
e
_
D
e
g
r
a
d
a
t
i
o
n
'
]
:
+
.
1
f
}
%
"
)

 
 
 
 
p
r
i
n
t
(
"
=
"
*
9
0
)


 
 
 
 
#
 
E
q
u
i
t
y
 
c
u
r
v
e
s
 
f
o
r
 
b
e
s
t
 
I
S
 
p
e
r
f
o
r
m
e
r

 
 
 
 
b
i
 
=
 
r
e
s
u
l
t
s
_
d
f
.
l
o
c
[
r
e
s
u
l
t
s
_
d
f
[
'
s
h
a
r
p
e
_
r
a
t
i
o
'
]
.
i
d
x
m
a
x
(
)
]

 
 
 
 
b
_
m
p
 
=
 
i
n
t
(
b
i
[
'
m
a
_
p
e
r
i
o
d
'
]
)
;
 
b
_
m
t
 
=
 
b
i
[
'
m
a
_
t
y
p
e
'
]

 
 
 
 
b
_
s
l
 
=
 
i
n
t
(
b
i
[
'
s
l
o
p
e
_
l
o
o
k
b
a
c
k
'
]
)
;
 
b
_
s
w
 
=
 
i
n
t
(
b
i
[
'
s
w
i
n
g
_
l
o
o
k
b
a
c
k
'
]
)


 
 
 
 
e
t
,
 
x
t
 
=
 
_
l
o
s
t
_
m
o
m
e
n
t
u
m
_
s
i
g
n
a
l
s
(
t
r
a
i
n
_
c
l
o
s
e
,
 
t
r
a
i
n
_
l
o
w
,
 
b
_
m
p
,
 
b
_
m
t
,
 
b
_
s
l
,
 
b
_
s
w
)

 
 
 
 
p
f
t
 
=
 
v
b
t
.
P
o
r
t
f
o
l
i
o
.
f
r
o
m
_
s
i
g
n
a
l
s
(
c
l
o
s
e
=
t
r
a
i
n
_
c
l
o
s
e
,
 
e
n
t
r
i
e
s
=
e
t
,
 
e
x
i
t
s
=
x
t
,
 
i
n
i
t
_
c
a
s
h
=
1
0
0
_
0
0
0
,
 
f
e
e
s
=
0
.
0
0
0
5
,
 
s
l
i
p
p
a
g
e
=
0
.
0
0
0
5
,
 
f
r
e
q
=
'
D
'
)

 
 
 
 
e
v
,
 
x
v
 
=
 
_
l
o
s
t
_
m
o
m
e
n
t
u
m
_
s
i
g
n
a
l
s
(
v
a
l
_
c
l
o
s
e
,
 
v
a
l
_
l
o
w
,
 
b
_
m
p
,
 
b
_
m
t
,
 
b
_
s
l
,
 
b
_
s
w
)

 
 
 
 
p
f
v
 
=
 
v
b
t
.
P
o
r
t
f
o
l
i
o
.
f
r
o
m
_
s
i
g
n
a
l
s
(
c
l
o
s
e
=
v
a
l
_
c
l
o
s
e
,
 
e
n
t
r
i
e
s
=
e
v
,
 
e
x
i
t
s
=
x
v
,
 
i
n
i
t
_
c
a
s
h
=
1
0
0
_
0
0
0
,
 
f
e
e
s
=
0
.
0
0
0
5
,
 
s
l
i
p
p
a
g
e
=
0
.
0
0
0
5
,
 
f
r
e
q
=
'
D
'
)


 
 
 
 
f
i
g
,
 
a
x
 
=
 
p
l
t
.
s
u
b
p
l
o
t
s
(
f
i
g
s
i
z
e
=
(
1
4
,
 
7
)
)

 
 
 
 
a
x
.
p
l
o
t
(
t
r
a
i
n
_
c
l
o
s
e
.
i
n
d
e
x
,
 
(
1
+
p
f
t
.
r
e
t
u
r
n
s
(
)
)
.
c
u
m
p
r
o
d
(
)
.
v
a
l
u
e
s
,
 
l
a
b
e
l
=
'
T
r
a
i
n
 
(
I
S
)
'
,
 
c
o
l
o
r
=
'
b
l
u
e
'
,
 
l
i
n
e
w
i
d
t
h
=
1
.
5
,
 
a
l
p
h
a
=
0
.
8
)

 
 
 
 
a
x
.
p
l
o
t
(
v
a
l
_
c
l
o
s
e
.
i
n
d
e
x
,
 
(
1
+
p
f
v
.
r
e
t
u
r
n
s
(
)
)
.
c
u
m
p
r
o
d
(
)
.
v
a
l
u
e
s
,
 
l
a
b
e
l
=
'
V
a
l
i
d
a
t
i
o
n
 
(
O
O
S
)
'
,
 
c
o
l
o
r
=
'
o
r
a
n
g
e
'
,
 
l
i
n
e
w
i
d
t
h
=
1
.
5
,
 
a
l
p
h
a
=
0
.
8
)

 
 
 
 
a
x
.
a
x
v
l
i
n
e
(
x
=
t
r
a
i
n
_
c
l
o
s
e
.
i
n
d
e
x
[
-
1
]
,
 
c
o
l
o
r
=
'
r
e
d
'
,
 
l
i
n
e
s
t
y
l
e
=
'
-
-
'
,
 
a
l
p
h
a
=
0
.
5
,
 
l
a
b
e
l
=
'
T
r
a
i
n
/
V
a
l
 
S
p
l
i
t
'
)

 
 
 
 
a
x
.
s
e
t
_
t
i
t
l
e
(
f
'
L
o
s
t
 
M
o
m
e
n
t
u
m
 
{
b
_
m
t
}
(
m
a
=
{
b
_
m
p
}
,
 
s
l
o
p
e
_
l
b
=
{
b
_
s
l
}
,
 
s
w
i
n
g
_
l
b
=
{
b
_
s
w
}
)
 
-
 
E
q
u
i
t
y
 
C
u
r
v
e
s
'
,
 
f
o
n
t
s
i
z
e
=
1
4
,
 
f
o
n
t
w
e
i
g
h
t
=
'
b
o
l
d
'
)

 
 
 
 
a
x
.
s
e
t
_
x
l
a
b
e
l
(
'
D
a
t
e
'
,
 
f
o
n
t
s
i
z
e
=
1
2
)
;
 
a
x
.
s
e
t
_
y
l
a
b
e
l
(
'
C
u
m
u
l
a
t
i
v
e
 
R
e
t
u
r
n
s
'
,
 
f
o
n
t
s
i
z
e
=
1
2
)

 
 
 
 
a
x
.
g
r
i
d
(
T
r
u
e
,
 
a
l
p
h
a
=
0
.
3
)
;
 
a
x
.
l
e
g
e
n
d
(
l
o
c
=
'
b
e
s
t
'
)
;
 
p
l
t
.
t
i
g
h
t
_
l
a
y
o
u
t
(
)
;
 
p
l
t
.
s
h
o
w
(
)

P
A
R
A
M
E
T
E
R
 
S
E
N
S
I
T
I
V
I
T
Y
 
A
N
A
L
Y
S
I
S

-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-


F
o
r
 
e
a
c
h
 
p
a
r
a
m
e
t
e
r
 
(
`
m
a
_
p
e
r
i
o
d
`
,
 
`
s
l
o
p
e
_
l
o
o
k
b
a
c
k
`
,
 
`
s
w
i
n
g
_
l
o
o
k
b
a
c
k
`
)
,
 
w
e
 
v
a
r
y
 
i
t
 
w
h
i
l
e
 
h
o
l
d
i
n
g
 
t
h
e
 
o
t
h
e
r
s
 
f
i
x
e
d
 
a
t
 
t
h
e
i
r
 
b
e
s
t
 
v
a
l
u
e
s
.
 
W
e
 
a
l
s
o
 
t
e
s
t
 
b
o
t
h
 
M
A
 
t
y
p
e
s
.


*
*
C
o
l
o
r
 
c
o
d
i
n
g
*
*
:
 
D
a
r
k
 
g
r
e
e
n
 
(
>
+
1
0
%
 
v
s
 
b
a
s
e
)
,
 
L
i
g
h
t
 
g
r
e
e
n
 
(
0
-
1
0
%
)
,
 
O
r
a
n
g
e
 
(
-
1
0
-
0
%
)
,
 
R
e
d
 
(
<
-
1
0
%
)
.


-
-
-

In [ ]:
#
 
P
A
R
A
M
E
T
E
R
 
S
E
N
S
I
T
I
V
I
T
Y
 
A
N
A
L
Y
S
I
S
 
-
 
B
A
R
 
C
H
A
R
T
S


i
f
 
r
e
s
u
l
t
s
_
d
f
.
e
m
p
t
y
:

 
 
 
 
p
r
i
n
t
(
"
N
o
 
r
e
s
u
l
t
s
.
"
)

e
l
s
e
:

 
 
 
 
b
e
s
t
 
=
 
r
e
s
u
l
t
s
_
d
f
.
l
o
c
[
r
e
s
u
l
t
s
_
d
f
[
'
s
h
a
r
p
e
_
r
a
t
i
o
'
]
.
i
d
x
m
a
x
(
)
]

 
 
 
 
b
_
m
p
 
=
 
i
n
t
(
b
e
s
t
[
'
m
a
_
p
e
r
i
o
d
'
]
)
;
 
b
_
m
t
 
=
 
b
e
s
t
[
'
m
a
_
t
y
p
e
'
]

 
 
 
 
b
_
s
l
 
=
 
i
n
t
(
b
e
s
t
[
'
s
l
o
p
e
_
l
o
o
k
b
a
c
k
'
]
)
;
 
b
_
s
w
 
=
 
i
n
t
(
b
e
s
t
[
'
s
w
i
n
g
_
l
o
o
k
b
a
c
k
'
]
)

 
 
 
 
b
a
s
e
_
i
s
_
s
h
a
r
p
e
 
=
 
f
l
o
a
t
(
b
e
s
t
[
'
s
h
a
r
p
e
_
r
a
t
i
o
'
]
)
;
 
b
a
s
e
_
o
o
s
_
s
h
a
r
p
e
 
=
 
n
p
.
n
a
n

 
 
 
 
t
r
y
:

 
 
 
 
 
 
 
 
e
_
c
,
 
x
_
c
 
=
 
_
l
o
s
t
_
m
o
m
e
n
t
u
m
_
s
i
g
n
a
l
s
(
v
a
l
_
c
l
o
s
e
,
 
v
a
l
_
l
o
w
,
 
b
_
m
p
,
 
b
_
m
t
,
 
b
_
s
l
,
 
b
_
s
w
)

 
 
 
 
 
 
 
 
p
f
_
c
 
=
 
v
b
t
.
P
o
r
t
f
o
l
i
o
.
f
r
o
m
_
s
i
g
n
a
l
s
(
c
l
o
s
e
=
v
a
l
_
c
l
o
s
e
,
 
e
n
t
r
i
e
s
=
e
_
c
,
 
e
x
i
t
s
=
x
_
c
,
 
i
n
i
t
_
c
a
s
h
=
1
0
0
_
0
0
0
,
 
f
e
e
s
=
0
.
0
0
0
5
,
 
s
l
i
p
p
a
g
e
=
0
.
0
0
0
5
,
 
f
r
e
q
=
'
D
'
)

 
 
 
 
 
 
 
 
b
a
s
e
_
o
o
s
_
s
h
a
r
p
e
 
=
 
f
l
o
a
t
(
p
f
_
c
.
s
h
a
r
p
e
_
r
a
t
i
o
(
f
r
e
q
=
'
D
'
)
)

 
 
 
 
e
x
c
e
p
t
:
 
p
a
s
s

 
 
 
 
p
r
i
n
t
(
f
"
\
U
0
0
0
1
f
5
2
c
 
S
E
N
S
I
T
I
V
I
T
Y
 
A
N
A
L
Y
S
I
S
 
-
 
B
a
s
e
:
 
{
b
_
m
t
}
(
m
a
=
{
b
_
m
p
}
,
 
s
l
o
p
e
_
l
b
=
{
b
_
s
l
}
,
 
s
w
i
n
g
_
l
b
=
{
b
_
s
w
}
)
"
)

 
 
 
 
p
r
i
n
t
(
f
"
I
S
 
S
h
a
r
p
e
:
 
{
b
a
s
e
_
i
s
_
s
h
a
r
p
e
:
.
3
f
}
"
 
+
 
(
f
"
 
|
 
O
O
S
 
S
h
a
r
p
e
:
 
{
b
a
s
e
_
o
o
s
_
s
h
a
r
p
e
:
.
3
f
}
"
 
i
f
 
n
o
t
 
n
p
.
i
s
n
a
n
(
b
a
s
e
_
o
o
s
_
s
h
a
r
p
e
)
 
e
l
s
e
 
"
"
)
)


 
 
 
 
#
 
P
a
r
a
m
e
t
e
r
 
v
a
r
i
a
t
i
o
n
s

 
 
 
 
m
a
_
s
e
n
s
 
=
 
[
2
0
,
 
3
0
,
 
4
0
,
 
5
0
,
 
6
2
,
 
8
0
,
 
1
0
0
,
 
1
2
0
,
 
1
5
0
]

 
 
 
 
s
l
o
p
e
_
s
e
n
s
 
=
 
[
2
,
 
3
,
 
5
,
 
8
,
 
1
0
,
 
1
2
,
 
1
5
]

 
 
 
 
s
w
i
n
g
_
s
e
n
s
 
=
 
[
5
,
 
8
,
 
1
0
,
 
1
5
,
 
2
0
,
 
2
5
,
 
3
0
]


 
 
 
 
c
o
m
b
o
s
_
m
a
 
 
 
 
=
 
[
(
m
,
 
b
_
m
t
,
 
b
_
s
l
,
 
b
_
s
w
)
 
f
o
r
 
m
 
i
n
 
m
a
_
s
e
n
s
]

 
 
 
 
c
o
m
b
o
s
_
s
l
o
p
e
 
=
 
[
(
b
_
m
p
,
 
b
_
m
t
,
 
s
,
 
b
_
s
w
)
 
f
o
r
 
s
 
i
n
 
s
l
o
p
e
_
s
e
n
s
]

 
 
 
 
c
o
m
b
o
s
_
s
w
i
n
g
 
=
 
[
(
b
_
m
p
,
 
b
_
m
t
,
 
b
_
s
l
,
 
s
)
 
f
o
r
 
s
 
i
n
 
s
w
i
n
g
_
s
e
n
s
]

 
 
 
 
a
l
l
_
c
o
m
b
o
s
 
=
 
c
o
m
b
o
s
_
m
a
 
+
 
c
o
m
b
o
s
_
s
l
o
p
e
 
+
 
c
o
m
b
o
s
_
s
w
i
n
g


 
 
 
 
d
e
f
 
e
v
a
l
_
b
o
t
h
(
m
p
,
 
m
t
,
 
s
l
,
 
s
w
)
:

 
 
 
 
 
 
 
 
e
_
i
s
,
 
x
_
i
s
 
=
 
_
l
o
s
t
_
m
o
m
e
n
t
u
m
_
s
i
g
n
a
l
s
(
t
r
a
i
n
_
c
l
o
s
e
,
 
t
r
a
i
n
_
l
o
w
,
 
m
p
,
 
m
t
,
 
s
l
,
 
s
w
)

 
 
 
 
 
 
 
 
p
f
_
i
s
 
=
 
v
b
t
.
P
o
r
t
f
o
l
i
o
.
f
r
o
m
_
s
i
g
n
a
l
s
(
c
l
o
s
e
=
t
r
a
i
n
_
c
l
o
s
e
,
 
e
n
t
r
i
e
s
=
e
_
i
s
,
 
e
x
i
t
s
=
x
_
i
s
,
 
i
n
i
t
_
c
a
s
h
=
1
0
0
_
0
0
0
,
 
f
e
e
s
=
0
.
0
0
0
5
,
 
s
l
i
p
p
a
g
e
=
0
.
0
0
0
5
,
 
f
r
e
q
=
'
D
'
)

 
 
 
 
 
 
 
 
e
_
o
o
s
,
 
x
_
o
o
s
 
=
 
_
l
o
s
t
_
m
o
m
e
n
t
u
m
_
s
i
g
n
a
l
s
(
v
a
l
_
c
l
o
s
e
,
 
v
a
l
_
l
o
w
,
 
m
p
,
 
m
t
,
 
s
l
,
 
s
w
)

 
 
 
 
 
 
 
 
p
f
_
o
o
s
 
=
 
v
b
t
.
P
o
r
t
f
o
l
i
o
.
f
r
o
m
_
s
i
g
n
a
l
s
(
c
l
o
s
e
=
v
a
l
_
c
l
o
s
e
,
 
e
n
t
r
i
e
s
=
e
_
o
o
s
,
 
e
x
i
t
s
=
x
_
o
o
s
,
 
i
n
i
t
_
c
a
s
h
=
1
0
0
_
0
0
0
,
 
f
e
e
s
=
0
.
0
0
0
5
,
 
s
l
i
p
p
a
g
e
=
0
.
0
0
0
5
,
 
f
r
e
q
=
'
D
'
)

 
 
 
 
 
 
 
 
r
e
t
u
r
n
 
{
'
m
a
_
p
e
r
i
o
d
'
:
 
m
p
,
 
'
m
a
_
t
y
p
e
'
:
 
m
t
,
 
'
s
l
o
p
e
_
l
o
o
k
b
a
c
k
'
:
 
s
l
,
 
'
s
w
i
n
g
_
l
o
o
k
b
a
c
k
'
:
 
s
w
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
'
i
s
_
s
h
a
r
p
e
'
:
 
f
l
o
a
t
(
p
f
_
i
s
.
s
h
a
r
p
e
_
r
a
t
i
o
(
f
r
e
q
=
'
D
'
)
)
,
 
'
o
o
s
_
s
h
a
r
p
e
'
:
 
f
l
o
a
t
(
p
f
_
o
o
s
.
s
h
a
r
p
e
_
r
a
t
i
o
(
f
r
e
q
=
'
D
'
)
)
}


 
 
 
 
p
r
i
n
t
(
f
"
\
n
\
U
0
0
0
1
f
5
0
4
 
T
e
s
t
i
n
g
 
{
l
e
n
(
a
l
l
_
c
o
m
b
o
s
)
}
 
p
a
r
a
m
e
t
e
r
 
v
a
r
i
a
t
i
o
n
s
.
.
.
"
)

 
 
 
 
r
o
w
s
 
=
 
[
]

 
 
 
 
f
o
r
 
c
 
i
n
 
a
l
l
_
c
o
m
b
o
s
:

 
 
 
 
 
 
 
 
t
r
y
:
 
r
o
w
s
.
a
p
p
e
n
d
(
e
v
a
l
_
b
o
t
h
(
*
c
)
)

 
 
 
 
 
 
 
 
e
x
c
e
p
t
:
 
p
a
s
s


 
 
 
 
i
f
 
n
o
t
 
r
o
w
s
:

 
 
 
 
 
 
 
 
p
r
i
n
t
(
"
\
u
2
7
4
c
 
N
o
 
s
e
n
s
i
t
i
v
i
t
y
 
r
e
s
u
l
t
s
.
"
)

 
 
 
 
e
l
s
e
:

 
 
 
 
 
 
 
 
s
e
n
s
_
d
f
 
=
 
p
d
.
D
a
t
a
F
r
a
m
e
(
r
o
w
s
)


 
 
 
 
 
 
 
 
#
 
F
i
l
t
e
r
 
b
y
 
v
a
r
y
i
n
g
 
p
a
r
a
m
e
t
e
r

 
 
 
 
 
 
 
 
m
v
 
=
 
s
e
n
s
_
d
f
[
(
s
e
n
s
_
d
f
[
'
m
a
_
t
y
p
e
'
]
=
=
b
_
m
t
)
&
(
s
e
n
s
_
d
f
[
'
s
l
o
p
e
_
l
o
o
k
b
a
c
k
'
]
=
=
b
_
s
l
)
&
(
s
e
n
s
_
d
f
[
'
s
w
i
n
g
_
l
o
o
k
b
a
c
k
'
]
=
=
b
_
s
w
)
]
.
c
o
p
y
(
)
.
s
o
r
t
_
v
a
l
u
e
s
(
'
m
a
_
p
e
r
i
o
d
'
)

 
 
 
 
 
 
 
 
s
v
 
=
 
s
e
n
s
_
d
f
[
(
s
e
n
s
_
d
f
[
'
m
a
_
t
y
p
e
'
]
=
=
b
_
m
t
)
&
(
s
e
n
s
_
d
f
[
'
m
a
_
p
e
r
i
o
d
'
]
=
=
b
_
m
p
)
&
(
s
e
n
s
_
d
f
[
'
s
w
i
n
g
_
l
o
o
k
b
a
c
k
'
]
=
=
b
_
s
w
)
]
.
c
o
p
y
(
)
.
s
o
r
t
_
v
a
l
u
e
s
(
'
s
l
o
p
e
_
l
o
o
k
b
a
c
k
'
)

 
 
 
 
 
 
 
 
s
w
v
 
=
 
s
e
n
s
_
d
f
[
(
s
e
n
s
_
d
f
[
'
m
a
_
t
y
p
e
'
]
=
=
b
_
m
t
)
&
(
s
e
n
s
_
d
f
[
'
m
a
_
p
e
r
i
o
d
'
]
=
=
b
_
m
p
)
&
(
s
e
n
s
_
d
f
[
'
s
l
o
p
e
_
l
o
o
k
b
a
c
k
'
]
=
=
b
_
s
l
)
]
.
c
o
p
y
(
)
.
s
o
r
t
_
v
a
l
u
e
s
(
'
s
w
i
n
g
_
l
o
o
k
b
a
c
k
'
)


 
 
 
 
 
 
 
 
m
v
[
'
i
s
_
s
h
a
r
p
e
_
d
e
l
t
a
'
]
 
=
 
(
m
v
[
'
i
s
_
s
h
a
r
p
e
'
]
-
b
a
s
e
_
i
s
_
s
h
a
r
p
e
)
/
a
b
s
(
b
a
s
e
_
i
s
_
s
h
a
r
p
e
)
*
1
0
0

 
 
 
 
 
 
 
 
s
v
[
'
i
s
_
s
h
a
r
p
e
_
d
e
l
t
a
'
]
 
=
 
(
s
v
[
'
i
s
_
s
h
a
r
p
e
'
]
-
b
a
s
e
_
i
s
_
s
h
a
r
p
e
)
/
a
b
s
(
b
a
s
e
_
i
s
_
s
h
a
r
p
e
)
*
1
0
0

 
 
 
 
 
 
 
 
s
w
v
[
'
i
s
_
s
h
a
r
p
e
_
d
e
l
t
a
'
]
 
=
 
(
s
w
v
[
'
i
s
_
s
h
a
r
p
e
'
]
-
b
a
s
e
_
i
s
_
s
h
a
r
p
e
)
/
a
b
s
(
b
a
s
e
_
i
s
_
s
h
a
r
p
e
)
*
1
0
0

 
 
 
 
 
 
 
 
i
f
 
n
o
t
 
n
p
.
i
s
n
a
n
(
b
a
s
e
_
o
o
s
_
s
h
a
r
p
e
)
:

 
 
 
 
 
 
 
 
 
 
 
 
m
v
[
'
o
o
s
_
s
h
a
r
p
e
_
d
e
l
t
a
'
]
 
=
 
(
m
v
[
'
o
o
s
_
s
h
a
r
p
e
'
]
-
b
a
s
e
_
o
o
s
_
s
h
a
r
p
e
)
/
a
b
s
(
b
a
s
e
_
o
o
s
_
s
h
a
r
p
e
)
*
1
0
0

 
 
 
 
 
 
 
 
 
 
 
 
s
v
[
'
o
o
s
_
s
h
a
r
p
e
_
d
e
l
t
a
'
]
 
=
 
(
s
v
[
'
o
o
s
_
s
h
a
r
p
e
'
]
-
b
a
s
e
_
o
o
s
_
s
h
a
r
p
e
)
/
a
b
s
(
b
a
s
e
_
o
o
s
_
s
h
a
r
p
e
)
*
1
0
0

 
 
 
 
 
 
 
 
 
 
 
 
s
w
v
[
'
o
o
s
_
s
h
a
r
p
e
_
d
e
l
t
a
'
]
 
=
 
(
s
w
v
[
'
o
o
s
_
s
h
a
r
p
e
'
]
-
b
a
s
e
_
o
o
s
_
s
h
a
r
p
e
)
/
a
b
s
(
b
a
s
e
_
o
o
s
_
s
h
a
r
p
e
)
*
1
0
0


 
 
 
 
 
 
 
 
f
i
g
,
 
a
x
e
s
 
=
 
p
l
t
.
s
u
b
p
l
o
t
s
(
2
,
 
3
,
 
f
i
g
s
i
z
e
=
(
2
0
,
 
1
0
)
)

 
 
 
 
 
 
 
 
f
i
g
.
s
u
p
t
i
t
l
e
(
f
'
S
e
n
s
i
t
i
v
i
t
y
 
-
 
B
a
s
e
:
 
{
b
_
m
t
}
(
m
a
=
{
b
_
m
p
}
,
 
s
l
o
p
e
_
l
b
=
{
b
_
s
l
}
,
 
s
w
i
n
g
_
l
b
=
{
b
_
s
w
}
)
'
,
 
f
o
n
t
s
i
z
e
=
1
6
,
 
f
o
n
t
w
e
i
g
h
t
=
'
b
o
l
d
'
)


 
 
 
 
 
 
 
 
f
o
r
 
c
i
,
 
(
p
n
,
 
v
a
r
,
 
p
c
,
 
b
v
)
 
i
n
 
e
n
u
m
e
r
a
t
e
(
[

 
 
 
 
 
 
 
 
 
 
 
 
(
'
M
A
 
P
e
r
i
o
d
'
,
 
m
v
,
 
'
m
a
_
p
e
r
i
o
d
'
,
 
b
_
m
p
)
,

 
 
 
 
 
 
 
 
 
 
 
 
(
'
S
l
o
p
e
 
L
o
o
k
b
a
c
k
'
,
 
s
v
,
 
'
s
l
o
p
e
_
l
o
o
k
b
a
c
k
'
,
 
b
_
s
l
)
,

 
 
 
 
 
 
 
 
 
 
 
 
(
'
S
w
i
n
g
 
L
o
o
k
b
a
c
k
'
,
 
s
w
v
,
 
'
s
w
i
n
g
_
l
o
o
k
b
a
c
k
'
,
 
b
_
s
w
)

 
 
 
 
 
 
 
 
]
)
:

 
 
 
 
 
 
 
 
 
 
 
 
i
f
 
v
a
r
.
e
m
p
t
y
:

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
 
c
i
]
.
t
e
x
t
(
0
.
5
,
 
0
.
5
,
 
'
N
o
 
d
a
t
a
'
,
 
h
a
=
'
c
e
n
t
e
r
'
,
 
v
a
=
'
c
e
n
t
e
r
'
)

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
 
c
i
]
.
t
e
x
t
(
0
.
5
,
 
0
.
5
,
 
'
N
o
 
d
a
t
a
'
,
 
h
a
=
'
c
e
n
t
e
r
'
,
 
v
a
=
'
c
e
n
t
e
r
'
)

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
c
o
n
t
i
n
u
e


 
 
 
 
 
 
 
 
 
 
 
 
#
 
I
S
 
r
o
w

 
 
 
 
 
 
 
 
 
 
 
 
c
o
l
o
r
s
 
=
 
[
'
r
e
d
'
 
i
f
 
x
 
<
 
-
1
0
 
e
l
s
e
 
'
o
r
a
n
g
e
'
 
i
f
 
x
 
<
 
0
 
e
l
s
e
 
'
l
i
g
h
t
g
r
e
e
n
'
 
i
f
 
x
 
<
 
1
0
 
e
l
s
e
 
'
d
a
r
k
g
r
e
e
n
'
 
f
o
r
 
x
 
i
n
 
v
a
r
[
'
i
s
_
s
h
a
r
p
e
_
d
e
l
t
a
'
]
]

 
 
 
 
 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
 
c
i
]
.
b
a
r
(
v
a
r
[
p
c
]
.
a
s
t
y
p
e
(
s
t
r
)
,
 
v
a
r
[
'
i
s
_
s
h
a
r
p
e
_
d
e
l
t
a
'
]
,
 
c
o
l
o
r
=
c
o
l
o
r
s
,
 
e
d
g
e
c
o
l
o
r
=
'
b
l
a
c
k
'
,
 
l
i
n
e
w
i
d
t
h
=
0
.
5
)

 
 
 
 
 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
 
c
i
]
.
a
x
h
l
i
n
e
(
0
,
 
c
o
l
o
r
=
'
b
l
a
c
k
'
,
 
l
i
n
e
w
i
d
t
h
=
1
.
5
)

 
 
 
 
 
 
 
 
 
 
 
 
b
v
_
s
t
r
 
=
 
s
t
r
(
b
v
)

 
 
 
 
 
 
 
 
 
 
 
 
i
f
 
b
v
_
s
t
r
 
i
n
 
v
a
r
[
p
c
]
.
a
s
t
y
p
e
(
s
t
r
)
.
v
a
l
u
e
s
:

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
b
v
_
i
d
x
 
=
 
l
i
s
t
(
v
a
r
[
p
c
]
.
a
s
t
y
p
e
(
s
t
r
)
.
v
a
l
u
e
s
)
.
i
n
d
e
x
(
b
v
_
s
t
r
)

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
 
c
i
]
.
a
x
v
l
i
n
e
(
b
v
_
i
d
x
,
 
c
o
l
o
r
=
'
b
l
u
e
'
,
 
l
i
n
e
s
t
y
l
e
=
'
-
-
'
,
 
l
i
n
e
w
i
d
t
h
=
2
.
5
,
 
a
l
p
h
a
=
0
.
7
,
 
l
a
b
e
l
=
f
'
B
a
s
e
=
{
b
v
}
'
)

 
 
 
 
 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
 
c
i
]
.
s
e
t
_
t
i
t
l
e
(
f
'
{
p
n
}
 
(
I
S
 
S
h
a
r
p
e
 
%
\
u
0
3
9
4
)
'
,
 
f
o
n
t
s
i
z
e
=
1
2
,
 
f
o
n
t
w
e
i
g
h
t
=
'
b
o
l
d
'
)

 
 
 
 
 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
 
c
i
]
.
s
e
t
_
y
l
a
b
e
l
(
'
S
h
a
r
p
e
 
\
u
0
3
9
4
 
%
'
 
i
f
 
c
i
 
=
=
 
0
 
e
l
s
e
 
'
'
)

 
 
 
 
 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
 
c
i
]
.
l
e
g
e
n
d
(
f
o
n
t
s
i
z
e
=
9
)

 
 
 
 
 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
 
c
i
]
.
g
r
i
d
(
a
x
i
s
=
'
y
'
,
 
a
l
p
h
a
=
0
.
3
)


 
 
 
 
 
 
 
 
 
 
 
 
#
 
O
O
S
 
r
o
w

 
 
 
 
 
 
 
 
 
 
 
 
i
f
 
'
o
o
s
_
s
h
a
r
p
e
_
d
e
l
t
a
'
 
i
n
 
v
a
r
.
c
o
l
u
m
n
s
:

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
c
o
l
o
r
s
_
o
o
s
 
=
 
[
'
r
e
d
'
 
i
f
 
x
 
<
 
-
1
0
 
e
l
s
e
 
'
o
r
a
n
g
e
'
 
i
f
 
x
 
<
 
0
 
e
l
s
e
 
'
l
i
g
h
t
g
r
e
e
n
'
 
i
f
 
x
 
<
 
1
0
 
e
l
s
e
 
'
d
a
r
k
g
r
e
e
n
'
 
f
o
r
 
x
 
i
n
 
v
a
r
[
'
o
o
s
_
s
h
a
r
p
e
_
d
e
l
t
a
'
]
]

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
 
c
i
]
.
b
a
r
(
v
a
r
[
p
c
]
.
a
s
t
y
p
e
(
s
t
r
)
,
 
v
a
r
[
'
o
o
s
_
s
h
a
r
p
e
_
d
e
l
t
a
'
]
,
 
c
o
l
o
r
=
c
o
l
o
r
s
_
o
o
s
,
 
e
d
g
e
c
o
l
o
r
=
'
b
l
a
c
k
'
,
 
l
i
n
e
w
i
d
t
h
=
0
.
5
)

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
 
c
i
]
.
a
x
h
l
i
n
e
(
0
,
 
c
o
l
o
r
=
'
b
l
a
c
k
'
,
 
l
i
n
e
w
i
d
t
h
=
1
.
5
)

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
i
f
 
b
v
_
s
t
r
 
i
n
 
v
a
r
[
p
c
]
.
a
s
t
y
p
e
(
s
t
r
)
.
v
a
l
u
e
s
:

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
b
v
_
i
d
x
 
=
 
l
i
s
t
(
v
a
r
[
p
c
]
.
a
s
t
y
p
e
(
s
t
r
)
.
v
a
l
u
e
s
)
.
i
n
d
e
x
(
b
v
_
s
t
r
)

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
 
c
i
]
.
a
x
v
l
i
n
e
(
b
v
_
i
d
x
,
 
c
o
l
o
r
=
'
b
l
u
e
'
,
 
l
i
n
e
s
t
y
l
e
=
'
-
-
'
,
 
l
i
n
e
w
i
d
t
h
=
2
.
5
,
 
a
l
p
h
a
=
0
.
7
,
 
l
a
b
e
l
=
f
'
B
a
s
e
=
{
b
v
}
'
)

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
 
c
i
]
.
s
e
t
_
t
i
t
l
e
(
f
'
{
p
n
}
 
(
O
O
S
 
S
h
a
r
p
e
 
%
\
u
0
3
9
4
)
'
,
 
f
o
n
t
s
i
z
e
=
1
2
,
 
f
o
n
t
w
e
i
g
h
t
=
'
b
o
l
d
'
)

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
 
c
i
]
.
s
e
t
_
y
l
a
b
e
l
(
'
S
h
a
r
p
e
 
\
u
0
3
9
4
 
%
'
 
i
f
 
c
i
 
=
=
 
0
 
e
l
s
e
 
'
'
)

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
 
c
i
]
.
s
e
t
_
x
l
a
b
e
l
(
p
n
)

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
 
c
i
]
.
l
e
g
e
n
d
(
f
o
n
t
s
i
z
e
=
9
)

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
 
c
i
]
.
g
r
i
d
(
a
x
i
s
=
'
y
'
,
 
a
l
p
h
a
=
0
.
3
)

 
 
 
 
 
 
 
 
 
 
 
 
e
l
s
e
:

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
 
c
i
]
.
t
e
x
t
(
0
.
5
,
 
0
.
5
,
 
'
N
o
 
O
O
S
 
d
a
t
a
'
,
 
h
a
=
'
c
e
n
t
e
r
'
,
 
v
a
=
'
c
e
n
t
e
r
'
,
 
f
o
n
t
s
i
z
e
=
1
2
)


 
 
 
 
 
 
 
 
p
l
t
.
t
i
g
h
t
_
l
a
y
o
u
t
(
)

 
 
 
 
 
 
 
 
p
l
t
.
s
h
o
w
(
)


 
 
 
 
 
 
 
 
#
 
S
e
n
s
i
t
i
v
i
t
y
 
s
u
m
m
a
r
y

 
 
 
 
 
 
 
 
p
r
i
n
t
(
"
\
n
"
 
+
 
"
=
"
 
*
 
7
0
)

 
 
 
 
 
 
 
 
p
r
i
n
t
(
"
\
U
0
0
0
1
f
4
c
b
 
S
E
N
S
I
T
I
V
I
T
Y
 
S
U
M
M
A
R
Y
"
)

 
 
 
 
 
 
 
 
p
r
i
n
t
(
"
=
"
 
*
 
7
0
)

 
 
 
 
 
 
 
 
f
o
r
 
p
n
,
 
v
a
r
,
 
p
c
 
i
n
 
[
(
'
M
A
 
P
e
r
i
o
d
'
,
 
m
v
,
 
'
m
a
_
p
e
r
i
o
d
'
)
,
 
(
'
S
l
o
p
e
 
L
o
o
k
b
a
c
k
'
,
 
s
v
,
 
'
s
l
o
p
e
_
l
o
o
k
b
a
c
k
'
)
,
 
(
'
S
w
i
n
g
 
L
o
o
k
b
a
c
k
'
,
 
s
w
v
,
 
'
s
w
i
n
g
_
l
o
o
k
b
a
c
k
'
)
]
:

 
 
 
 
 
 
 
 
 
 
 
 
i
f
 
v
a
r
.
e
m
p
t
y
 
o
r
 
'
i
s
_
s
h
a
r
p
e
_
d
e
l
t
a
'
 
n
o
t
 
i
n
 
v
a
r
.
c
o
l
u
m
n
s
:

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
c
o
n
t
i
n
u
e

 
 
 
 
 
 
 
 
 
 
 
 
i
s
_
r
a
n
g
e
 
=
 
v
a
r
[
'
i
s
_
s
h
a
r
p
e
_
d
e
l
t
a
'
]
.
m
a
x
(
)
 
-
 
v
a
r
[
'
i
s
_
s
h
a
r
p
e
_
d
e
l
t
a
'
]
.
m
i
n
(
)

 
 
 
 
 
 
 
 
 
 
 
 
f
l
a
g
_
i
s
 
=
 
"
\
u
2
7
0
5
 
L
O
W
"
 
i
f
 
i
s
_
r
a
n
g
e
 
<
 
3
0
 
e
l
s
e
 
"
\
u
2
6
a
0
\
u
f
e
0
f
 
H
I
G
H
"

 
 
 
 
 
 
 
 
 
 
 
 
o
o
s
_
i
n
f
o
 
=
 
"
"

 
 
 
 
 
 
 
 
 
 
 
 
i
f
 
'
o
o
s
_
s
h
a
r
p
e
_
d
e
l
t
a
'
 
i
n
 
v
a
r
.
c
o
l
u
m
n
s
:

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
o
o
s
_
r
a
n
g
e
 
=
 
v
a
r
[
'
o
o
s
_
s
h
a
r
p
e
_
d
e
l
t
a
'
]
.
m
a
x
(
)
 
-
 
v
a
r
[
'
o
o
s
_
s
h
a
r
p
e
_
d
e
l
t
a
'
]
.
m
i
n
(
)

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
f
l
a
g
_
o
o
s
 
=
 
"
\
u
2
7
0
5
 
L
O
W
"
 
i
f
 
o
o
s
_
r
a
n
g
e
 
<
 
3
0
 
e
l
s
e
 
"
\
u
2
6
a
0
\
u
f
e
0
f
 
H
I
G
H
"

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
o
o
s
_
i
n
f
o
 
=
 
f
"
 
|
 
O
O
S
 
r
a
n
g
e
:
 
{
o
o
s
_
r
a
n
g
e
:
.
1
f
}
%
 
{
f
l
a
g
_
o
o
s
}
"

 
 
 
 
 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
 
 
{
p
n
:
2
0
s
}
 
I
S
 
r
a
n
g
e
:
 
{
i
s
_
r
a
n
g
e
:
.
1
f
}
%
 
{
f
l
a
g
_
i
s
}
{
o
o
s
_
i
n
f
o
}
"
)

 
 
 
 
 
 
 
 
p
r
i
n
t
(
"
=
"
 
*
 
7
0
)

In [ ]:
#
 
F
U
L
L
-
S
A
M
P
L
E
 
E
V
A
L
U
A
T
I
O
N
 
+
 
B
U
Y
 
&
 
H
O
L
D
 
C
O
M
P
A
R
I
S
O
N
 
+
 
A
N
N
O
T
A
T
E
D
 
E
Q
U
I
T
Y
 
C
U
R
V
E


r
e
s
u
l
t
s
_
d
f
 
=
 
p
d
.
D
a
t
a
F
r
a
m
e
(
g
r
i
d
_
s
e
a
r
c
h
_
r
e
s
u
l
t
s
)

F
R
E
Q
 
=
 
'
D
'


i
f
 
r
e
s
u
l
t
s
_
d
f
.
e
m
p
t
y
:

 
 
 
 
p
r
i
n
t
(
"
N
o
 
r
e
s
u
l
t
s
 
t
o
 
e
v
a
l
u
a
t
e
.
"
)

e
l
s
e
:

 
 
 
 
b
e
s
t
 
=
 
r
e
s
u
l
t
s
_
d
f
.
l
o
c
[
r
e
s
u
l
t
s
_
d
f
[
'
s
h
a
r
p
e
_
r
a
t
i
o
'
]
.
i
d
x
m
a
x
(
)
]

 
 
 
 
b
_
m
p
 
=
 
i
n
t
(
b
e
s
t
[
'
m
a
_
p
e
r
i
o
d
'
]
)
;
 
b
_
m
t
 
=
 
b
e
s
t
[
'
m
a
_
t
y
p
e
'
]

 
 
 
 
b
_
s
l
 
=
 
i
n
t
(
b
e
s
t
[
'
s
l
o
p
e
_
l
o
o
k
b
a
c
k
'
]
)
;
 
b
_
s
w
 
=
 
i
n
t
(
b
e
s
t
[
'
s
w
i
n
g
_
l
o
o
k
b
a
c
k
'
]
)


 
 
 
 
#
 
R
e
-
d
e
r
i
v
e
 
c
l
o
s
e
 
a
n
d
 
l
o
w
 
f
r
o
m
 
s
t
o
c
k
_
d
a
t
a

 
 
 
 
i
f
 
i
s
i
n
s
t
a
n
c
e
(
s
t
o
c
k
_
d
a
t
a
.
c
o
l
u
m
n
s
,
 
p
d
.
M
u
l
t
i
I
n
d
e
x
)
:

 
 
 
 
 
 
 
 
f
u
l
l
_
c
l
o
s
e
 
=
 
s
t
o
c
k
_
d
a
t
a
[
(
'
C
l
o
s
e
'
,
 
T
I
C
K
E
R
)
]
.
a
s
t
y
p
e
(
f
l
o
a
t
)
.
s
q
u
e
e
z
e
(
)

 
 
 
 
 
 
 
 
f
u
l
l
_
l
o
w
 
=
 
s
t
o
c
k
_
d
a
t
a
[
(
'
L
o
w
'
,
 
T
I
C
K
E
R
)
]
.
a
s
t
y
p
e
(
f
l
o
a
t
)
.
s
q
u
e
e
z
e
(
)

 
 
 
 
e
l
s
e
:

 
 
 
 
 
 
 
 
f
u
l
l
_
c
l
o
s
e
 
=
 
s
t
o
c
k
_
d
a
t
a
[
'
C
l
o
s
e
'
]
.
a
s
t
y
p
e
(
f
l
o
a
t
)
.
s
q
u
e
e
z
e
(
)

 
 
 
 
 
 
 
 
f
u
l
l
_
l
o
w
 
=
 
s
t
o
c
k
_
d
a
t
a
[
'
L
o
w
'
]
.
a
s
t
y
p
e
(
f
l
o
a
t
)
.
s
q
u
e
e
z
e
(
)

 
 
 
 
f
u
l
l
_
c
l
o
s
e
.
n
a
m
e
 
=
 
'
p
r
i
c
e
'
;
 
f
u
l
l
_
l
o
w
.
n
a
m
e
 
=
 
'
l
o
w
'


 
 
 
 
s
p
l
i
t
_
i
d
x
 
=
 
i
n
t
(
l
e
n
(
f
u
l
l
_
c
l
o
s
e
)
 
*
 
0
.
6
0
)

 
 
 
 
t
r
a
i
n
_
c
l
o
s
e
 
=
 
f
u
l
l
_
c
l
o
s
e
.
i
l
o
c
[
:
s
p
l
i
t
_
i
d
x
]
.
c
o
p
y
(
)

 
 
 
 
v
a
l
_
c
l
o
s
e
 
=
 
f
u
l
l
_
c
l
o
s
e
.
i
l
o
c
[
s
p
l
i
t
_
i
d
x
:
]
.
c
o
p
y
(
)


 
 
 
 
#
 
S
t
r
a
t
e
g
y
 
o
n
 
f
u
l
l
 
s
a
m
p
l
e

 
 
 
 
e
_
f
u
l
l
,
 
x
_
f
u
l
l
 
=
 
_
l
o
s
t
_
m
o
m
e
n
t
u
m
_
s
i
g
n
a
l
s
(
f
u
l
l
_
c
l
o
s
e
,
 
f
u
l
l
_
l
o
w
,
 
b
_
m
p
,
 
b
_
m
t
,
 
b
_
s
l
,
 
b
_
s
w
)

 
 
 
 
p
f
 
=
 
v
b
t
.
P
o
r
t
f
o
l
i
o
.
f
r
o
m
_
s
i
g
n
a
l
s
(
c
l
o
s
e
=
f
u
l
l
_
c
l
o
s
e
,
 
e
n
t
r
i
e
s
=
e
_
f
u
l
l
,
 
e
x
i
t
s
=
x
_
f
u
l
l
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
i
n
i
t
_
c
a
s
h
=
1
0
0
_
0
0
0
,
 
f
e
e
s
=
0
.
0
0
0
5
,
 
s
l
i
p
p
a
g
e
=
0
.
0
0
0
5
,
 
f
r
e
q
=
F
R
E
Q
)


 
 
 
 
#
 
B
u
y
 
&
 
H
o
l
d

 
 
 
 
b
h
_
e
n
t
r
i
e
s
 
=
 
p
d
.
S
e
r
i
e
s
(
F
a
l
s
e
,
 
i
n
d
e
x
=
f
u
l
l
_
c
l
o
s
e
.
i
n
d
e
x
,
 
d
t
y
p
e
=
b
o
o
l
)
;
 
b
h
_
e
n
t
r
i
e
s
.
i
l
o
c
[
0
]
 
=
 
T
r
u
e

 
 
 
 
b
h
_
e
x
i
t
s
 
=
 
p
d
.
S
e
r
i
e
s
(
F
a
l
s
e
,
 
i
n
d
e
x
=
f
u
l
l
_
c
l
o
s
e
.
i
n
d
e
x
,
 
d
t
y
p
e
=
b
o
o
l
)

 
 
 
 
p
f
_
b
h
 
=
 
v
b
t
.
P
o
r
t
f
o
l
i
o
.
f
r
o
m
_
s
i
g
n
a
l
s
(
c
l
o
s
e
=
f
u
l
l
_
c
l
o
s
e
,
 
e
n
t
r
i
e
s
=
b
h
_
e
n
t
r
i
e
s
,
 
e
x
i
t
s
=
b
h
_
e
x
i
t
s
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
i
n
i
t
_
c
a
s
h
=
1
0
0
_
0
0
0
,
 
f
e
e
s
=
0
.
0
0
0
5
,
 
s
l
i
p
p
a
g
e
=
0
.
0
0
0
5
,
 
f
r
e
q
=
F
R
E
Q
)


 
 
 
 
#
 
M
e
t
r
i
c
s

 
 
 
 
s
t
r
a
t
_
s
h
a
r
p
e
 
=
 
f
l
o
a
t
(
p
f
.
s
h
a
r
p
e
_
r
a
t
i
o
(
f
r
e
q
=
F
R
E
Q
)
)

 
 
 
 
s
t
r
a
t
_
r
e
t
u
r
n
 
=
 
f
l
o
a
t
(
p
f
.
t
o
t
a
l
_
r
e
t
u
r
n
(
)
)

 
 
 
 
s
t
r
a
t
_
d
d
 
=
 
f
l
o
a
t
(
p
f
.
m
a
x
_
d
r
a
w
d
o
w
n
(
)
)

 
 
 
 
s
t
r
a
t
_
s
o
r
t
i
n
o
 
=
 
f
l
o
a
t
(
p
f
.
s
o
r
t
i
n
o
_
r
a
t
i
o
(
f
r
e
q
=
F
R
E
Q
)
)

 
 
 
 
s
t
r
a
t
_
v
o
l
 
=
 
f
l
o
a
t
(
p
f
.
a
n
n
u
a
l
i
z
e
d
_
v
o
l
a
t
i
l
i
t
y
(
f
r
e
q
=
F
R
E
Q
)
)

 
 
 
 
b
h
_
s
h
a
r
p
e
 
=
 
f
l
o
a
t
(
p
f
_
b
h
.
s
h
a
r
p
e
_
r
a
t
i
o
(
f
r
e
q
=
F
R
E
Q
)
)

 
 
 
 
b
h
_
r
e
t
u
r
n
 
=
 
f
l
o
a
t
(
p
f
_
b
h
.
t
o
t
a
l
_
r
e
t
u
r
n
(
)
)

 
 
 
 
b
h
_
d
d
 
=
 
f
l
o
a
t
(
p
f
_
b
h
.
m
a
x
_
d
r
a
w
d
o
w
n
(
)
)


 
 
 
 
t
r
a
d
e
s
_
o
b
j
 
=
 
p
f
.
t
r
a
d
e
s

 
 
 
 
t
r
 
=
 
n
p
.
a
s
a
r
r
a
y
(
t
r
a
d
e
s
_
o
b
j
.
r
e
t
u
r
n
s
.
v
a
l
u
e
s
 
i
f
 
h
a
s
a
t
t
r
(
t
r
a
d
e
s
_
o
b
j
.
r
e
t
u
r
n
s
,
 
'
v
a
l
u
e
s
'
)
 
e
l
s
e
 
t
r
a
d
e
s
_
o
b
j
.
r
e
t
u
r
n
s
)
.
r
a
v
e
l
(
)

 
 
 
 
p
o
s
 
=
 
t
r
[
t
r
 
>
 
0
]
;
 
n
e
g
 
=
 
t
r
[
t
r
 
<
 
0
]

 
 
 
 
w
r
 
=
 
l
e
n
(
p
o
s
)
/
l
e
n
(
t
r
)
*
1
0
0
 
i
f
 
l
e
n
(
t
r
)
 
>
 
0
 
e
l
s
e
 
0

 
 
 
 
p
f
_
r
a
t
i
o
 
=
 
p
o
s
.
s
u
m
(
)
/
a
b
s
(
n
e
g
.
s
u
m
(
)
)
 
i
f
 
l
e
n
(
n
e
g
)
 
>
 
0
 
a
n
d
 
a
b
s
(
n
e
g
.
s
u
m
(
)
)
 
>
 
0
 
e
l
s
e
 
n
p
.
i
n
f


 
 
 
 
p
r
i
n
t
(
"
=
"
 
*
 
7
0
)

 
 
 
 
p
r
i
n
t
(
f
"
\
U
0
0
0
1
f
4
c
a
 
F
U
L
L
-
S
A
M
P
L
E
 
E
V
A
L
U
A
T
I
O
N
 
-
 
L
o
s
t
 
M
o
m
e
n
t
u
m
 
{
b
_
m
t
}
(
m
a
=
{
b
_
m
p
}
,
 
s
l
o
p
e
_
l
b
=
{
b
_
s
l
}
,
 
s
w
i
n
g
_
l
b
=
{
b
_
s
w
}
)
"
)

 
 
 
 
p
r
i
n
t
(
"
=
"
 
*
 
7
0
)

 
 
 
 
p
r
i
n
t
(
f
"
{
'
M
e
t
r
i
c
'
:
<
2
5
}
 
{
'
S
t
r
a
t
e
g
y
'
:
>
1
5
}
 
{
'
B
u
y
 
&
 
H
o
l
d
'
:
>
1
5
}
"
)

 
 
 
 
p
r
i
n
t
(
"
-
"
 
*
 
5
5
)

 
 
 
 
p
r
i
n
t
(
f
"
{
'
T
o
t
a
l
 
R
e
t
u
r
n
'
:
<
2
5
}
 
{
s
t
r
a
t
_
r
e
t
u
r
n
:
>
1
4
.
2
%
}
 
{
b
h
_
r
e
t
u
r
n
:
>
1
4
.
2
%
}
"
)

 
 
 
 
p
r
i
n
t
(
f
"
{
'
S
h
a
r
p
e
 
R
a
t
i
o
'
:
<
2
5
}
 
{
s
t
r
a
t
_
s
h
a
r
p
e
:
>
1
5
.
3
f
}
 
{
b
h
_
s
h
a
r
p
e
:
>
1
5
.
3
f
}
"
)

 
 
 
 
p
r
i
n
t
(
f
"
{
'
S
o
r
t
i
n
o
 
R
a
t
i
o
'
:
<
2
5
}
 
{
s
t
r
a
t
_
s
o
r
t
i
n
o
:
>
1
5
.
3
f
}
 
{
'
-
-
'
:
>
1
5
}
"
)

 
 
 
 
p
r
i
n
t
(
f
"
{
'
M
a
x
 
D
r
a
w
d
o
w
n
'
:
<
2
5
}
 
{
s
t
r
a
t
_
d
d
:
>
1
4
.
2
%
}
 
{
b
h
_
d
d
:
>
1
4
.
2
%
}
"
)

 
 
 
 
p
r
i
n
t
(
f
"
{
'
V
o
l
a
t
i
l
i
t
y
'
:
<
2
5
}
 
{
s
t
r
a
t
_
v
o
l
:
>
1
4
.
2
%
}
 
{
'
-
-
'
:
>
1
5
}
"
)

 
 
 
 
p
r
i
n
t
(
f
"
{
'
W
i
n
 
R
a
t
e
'
:
<
2
5
}
 
{
w
r
:
>
1
4
.
1
f
}
%
 
{
'
-
-
'
:
>
1
5
}
"
)

 
 
 
 
p
r
i
n
t
(
f
"
{
'
P
r
o
f
i
t
 
F
a
c
t
o
r
'
:
<
2
5
}
 
{
p
f
_
r
a
t
i
o
:
>
1
5
.
2
f
}
 
{
'
-
-
'
:
>
1
5
}
"
)

 
 
 
 
p
r
i
n
t
(
f
"
{
'
T
o
t
a
l
 
T
r
a
d
e
s
'
:
<
2
5
}
 
{
l
e
n
(
t
r
a
d
e
s
_
o
b
j
)
:
>
1
5
}
 
{
'
1
'
:
>
1
5
}
"
)

 
 
 
 
p
r
i
n
t
(
"
=
"
 
*
 
7
0
)


 
 
 
 
#
 
A
n
n
o
t
a
t
e
d
 
e
q
u
i
t
y
 
c
u
r
v
e
 
+
 
d
r
a
w
d
o
w
n

 
 
 
 
f
i
g
,
 
(
a
x
1
,
 
a
x
2
)
 
=
 
p
l
t
.
s
u
b
p
l
o
t
s
(
2
,
 
1
,
 
f
i
g
s
i
z
e
=
(
1
6
,
 
1
0
)
,
 
g
r
i
d
s
p
e
c
_
k
w
=
{
'
h
e
i
g
h
t
_
r
a
t
i
o
s
'
:
 
[
3
,
 
1
]
}
)


 
 
 
 
e
q
_
s
t
r
a
t
 
=
 
p
f
.
v
a
l
u
e
(
)
;
 
e
q
_
b
h
 
=
 
p
f
_
b
h
.
v
a
l
u
e
(
)

 
 
 
 
a
x
1
.
p
l
o
t
(
f
u
l
l
_
c
l
o
s
e
.
i
n
d
e
x
[
:
s
p
l
i
t
_
i
d
x
]
,
 
e
q
_
s
t
r
a
t
.
i
l
o
c
[
:
s
p
l
i
t
_
i
d
x
]
.
v
a
l
u
e
s
,
 
c
o
l
o
r
=
'
#
3
4
9
8
d
b
'
,
 
l
i
n
e
w
i
d
t
h
=
1
.
5
,
 
l
a
b
e
l
=
'
S
t
r
a
t
e
g
y
 
(
I
S
)
'
)

 
 
 
 
a
x
1
.
p
l
o
t
(
f
u
l
l
_
c
l
o
s
e
.
i
n
d
e
x
[
s
p
l
i
t
_
i
d
x
:
]
,
 
e
q
_
s
t
r
a
t
.
i
l
o
c
[
s
p
l
i
t
_
i
d
x
:
]
.
v
a
l
u
e
s
,
 
c
o
l
o
r
=
'
#
e
6
7
e
2
2
'
,
 
l
i
n
e
w
i
d
t
h
=
1
.
5
,
 
l
a
b
e
l
=
'
S
t
r
a
t
e
g
y
 
(
O
O
S
)
'
)

 
 
 
 
a
x
1
.
p
l
o
t
(
f
u
l
l
_
c
l
o
s
e
.
i
n
d
e
x
,
 
e
q
_
b
h
.
v
a
l
u
e
s
,
 
c
o
l
o
r
=
'
g
r
a
y
'
,
 
l
i
n
e
w
i
d
t
h
=
1
,
 
a
l
p
h
a
=
0
.
5
,
 
l
i
n
e
s
t
y
l
e
=
'
-
-
'
,
 
l
a
b
e
l
=
'
B
u
y
 
&
 
H
o
l
d
'
)

 
 
 
 
a
x
1
.
a
x
v
l
i
n
e
(
x
=
f
u
l
l
_
c
l
o
s
e
.
i
n
d
e
x
[
s
p
l
i
t
_
i
d
x
]
,
 
c
o
l
o
r
=
'
r
e
d
'
,
 
l
i
n
e
s
t
y
l
e
=
'
:
'
,
 
a
l
p
h
a
=
0
.
4
,
 
l
a
b
e
l
=
'
T
r
a
i
n
/
V
a
l
 
S
p
l
i
t
'
)

 
 
 
 
a
x
1
.
s
e
t
_
t
i
t
l
e
(
f
'
L
o
s
t
 
M
o
m
e
n
t
u
m
 
{
b
_
m
t
}
(
m
a
=
{
b
_
m
p
}
,
 
s
l
o
p
e
_
l
b
=
{
b
_
s
l
}
,
 
s
w
i
n
g
_
l
b
=
{
b
_
s
w
}
)
 
v
s
 
B
u
y
 
&
 
H
o
l
d
 
-
 
{
T
I
C
K
E
R
}
'
,
 
f
o
n
t
s
i
z
e
=
1
4
,
 
f
o
n
t
w
e
i
g
h
t
=
'
b
o
l
d
'
)

 
 
 
 
a
x
1
.
s
e
t
_
y
l
a
b
e
l
(
'
P
o
r
t
f
o
l
i
o
 
V
a
l
u
e
 
(
$
)
'
,
 
f
o
n
t
s
i
z
e
=
1
2
)

 
 
 
 
a
x
1
.
y
a
x
i
s
.
s
e
t
_
m
a
j
o
r
_
f
o
r
m
a
t
t
e
r
(
p
l
t
.
F
u
n
c
F
o
r
m
a
t
t
e
r
(
l
a
m
b
d
a
 
x
,
 
_
:
 
f
'
$
{
x
:
,
.
0
f
}
'
)
)

 
 
 
 
a
x
1
.
l
e
g
e
n
d
(
l
o
c
=
'
u
p
p
e
r
 
l
e
f
t
'
,
 
f
o
n
t
s
i
z
e
=
1
0
)

 
 
 
 
a
x
1
.
g
r
i
d
(
T
r
u
e
,
 
a
l
p
h
a
=
0
.
3
)


 
 
 
 
s
t
a
t
s
_
t
e
x
t
 
=
 
(
f
"
S
h
a
r
p
e
:
 
{
s
t
r
a
t
_
s
h
a
r
p
e
:
.
3
f
}
 
|
 
R
e
t
u
r
n
:
 
{
s
t
r
a
t
_
r
e
t
u
r
n
:
.
2
%
}
 
|
 
"

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
f
"
M
a
x
D
D
:
 
{
s
t
r
a
t
_
d
d
:
.
2
%
}
 
|
 
W
R
:
 
{
w
r
:
.
1
f
}
%
 
|
 
P
F
:
 
{
p
f
_
r
a
t
i
o
:
.
2
f
}
 
|
 
T
r
a
d
e
s
:
 
{
l
e
n
(
t
r
a
d
e
s
_
o
b
j
)
}
"
)

 
 
 
 
a
x
1
.
t
e
x
t
(
0
.
5
,
 
0
.
0
2
,
 
s
t
a
t
s
_
t
e
x
t
,
 
t
r
a
n
s
f
o
r
m
=
a
x
1
.
t
r
a
n
s
A
x
e
s
,
 
f
o
n
t
s
i
z
e
=
9
,
 
h
a
=
'
c
e
n
t
e
r
'
,

 
 
 
 
 
 
 
 
 
 
 
 
 
b
b
o
x
=
d
i
c
t
(
b
o
x
s
t
y
l
e
=
'
r
o
u
n
d
,
p
a
d
=
0
.
4
'
,
 
f
a
c
e
c
o
l
o
r
=
'
l
i
g
h
t
y
e
l
l
o
w
'
,
 
e
d
g
e
c
o
l
o
r
=
'
g
r
a
y
'
,
 
a
l
p
h
a
=
0
.
9
)
)


 
 
 
 
d
d
 
=
 
p
f
.
d
r
a
w
d
o
w
n
(
)
 
*
 
1
0
0

 
 
 
 
a
x
2
.
f
i
l
l
_
b
e
t
w
e
e
n
(
f
u
l
l
_
c
l
o
s
e
.
i
n
d
e
x
,
 
d
d
.
v
a
l
u
e
s
,
 
0
,
 
c
o
l
o
r
=
'
#
e
7
4
c
3
c
'
,
 
a
l
p
h
a
=
0
.
5
)

 
 
 
 
a
x
2
.
s
e
t
_
y
l
a
b
e
l
(
'
D
r
a
w
d
o
w
n
 
%
'
,
 
f
o
n
t
s
i
z
e
=
1
2
)

 
 
 
 
a
x
2
.
s
e
t
_
x
l
a
b
e
l
(
'
D
a
t
e
'
,
 
f
o
n
t
s
i
z
e
=
1
2
)

 
 
 
 
a
x
2
.
g
r
i
d
(
T
r
u
e
,
 
a
l
p
h
a
=
0
.
3
)


 
 
 
 
p
l
t
.
t
i
g
h
t
_
l
a
y
o
u
t
(
)

 
 
 
 
p
l
t
.
s
h
o
w
(
)

In [ ]:
#
 
T
R
A
D
E
-
B
Y
-
T
R
A
D
E
 
P
R
O
F
I
T
 
A
N
A
L
Y
S
I
S


i
f
 
r
e
s
u
l
t
s
_
d
f
.
e
m
p
t
y
:

 
 
 
 
p
r
i
n
t
(
"
N
o
 
r
e
s
u
l
t
s
.
"
)

e
l
s
e
:

 
 
 
 
b
e
s
t
 
=
 
r
e
s
u
l
t
s
_
d
f
.
l
o
c
[
r
e
s
u
l
t
s
_
d
f
[
'
s
h
a
r
p
e
_
r
a
t
i
o
'
]
.
i
d
x
m
a
x
(
)
]

 
 
 
 
b
_
m
p
 
=
 
i
n
t
(
b
e
s
t
[
'
m
a
_
p
e
r
i
o
d
'
]
)
;
 
b
_
m
t
 
=
 
b
e
s
t
[
'
m
a
_
t
y
p
e
'
]

 
 
 
 
b
_
s
l
 
=
 
i
n
t
(
b
e
s
t
[
'
s
l
o
p
e
_
l
o
o
k
b
a
c
k
'
]
)
;
 
b
_
s
w
 
=
 
i
n
t
(
b
e
s
t
[
'
s
w
i
n
g
_
l
o
o
k
b
a
c
k
'
]
)


 
 
 
 
#
 
R
e
-
d
e
r
i
v
e
 
f
u
l
l
 
c
l
o
s
e
 
a
n
d
 
l
o
w

 
 
 
 
i
f
 
i
s
i
n
s
t
a
n
c
e
(
s
t
o
c
k
_
d
a
t
a
.
c
o
l
u
m
n
s
,
 
p
d
.
M
u
l
t
i
I
n
d
e
x
)
:

 
 
 
 
 
 
 
 
f
u
l
l
_
c
l
o
s
e
 
=
 
s
t
o
c
k
_
d
a
t
a
[
(
'
C
l
o
s
e
'
,
 
T
I
C
K
E
R
)
]
.
a
s
t
y
p
e
(
f
l
o
a
t
)
.
s
q
u
e
e
z
e
(
)

 
 
 
 
 
 
 
 
f
u
l
l
_
l
o
w
 
=
 
s
t
o
c
k
_
d
a
t
a
[
(
'
L
o
w
'
,
 
T
I
C
K
E
R
)
]
.
a
s
t
y
p
e
(
f
l
o
a
t
)
.
s
q
u
e
e
z
e
(
)

 
 
 
 
e
l
s
e
:

 
 
 
 
 
 
 
 
f
u
l
l
_
c
l
o
s
e
 
=
 
s
t
o
c
k
_
d
a
t
a
[
'
C
l
o
s
e
'
]
.
a
s
t
y
p
e
(
f
l
o
a
t
)
.
s
q
u
e
e
z
e
(
)

 
 
 
 
 
 
 
 
f
u
l
l
_
l
o
w
 
=
 
s
t
o
c
k
_
d
a
t
a
[
'
L
o
w
'
]
.
a
s
t
y
p
e
(
f
l
o
a
t
)
.
s
q
u
e
e
z
e
(
)

 
 
 
 
f
u
l
l
_
c
l
o
s
e
.
n
a
m
e
 
=
 
'
p
r
i
c
e
'
;
 
f
u
l
l
_
l
o
w
.
n
a
m
e
 
=
 
'
l
o
w
'


 
 
 
 
#
 
S
i
g
n
a
l
s

 
 
 
 
e
_
f
u
l
l
,
 
x
_
f
u
l
l
 
=
 
_
l
o
s
t
_
m
o
m
e
n
t
u
m
_
s
i
g
n
a
l
s
(
f
u
l
l
_
c
l
o
s
e
,
 
f
u
l
l
_
l
o
w
,
 
b
_
m
p
,
 
b
_
m
t
,
 
b
_
s
l
,
 
b
_
s
w
)

 
 
 
 
p
f
 
=
 
v
b
t
.
P
o
r
t
f
o
l
i
o
.
f
r
o
m
_
s
i
g
n
a
l
s
(
c
l
o
s
e
=
f
u
l
l
_
c
l
o
s
e
,
 
e
n
t
r
i
e
s
=
e
_
f
u
l
l
,
 
e
x
i
t
s
=
x
_
f
u
l
l
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
i
n
i
t
_
c
a
s
h
=
1
0
0
_
0
0
0
,
 
f
e
e
s
=
0
.
0
0
0
5
,
 
s
l
i
p
p
a
g
e
=
0
.
0
0
0
5
,
 
f
r
e
q
=
'
D
'
)


 
 
 
 
t
r
a
d
e
s
_
o
b
j
 
=
 
p
f
.
t
r
a
d
e
s

 
 
 
 
t
r
a
d
e
_
r
e
t
u
r
n
s
 
=
 
t
r
a
d
e
s
_
o
b
j
.
r
e
t
u
r
n
s
.
v
a
l
u
e
s
 
i
f
 
h
a
s
a
t
t
r
(
t
r
a
d
e
s
_
o
b
j
.
r
e
t
u
r
n
s
,
 
'
v
a
l
u
e
s
'
)
 
e
l
s
e
 
n
p
.
a
s
a
r
r
a
y
(
t
r
a
d
e
s
_
o
b
j
.
r
e
t
u
r
n
s
)

 
 
 
 
t
r
a
d
e
_
r
e
t
u
r
n
s
 
=
 
n
p
.
a
s
a
r
r
a
y
(
t
r
a
d
e
_
r
e
t
u
r
n
s
)
.
r
a
v
e
l
(
)

 
 
 
 
t
r
a
d
e
_
p
n
l
 
=
 
t
r
a
d
e
s
_
o
b
j
.
p
n
l
.
v
a
l
u
e
s
 
i
f
 
h
a
s
a
t
t
r
(
t
r
a
d
e
s
_
o
b
j
.
p
n
l
,
 
'
v
a
l
u
e
s
'
)
 
e
l
s
e
 
n
p
.
a
s
a
r
r
a
y
(
t
r
a
d
e
s
_
o
b
j
.
p
n
l
)

 
 
 
 
t
r
a
d
e
_
p
n
l
 
=
 
n
p
.
a
s
a
r
r
a
y
(
t
r
a
d
e
_
p
n
l
)
.
r
a
v
e
l
(
)


 
 
 
 
i
f
 
t
r
a
d
e
_
r
e
t
u
r
n
s
.
s
i
z
e
 
=
=
 
0
:

 
 
 
 
 
 
 
 
p
r
i
n
t
(
"
N
o
 
t
r
a
d
e
s
 
t
o
 
p
l
o
t
.
"
)

 
 
 
 
e
l
s
e
:

 
 
 
 
 
 
 
 
n
_
t
r
a
d
e
s
 
=
 
l
e
n
(
t
r
a
d
e
_
r
e
t
u
r
n
s
)

 
 
 
 
 
 
 
 
w
i
n
n
e
r
s
 
=
 
t
r
a
d
e
_
r
e
t
u
r
n
s
 
>
 
0

 
 
 
 
 
 
 
 
l
o
s
e
r
s
 
=
 
t
r
a
d
e
_
r
e
t
u
r
n
s
 
<
 
0


 
 
 
 
 
 
 
 
f
i
g
,
 
a
x
e
s
 
=
 
p
l
t
.
s
u
b
p
l
o
t
s
(
2
,
 
2
,
 
f
i
g
s
i
z
e
=
(
1
8
,
 
1
2
)
)

 
 
 
 
 
 
 
 
f
i
g
.
s
u
p
t
i
t
l
e
(
f
'
T
r
a
d
e
-
b
y
-
T
r
a
d
e
 
A
n
a
l
y
s
i
s
 
\
u
2
0
1
4
 
L
o
s
t
 
M
o
m
e
n
t
u
m
 
{
b
_
m
t
}
(
m
a
=
{
b
_
m
p
}
,
 
s
l
=
{
b
_
s
l
}
,
 
s
w
=
{
b
_
s
w
}
)
 
\
u
2
0
1
4
 
{
n
_
t
r
a
d
e
s
}
 
T
r
a
d
e
s
'
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
f
o
n
t
s
i
z
e
=
1
6
,
 
f
o
n
t
w
e
i
g
h
t
=
'
b
o
l
d
'
)


 
 
 
 
 
 
 
 
#
 
(
0
,
0
)
 
I
n
d
i
v
i
d
u
a
l
 
t
r
a
d
e
 
r
e
t
u
r
n
s
 
%

 
 
 
 
 
 
 
 
c
o
l
o
r
s
 
=
 
[
'
#
2
c
a
0
2
c
'
 
i
f
 
r
 
>
 
0
 
e
l
s
e
 
'
#
d
6
2
7
2
8
'
 
f
o
r
 
r
 
i
n
 
t
r
a
d
e
_
r
e
t
u
r
n
s
]

 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
 
0
]
.
b
a
r
(
r
a
n
g
e
(
n
_
t
r
a
d
e
s
)
,
 
t
r
a
d
e
_
r
e
t
u
r
n
s
 
*
 
1
0
0
,
 
c
o
l
o
r
=
c
o
l
o
r
s
,
 
e
d
g
e
c
o
l
o
r
=
'
b
l
a
c
k
'
,
 
l
i
n
e
w
i
d
t
h
=
0
.
4
)

 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
 
0
]
.
a
x
h
l
i
n
e
(
0
,
 
c
o
l
o
r
=
'
b
l
a
c
k
'
,
 
l
i
n
e
w
i
d
t
h
=
1
)

 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
 
0
]
.
a
x
h
l
i
n
e
(
n
p
.
m
e
a
n
(
t
r
a
d
e
_
r
e
t
u
r
n
s
)
 
*
 
1
0
0
,
 
c
o
l
o
r
=
'
b
l
u
e
'
,
 
l
i
n
e
s
t
y
l
e
=
'
-
-
'
,
 
l
i
n
e
w
i
d
t
h
=
1
.
5
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
l
a
b
e
l
=
f
'
A
v
g
:
 
{
n
p
.
m
e
a
n
(
t
r
a
d
e
_
r
e
t
u
r
n
s
)
*
1
0
0
:
.
2
f
}
%
'
)

 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
 
0
]
.
s
e
t
_
t
i
t
l
e
(
'
I
n
d
i
v
i
d
u
a
l
 
T
r
a
d
e
 
R
e
t
u
r
n
s
 
(
%
)
'
,
 
f
o
n
t
s
i
z
e
=
1
3
,
 
f
o
n
t
w
e
i
g
h
t
=
'
b
o
l
d
'
)

 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
 
0
]
.
s
e
t
_
x
l
a
b
e
l
(
'
T
r
a
d
e
 
#
'
,
 
f
o
n
t
s
i
z
e
=
1
1
)

 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
 
0
]
.
s
e
t
_
y
l
a
b
e
l
(
'
R
e
t
u
r
n
 
%
'
,
 
f
o
n
t
s
i
z
e
=
1
1
)

 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
 
0
]
.
l
e
g
e
n
d
(
f
o
n
t
s
i
z
e
=
1
0
)

 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
 
0
]
.
g
r
i
d
(
a
x
i
s
=
'
y
'
,
 
a
l
p
h
a
=
0
.
3
)


 
 
 
 
 
 
 
 
#
 
(
0
,
1
)
 
I
n
d
i
v
i
d
u
a
l
 
t
r
a
d
e
 
P
&
L
 
(
$
)

 
 
 
 
 
 
 
 
c
o
l
o
r
s
_
p
n
l
 
=
 
[
'
#
2
c
a
0
2
c
'
 
i
f
 
p
 
>
 
0
 
e
l
s
e
 
'
#
d
6
2
7
2
8
'
 
f
o
r
 
p
 
i
n
 
t
r
a
d
e
_
p
n
l
]

 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
 
1
]
.
b
a
r
(
r
a
n
g
e
(
n
_
t
r
a
d
e
s
)
,
 
t
r
a
d
e
_
p
n
l
,
 
c
o
l
o
r
=
c
o
l
o
r
s
_
p
n
l
,
 
e
d
g
e
c
o
l
o
r
=
'
b
l
a
c
k
'
,
 
l
i
n
e
w
i
d
t
h
=
0
.
4
)

 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
 
1
]
.
a
x
h
l
i
n
e
(
0
,
 
c
o
l
o
r
=
'
b
l
a
c
k
'
,
 
l
i
n
e
w
i
d
t
h
=
1
)

 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
 
1
]
.
a
x
h
l
i
n
e
(
n
p
.
m
e
a
n
(
t
r
a
d
e
_
p
n
l
)
,
 
c
o
l
o
r
=
'
b
l
u
e
'
,
 
l
i
n
e
s
t
y
l
e
=
'
-
-
'
,
 
l
i
n
e
w
i
d
t
h
=
1
.
5
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
l
a
b
e
l
=
f
'
A
v
g
:
 
$
{
n
p
.
m
e
a
n
(
t
r
a
d
e
_
p
n
l
)
:
,
.
0
f
}
'
)

 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
 
1
]
.
s
e
t
_
t
i
t
l
e
(
'
I
n
d
i
v
i
d
u
a
l
 
T
r
a
d
e
 
P
&
L
 
(
$
)
'
,
 
f
o
n
t
s
i
z
e
=
1
3
,
 
f
o
n
t
w
e
i
g
h
t
=
'
b
o
l
d
'
)

 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
 
1
]
.
s
e
t
_
x
l
a
b
e
l
(
'
T
r
a
d
e
 
#
'
,
 
f
o
n
t
s
i
z
e
=
1
1
)

 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
 
1
]
.
s
e
t
_
y
l
a
b
e
l
(
'
P
&
L
 
(
$
)
'
,
 
f
o
n
t
s
i
z
e
=
1
1
)

 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
 
1
]
.
y
a
x
i
s
.
s
e
t
_
m
a
j
o
r
_
f
o
r
m
a
t
t
e
r
(
p
l
t
.
F
u
n
c
F
o
r
m
a
t
t
e
r
(
l
a
m
b
d
a
 
x
,
 
_
:
 
f
'
$
{
x
:
,
.
0
f
}
'
)
)

 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
 
1
]
.
l
e
g
e
n
d
(
f
o
n
t
s
i
z
e
=
1
0
)

 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
 
1
]
.
g
r
i
d
(
a
x
i
s
=
'
y
'
,
 
a
l
p
h
a
=
0
.
3
)


 
 
 
 
 
 
 
 
#
 
(
1
,
0
)
 
C
u
m
u
l
a
t
i
v
e
 
P
&
L

 
 
 
 
 
 
 
 
c
u
m
_
p
n
l
 
=
 
n
p
.
c
u
m
s
u
m
(
t
r
a
d
e
_
p
n
l
)

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
 
0
]
.
p
l
o
t
(
r
a
n
g
e
(
1
,
 
n
_
t
r
a
d
e
s
 
+
 
1
)
,
 
c
u
m
_
p
n
l
,
 
c
o
l
o
r
=
'
#
1
f
7
7
b
4
'
,
 
l
i
n
e
w
i
d
t
h
=
2
,
 
m
a
r
k
e
r
=
'
o
'
,
 
m
a
r
k
e
r
s
i
z
e
=
4
)

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
 
0
]
.
f
i
l
l
_
b
e
t
w
e
e
n
(
r
a
n
g
e
(
1
,
 
n
_
t
r
a
d
e
s
 
+
 
1
)
,
 
c
u
m
_
p
n
l
,
 
0
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
w
h
e
r
e
=
c
u
m
_
p
n
l
 
>
=
 
0
,
 
a
l
p
h
a
=
0
.
1
5
,
 
c
o
l
o
r
=
'
g
r
e
e
n
'
)

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
 
0
]
.
f
i
l
l
_
b
e
t
w
e
e
n
(
r
a
n
g
e
(
1
,
 
n
_
t
r
a
d
e
s
 
+
 
1
)
,
 
c
u
m
_
p
n
l
,
 
0
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
w
h
e
r
e
=
c
u
m
_
p
n
l
 
<
 
0
,
 
a
l
p
h
a
=
0
.
1
5
,
 
c
o
l
o
r
=
'
r
e
d
'
)

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
 
0
]
.
a
x
h
l
i
n
e
(
0
,
 
c
o
l
o
r
=
'
b
l
a
c
k
'
,
 
l
i
n
e
w
i
d
t
h
=
1
)

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
 
0
]
.
s
e
t
_
t
i
t
l
e
(
'
C
u
m
u
l
a
t
i
v
e
 
P
&
L
 
(
T
r
a
d
e
-
b
y
-
T
r
a
d
e
 
E
q
u
i
t
y
)
'
,
 
f
o
n
t
s
i
z
e
=
1
3
,
 
f
o
n
t
w
e
i
g
h
t
=
'
b
o
l
d
'
)

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
 
0
]
.
s
e
t
_
x
l
a
b
e
l
(
'
T
r
a
d
e
 
#
'
,
 
f
o
n
t
s
i
z
e
=
1
1
)

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
 
0
]
.
s
e
t
_
y
l
a
b
e
l
(
'
C
u
m
u
l
a
t
i
v
e
 
P
&
L
 
(
$
)
'
,
 
f
o
n
t
s
i
z
e
=
1
1
)

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
 
0
]
.
y
a
x
i
s
.
s
e
t
_
m
a
j
o
r
_
f
o
r
m
a
t
t
e
r
(
p
l
t
.
F
u
n
c
F
o
r
m
a
t
t
e
r
(
l
a
m
b
d
a
 
x
,
 
_
:
 
f
'
$
{
x
:
,
.
0
f
}
'
)
)

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
 
0
]
.
g
r
i
d
(
T
r
u
e
,
 
a
l
p
h
a
=
0
.
3
)


 
 
 
 
 
 
 
 
#
 
(
1
,
1
)
 
R
e
t
u
r
n
 
d
i
s
t
r
i
b
u
t
i
o
n
 
h
i
s
t
o
g
r
a
m

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
 
1
]
.
h
i
s
t
(
t
r
a
d
e
_
r
e
t
u
r
n
s
 
*
 
1
0
0
,
 
b
i
n
s
=
m
i
n
(
3
0
,
 
m
a
x
(
1
0
,
 
n
_
t
r
a
d
e
s
 
/
/
 
3
)
)
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
c
o
l
o
r
=
'
#
1
f
7
7
b
4
'
,
 
e
d
g
e
c
o
l
o
r
=
'
b
l
a
c
k
'
,
 
a
l
p
h
a
=
0
.
7
)

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
 
1
]
.
a
x
v
l
i
n
e
(
0
,
 
c
o
l
o
r
=
'
b
l
a
c
k
'
,
 
l
i
n
e
w
i
d
t
h
=
1
.
5
)

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
 
1
]
.
a
x
v
l
i
n
e
(
n
p
.
m
e
a
n
(
t
r
a
d
e
_
r
e
t
u
r
n
s
)
 
*
 
1
0
0
,
 
c
o
l
o
r
=
'
r
e
d
'
,
 
l
i
n
e
s
t
y
l
e
=
'
-
-
'
,
 
l
i
n
e
w
i
d
t
h
=
2
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
l
a
b
e
l
=
f
'
M
e
a
n
:
 
{
n
p
.
m
e
a
n
(
t
r
a
d
e
_
r
e
t
u
r
n
s
)
*
1
0
0
:
.
2
f
}
%
'
)

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
 
1
]
.
a
x
v
l
i
n
e
(
n
p
.
m
e
d
i
a
n
(
t
r
a
d
e
_
r
e
t
u
r
n
s
)
 
*
 
1
0
0
,
 
c
o
l
o
r
=
'
o
r
a
n
g
e
'
,
 
l
i
n
e
s
t
y
l
e
=
'
-
-
'
,
 
l
i
n
e
w
i
d
t
h
=
2
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
l
a
b
e
l
=
f
'
M
e
d
i
a
n
:
 
{
n
p
.
m
e
d
i
a
n
(
t
r
a
d
e
_
r
e
t
u
r
n
s
)
*
1
0
0
:
.
2
f
}
%
'
)

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
 
1
]
.
s
e
t
_
t
i
t
l
e
(
'
T
r
a
d
e
 
R
e
t
u
r
n
 
D
i
s
t
r
i
b
u
t
i
o
n
'
,
 
f
o
n
t
s
i
z
e
=
1
3
,
 
f
o
n
t
w
e
i
g
h
t
=
'
b
o
l
d
'
)

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
 
1
]
.
s
e
t
_
x
l
a
b
e
l
(
'
R
e
t
u
r
n
 
%
'
,
 
f
o
n
t
s
i
z
e
=
1
1
)

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
 
1
]
.
s
e
t
_
y
l
a
b
e
l
(
'
F
r
e
q
u
e
n
c
y
'
,
 
f
o
n
t
s
i
z
e
=
1
1
)

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
 
1
]
.
l
e
g
e
n
d
(
f
o
n
t
s
i
z
e
=
1
0
)

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
 
1
]
.
g
r
i
d
(
a
x
i
s
=
'
y
'
,
 
a
l
p
h
a
=
0
.
3
)


 
 
 
 
 
 
 
 
p
l
t
.
t
i
g
h
t
_
l
a
y
o
u
t
(
)

 
 
 
 
 
 
 
 
p
l
t
.
s
h
o
w
(
)


 
 
 
 
 
 
 
 
#
 
S
u
m
m
a
r
y
 
S
t
a
t
s

 
 
 
 
 
 
 
 
w
i
n
_
r
e
t
s
 
=
 
t
r
a
d
e
_
r
e
t
u
r
n
s
[
w
i
n
n
e
r
s
]

 
 
 
 
 
 
 
 
l
o
s
s
_
r
e
t
s
 
=
 
t
r
a
d
e
_
r
e
t
u
r
n
s
[
l
o
s
e
r
s
]

 
 
 
 
 
 
 
 
p
r
i
n
t
(
"
\
n
"
 
+
 
"
=
"
 
*
 
6
0
)

 
 
 
 
 
 
 
 
p
r
i
n
t
(
"
\
U
0
0
0
1
f
4
c
b
 
T
R
A
D
E
 
S
T
A
T
I
S
T
I
C
S
 
S
U
M
M
A
R
Y
"
)

 
 
 
 
 
 
 
 
p
r
i
n
t
(
"
=
"
 
*
 
6
0
)

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
 
 
T
o
t
a
l
 
T
r
a
d
e
s
:
 
 
 
 
 
 
{
n
_
t
r
a
d
e
s
}
"
)

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
 
 
W
i
n
n
e
r
s
:
 
 
 
 
 
 
 
 
 
 
 
{
w
i
n
n
e
r
s
.
s
u
m
(
)
}
 
(
{
w
i
n
n
e
r
s
.
s
u
m
(
)
/
n
_
t
r
a
d
e
s
*
1
0
0
:
.
1
f
}
%
)
"
)

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
 
 
L
o
s
e
r
s
:
 
 
 
 
 
 
 
 
 
 
 
 
{
l
o
s
e
r
s
.
s
u
m
(
)
}
 
(
{
l
o
s
e
r
s
.
s
u
m
(
)
/
n
_
t
r
a
d
e
s
*
1
0
0
:
.
1
f
}
%
)
"
)

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
 
 
A
v
g
 
W
i
n
:
 
 
 
 
 
 
 
 
 
 
 
{
w
i
n
_
r
e
t
s
.
m
e
a
n
(
)
*
1
0
0
:
.
2
f
}
%
"
 
i
f
 
l
e
n
(
w
i
n
_
r
e
t
s
)
 
e
l
s
e
 
"
 
 
A
v
g
 
W
i
n
:
 
 
 
 
 
 
 
 
 
 
 
N
/
A
"
)

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
 
 
A
v
g
 
L
o
s
s
:
 
 
 
 
 
 
 
 
 
 
{
l
o
s
s
_
r
e
t
s
.
m
e
a
n
(
)
*
1
0
0
:
.
2
f
}
%
"
 
i
f
 
l
e
n
(
l
o
s
s
_
r
e
t
s
)
 
e
l
s
e
 
"
 
 
A
v
g
 
L
o
s
s
:
 
 
 
 
 
 
 
 
 
 
N
/
A
"
)

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
 
 
L
a
r
g
e
s
t
 
W
i
n
:
 
 
 
 
 
 
 
{
w
i
n
_
r
e
t
s
.
m
a
x
(
)
*
1
0
0
:
.
2
f
}
%
"
 
i
f
 
l
e
n
(
w
i
n
_
r
e
t
s
)
 
e
l
s
e
 
"
 
 
L
a
r
g
e
s
t
 
W
i
n
:
 
 
 
 
 
 
 
N
/
A
"
)

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
 
 
L
a
r
g
e
s
t
 
L
o
s
s
:
 
 
 
 
 
 
{
l
o
s
s
_
r
e
t
s
.
m
i
n
(
)
*
1
0
0
:
.
2
f
}
%
"
 
i
f
 
l
e
n
(
l
o
s
s
_
r
e
t
s
)
 
e
l
s
e
 
"
 
 
L
a
r
g
e
s
t
 
L
o
s
s
:
 
 
 
 
 
 
N
/
A
"
)

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
 
 
A
v
g
 
P
&
L
:
 
 
 
 
 
 
 
 
 
 
 
$
{
n
p
.
m
e
a
n
(
t
r
a
d
e
_
p
n
l
)
:
,
.
2
f
}
"
)

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
 
 
T
o
t
a
l
 
P
&
L
:
 
 
 
 
 
 
 
 
 
$
{
n
p
.
s
u
m
(
t
r
a
d
e
_
p
n
l
)
:
,
.
2
f
}
"
)

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
 
 
P
a
y
o
f
f
 
R
a
t
i
o
:
 
 
 
 
 
 
{
a
b
s
(
w
i
n
_
r
e
t
s
.
m
e
a
n
(
)
/
l
o
s
s
_
r
e
t
s
.
m
e
a
n
(
)
)
:
.
2
f
}
"
 
i
f
 
l
e
n
(
w
i
n
_
r
e
t
s
)
 
a
n
d
 
l
e
n
(
l
o
s
s
_
r
e
t
s
)
 
e
l
s
e
 
"
"
)

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
 
 
E
x
p
e
c
t
a
n
c
y
:
 
 
 
 
 
 
 
 
{
n
p
.
m
e
a
n
(
t
r
a
d
e
_
r
e
t
u
r
n
s
)
*
1
0
0
:
.
3
f
}
%
 
p
e
r
 
t
r
a
d
e
"
)

 
 
 
 
 
 
 
 
p
r
i
n
t
(
"
=
"
 
*
 
6
0
)

In [ ]:
#
 
M
O
N
T
E
 
C
A
R
L
O
 
S
I
M
U
L
A
T
I
O
N
 
—
 
F
T
M
O
 
C
H
A
L
L
E
N
G
E
 
P
A
S
S
 
P
R
O
B
A
B
I
L
I
T
Y


p
r
i
n
t
(
"
=
"
 
*
 
7
0
)

p
r
i
n
t
(
"
\
U
0
0
0
1
f
3
b
2
 
M
O
N
T
E
 
C
A
R
L
O
 
S
I
M
U
L
A
T
I
O
N
 
\
u
2
0
1
4
 
F
T
M
O
 
C
H
A
L
L
E
N
G
E
 
F
E
A
S
I
B
I
L
I
T
Y
"
)

p
r
i
n
t
(
"
=
"
 
*
 
7
0
)


r
e
s
u
l
t
s
_
d
f
 
=
 
p
d
.
D
a
t
a
F
r
a
m
e
(
g
r
i
d
_
s
e
a
r
c
h
_
r
e
s
u
l
t
s
)


i
f
 
r
e
s
u
l
t
s
_
d
f
.
e
m
p
t
y
:

 
 
 
 
p
r
i
n
t
(
"
N
o
 
r
e
s
u
l
t
s
.
"
)

e
l
s
e
:

 
 
 
 
b
e
s
t
 
=
 
r
e
s
u
l
t
s
_
d
f
.
l
o
c
[
r
e
s
u
l
t
s
_
d
f
[
'
s
h
a
r
p
e
_
r
a
t
i
o
'
]
.
i
d
x
m
a
x
(
)
]

 
 
 
 
b
_
m
p
 
=
 
i
n
t
(
b
e
s
t
[
'
m
a
_
p
e
r
i
o
d
'
]
)
;
 
b
_
m
t
 
=
 
b
e
s
t
[
'
m
a
_
t
y
p
e
'
]

 
 
 
 
b
_
s
l
 
=
 
i
n
t
(
b
e
s
t
[
'
s
l
o
p
e
_
l
o
o
k
b
a
c
k
'
]
)
;
 
b
_
s
w
 
=
 
i
n
t
(
b
e
s
t
[
'
s
w
i
n
g
_
l
o
o
k
b
a
c
k
'
]
)


 
 
 
 
#
 
R
e
-
d
e
r
i
v
e

 
 
 
 
i
f
 
i
s
i
n
s
t
a
n
c
e
(
s
t
o
c
k
_
d
a
t
a
.
c
o
l
u
m
n
s
,
 
p
d
.
M
u
l
t
i
I
n
d
e
x
)
:

 
 
 
 
 
 
 
 
f
u
l
l
_
c
l
o
s
e
 
=
 
s
t
o
c
k
_
d
a
t
a
[
(
'
C
l
o
s
e
'
,
 
T
I
C
K
E
R
)
]
.
a
s
t
y
p
e
(
f
l
o
a
t
)
.
s
q
u
e
e
z
e
(
)

 
 
 
 
 
 
 
 
f
u
l
l
_
l
o
w
 
=
 
s
t
o
c
k
_
d
a
t
a
[
(
'
L
o
w
'
,
 
T
I
C
K
E
R
)
]
.
a
s
t
y
p
e
(
f
l
o
a
t
)
.
s
q
u
e
e
z
e
(
)

 
 
 
 
e
l
s
e
:

 
 
 
 
 
 
 
 
f
u
l
l
_
c
l
o
s
e
 
=
 
s
t
o
c
k
_
d
a
t
a
[
'
C
l
o
s
e
'
]
.
a
s
t
y
p
e
(
f
l
o
a
t
)
.
s
q
u
e
e
z
e
(
)

 
 
 
 
 
 
 
 
f
u
l
l
_
l
o
w
 
=
 
s
t
o
c
k
_
d
a
t
a
[
'
L
o
w
'
]
.
a
s
t
y
p
e
(
f
l
o
a
t
)
.
s
q
u
e
e
z
e
(
)

 
 
 
 
f
u
l
l
_
c
l
o
s
e
.
n
a
m
e
 
=
 
'
p
r
i
c
e
'
;
 
f
u
l
l
_
l
o
w
.
n
a
m
e
 
=
 
'
l
o
w
'


 
 
 
 
e
_
f
u
l
l
,
 
x
_
f
u
l
l
 
=
 
_
l
o
s
t
_
m
o
m
e
n
t
u
m
_
s
i
g
n
a
l
s
(
f
u
l
l
_
c
l
o
s
e
,
 
f
u
l
l
_
l
o
w
,
 
b
_
m
p
,
 
b
_
m
t
,
 
b
_
s
l
,
 
b
_
s
w
)

 
 
 
 
p
f
 
=
 
v
b
t
.
P
o
r
t
f
o
l
i
o
.
f
r
o
m
_
s
i
g
n
a
l
s
(
c
l
o
s
e
=
f
u
l
l
_
c
l
o
s
e
,
 
e
n
t
r
i
e
s
=
e
_
f
u
l
l
,
 
e
x
i
t
s
=
x
_
f
u
l
l
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
i
n
i
t
_
c
a
s
h
=
1
0
0
_
0
0
0
,
 
f
e
e
s
=
0
.
0
0
0
5
,
 
s
l
i
p
p
a
g
e
=
0
.
0
0
0
5
,
 
f
r
e
q
=
'
D
'
)


 
 
 
 
#
 
G
e
t
 
d
a
i
l
y
 
r
e
t
u
r
n
s

 
 
 
 
d
a
i
l
y
_
r
e
t
s
 
=
 
p
f
.
r
e
t
u
r
n
s
(
)
.
v
a
l
u
e
s
.
r
a
v
e
l
(
)

 
 
 
 
d
a
i
l
y
_
r
e
t
s
 
=
 
d
a
i
l
y
_
r
e
t
s
[
~
n
p
.
i
s
n
a
n
(
d
a
i
l
y
_
r
e
t
s
)
]


 
 
 
 
#
 
F
T
M
O
 
P
a
r
a
m
e
t
e
r
s

 
 
 
 
F
T
M
O
_
A
C
C
O
U
N
T
 
 
 
 
 
=
 
1
0
0
_
0
0
0

 
 
 
 
P
R
O
F
I
T
_
T
A
R
G
E
T
 
 
 
 
=
 
0
.
1
0

 
 
 
 
M
A
X
_
D
A
I
L
Y
_
L
O
S
S
 
 
 
=
 
0
.
0
5

 
 
 
 
M
A
X
_
T
O
T
A
L
_
L
O
S
S
 
 
 
=
 
0
.
1
0

 
 
 
 
T
R
A
D
I
N
G
_
D
A
Y
S
 
 
 
 
 
=
 
3
0

 
 
 
 
N
_
S
I
M
U
L
A
T
I
O
N
S
 
 
 
 
=
 
1
0
_
0
0
0


 
 
 
 
p
r
i
n
t
(
f
"
\
n
S
t
r
a
t
e
g
y
:
 
L
o
s
t
 
M
o
m
e
n
t
u
m
 
{
b
_
m
t
}
(
m
a
=
{
b
_
m
p
}
,
 
s
l
o
p
e
_
l
b
=
{
b
_
s
l
}
,
 
s
w
i
n
g
_
l
b
=
{
b
_
s
w
}
)
"
)

 
 
 
 
p
r
i
n
t
(
f
"
O
b
s
e
r
v
e
d
 
d
a
i
l
y
 
r
e
t
u
r
n
s
:
 
{
l
e
n
(
d
a
i
l
y
_
r
e
t
s
)
}
 
d
a
y
s
"
)

 
 
 
 
p
r
i
n
t
(
f
"
 
 
M
e
a
n
 
d
a
i
l
y
 
r
e
t
u
r
n
:
 
 
 
 
{
n
p
.
m
e
a
n
(
d
a
i
l
y
_
r
e
t
s
)
*
1
0
0
:
.
4
f
}
%
"
)

 
 
 
 
p
r
i
n
t
(
f
"
 
 
S
t
d
 
d
a
i
l
y
 
r
e
t
u
r
n
:
 
 
 
 
 
{
n
p
.
s
t
d
(
d
a
i
l
y
_
r
e
t
s
)
*
1
0
0
:
.
4
f
}
%
"
)

 
 
 
 
p
r
i
n
t
(
f
"
 
 
S
k
e
w
n
e
s
s
:
 
 
 
 
 
 
 
 
 
 
 
 
 
{
f
l
o
a
t
(
s
t
a
t
s
.
s
k
e
w
(
d
a
i
l
y
_
r
e
t
s
)
)
:
.
3
f
}
"
)

 
 
 
 
p
r
i
n
t
(
f
"
 
 
K
u
r
t
o
s
i
s
:
 
 
 
 
 
 
 
 
 
 
 
 
 
{
f
l
o
a
t
(
s
t
a
t
s
.
k
u
r
t
o
s
i
s
(
d
a
i
l
y
_
r
e
t
s
)
)
:
.
3
f
}
"
)

 
 
 
 
p
r
i
n
t
(
f
"
\
n
F
T
M
O
 
C
h
a
l
l
e
n
g
e
 
R
u
l
e
s
:
"
)

 
 
 
 
p
r
i
n
t
(
f
"
 
 
A
c
c
o
u
n
t
 
S
i
z
e
:
 
 
 
 
 
 
 
 
 
$
{
F
T
M
O
_
A
C
C
O
U
N
T
:
,
.
0
f
}
"
)

 
 
 
 
p
r
i
n
t
(
f
"
 
 
P
r
o
f
i
t
 
T
a
r
g
e
t
:
 
 
 
 
 
 
 
 
{
P
R
O
F
I
T
_
T
A
R
G
E
T
:
.
0
%
}
 
(
$
{
F
T
M
O
_
A
C
C
O
U
N
T
 
*
 
P
R
O
F
I
T
_
T
A
R
G
E
T
:
,
.
0
f
}
)
"
)

 
 
 
 
p
r
i
n
t
(
f
"
 
 
M
a
x
 
D
a
i
l
y
 
L
o
s
s
:
 
 
 
 
 
 
 
{
M
A
X
_
D
A
I
L
Y
_
L
O
S
S
:
.
0
%
}
 
(
$
{
F
T
M
O
_
A
C
C
O
U
N
T
 
*
 
M
A
X
_
D
A
I
L
Y
_
L
O
S
S
:
,
.
0
f
}
)
"
)

 
 
 
 
p
r
i
n
t
(
f
"
 
 
M
a
x
 
T
o
t
a
l
 
L
o
s
s
:
 
 
 
 
 
 
 
{
M
A
X
_
T
O
T
A
L
_
L
O
S
S
:
.
0
%
}
 
(
$
{
F
T
M
O
_
A
C
C
O
U
N
T
 
*
 
M
A
X
_
T
O
T
A
L
_
L
O
S
S
:
,
.
0
f
}
)
"
)

 
 
 
 
p
r
i
n
t
(
f
"
 
 
S
i
m
u
l
a
t
i
o
n
 
W
i
n
d
o
w
:
 
 
 
 
{
T
R
A
D
I
N
G
_
D
A
Y
S
}
 
t
r
a
d
i
n
g
 
d
a
y
s
"
)

 
 
 
 
p
r
i
n
t
(
f
"
 
 
S
i
m
u
l
a
t
i
o
n
s
:
 
 
 
 
 
 
 
 
 
 
{
N
_
S
I
M
U
L
A
T
I
O
N
S
:
,
}
"
)


 
 
 
 
#
 
R
u
n
 
M
o
n
t
e
 
C
a
r
l
o

 
 
 
 
n
p
.
r
a
n
d
o
m
.
s
e
e
d
(
4
2
)

 
 
 
 
e
q
u
i
t
y
_
p
a
t
h
s
 
=
 
n
p
.
z
e
r
o
s
(
(
N
_
S
I
M
U
L
A
T
I
O
N
S
,
 
T
R
A
D
I
N
G
_
D
A
Y
S
 
+
 
1
)
)

 
 
 
 
e
q
u
i
t
y
_
p
a
t
h
s
[
:
,
 
0
]
 
=
 
F
T
M
O
_
A
C
C
O
U
N
T


 
 
 
 
p
a
s
s
e
d
 
 
 
 
 
 
=
 
n
p
.
z
e
r
o
s
(
N
_
S
I
M
U
L
A
T
I
O
N
S
,
 
d
t
y
p
e
=
b
o
o
l
)

 
 
 
 
b
l
o
w
n
 
 
 
 
 
 
 
=
 
n
p
.
z
e
r
o
s
(
N
_
S
I
M
U
L
A
T
I
O
N
S
,
 
d
t
y
p
e
=
b
o
o
l
)

 
 
 
 
d
a
i
l
y
_
b
l
o
w
n
 
=
 
n
p
.
z
e
r
o
s
(
N
_
S
I
M
U
L
A
T
I
O
N
S
,
 
d
t
y
p
e
=
b
o
o
l
)

 
 
 
 
f
i
n
a
l
_
e
q
u
i
t
y
 
=
 
n
p
.
z
e
r
o
s
(
N
_
S
I
M
U
L
A
T
I
O
N
S
)


 
 
 
 
f
o
r
 
s
i
m
 
i
n
 
r
a
n
g
e
(
N
_
S
I
M
U
L
A
T
I
O
N
S
)
:

 
 
 
 
 
 
 
 
s
i
m
_
r
e
t
s
 
=
 
n
p
.
r
a
n
d
o
m
.
c
h
o
i
c
e
(
d
a
i
l
y
_
r
e
t
s
,
 
s
i
z
e
=
T
R
A
D
I
N
G
_
D
A
Y
S
,
 
r
e
p
l
a
c
e
=
T
r
u
e
)

 
 
 
 
 
 
 
 
e
q
 
=
 
F
T
M
O
_
A
C
C
O
U
N
T

 
 
 
 
 
 
 
 
p
e
a
k
_
e
q
 
=
 
F
T
M
O
_
A
C
C
O
U
N
T

 
 
 
 
 
 
 
 
d
a
y
_
s
t
a
r
t
_
e
q
 
=
 
F
T
M
O
_
A
C
C
O
U
N
T

 
 
 
 
 
 
 
 
s
i
m
_
p
a
s
s
e
d
 
=
 
F
a
l
s
e

 
 
 
 
 
 
 
 
s
i
m
_
b
l
o
w
n
_
t
o
t
a
l
 
=
 
F
a
l
s
e

 
 
 
 
 
 
 
 
s
i
m
_
b
l
o
w
n
_
d
a
i
l
y
 
=
 
F
a
l
s
e


 
 
 
 
 
 
 
 
f
o
r
 
d
a
y
 
i
n
 
r
a
n
g
e
(
T
R
A
D
I
N
G
_
D
A
Y
S
)
:

 
 
 
 
 
 
 
 
 
 
 
 
d
a
y
_
s
t
a
r
t
_
e
q
 
=
 
e
q

 
 
 
 
 
 
 
 
 
 
 
 
e
q
 
=
 
e
q
 
*
 
(
1
 
+
 
s
i
m
_
r
e
t
s
[
d
a
y
]
)

 
 
 
 
 
 
 
 
 
 
 
 
e
q
u
i
t
y
_
p
a
t
h
s
[
s
i
m
,
 
d
a
y
 
+
 
1
]
 
=
 
e
q


 
 
 
 
 
 
 
 
 
 
 
 
d
a
i
l
y
_
l
o
s
s
 
=
 
(
e
q
 
-
 
d
a
y
_
s
t
a
r
t
_
e
q
)
 
/
 
F
T
M
O
_
A
C
C
O
U
N
T

 
 
 
 
 
 
 
 
 
 
 
 
i
f
 
d
a
i
l
y
_
l
o
s
s
 
<
 
-
M
A
X
_
D
A
I
L
Y
_
L
O
S
S
:

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
s
i
m
_
b
l
o
w
n
_
d
a
i
l
y
 
=
 
T
r
u
e

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
e
q
u
i
t
y
_
p
a
t
h
s
[
s
i
m
,
 
d
a
y
 
+
 
2
:
]
 
=
 
e
q

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
b
r
e
a
k


 
 
 
 
 
 
 
 
 
 
 
 
t
o
t
a
l
_
l
o
s
s
 
=
 
(
e
q
 
-
 
F
T
M
O
_
A
C
C
O
U
N
T
)
 
/
 
F
T
M
O
_
A
C
C
O
U
N
T

 
 
 
 
 
 
 
 
 
 
 
 
i
f
 
t
o
t
a
l
_
l
o
s
s
 
<
 
-
M
A
X
_
T
O
T
A
L
_
L
O
S
S
:

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
s
i
m
_
b
l
o
w
n
_
t
o
t
a
l
 
=
 
T
r
u
e

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
e
q
u
i
t
y
_
p
a
t
h
s
[
s
i
m
,
 
d
a
y
 
+
 
2
:
]
 
=
 
e
q

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
b
r
e
a
k


 
 
 
 
 
 
 
 
 
 
 
 
t
o
t
a
l
_
g
a
i
n
 
=
 
(
e
q
 
-
 
F
T
M
O
_
A
C
C
O
U
N
T
)
 
/
 
F
T
M
O
_
A
C
C
O
U
N
T

 
 
 
 
 
 
 
 
 
 
 
 
i
f
 
t
o
t
a
l
_
g
a
i
n
 
>
=
 
P
R
O
F
I
T
_
T
A
R
G
E
T
:

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
s
i
m
_
p
a
s
s
e
d
 
=
 
T
r
u
e

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
e
q
u
i
t
y
_
p
a
t
h
s
[
s
i
m
,
 
d
a
y
 
+
 
2
:
]
 
=
 
e
q

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
b
r
e
a
k


 
 
 
 
 
 
 
 
f
i
n
a
l
_
e
q
u
i
t
y
[
s
i
m
]
 
=
 
e
q
u
i
t
y
_
p
a
t
h
s
[
s
i
m
,
 
-
1
]

 
 
 
 
 
 
 
 
p
a
s
s
e
d
[
s
i
m
]
 
=
 
s
i
m
_
p
a
s
s
e
d

 
 
 
 
 
 
 
 
b
l
o
w
n
[
s
i
m
]
 
=
 
s
i
m
_
b
l
o
w
n
_
t
o
t
a
l

 
 
 
 
 
 
 
 
d
a
i
l
y
_
b
l
o
w
n
[
s
i
m
]
 
=
 
s
i
m
_
b
l
o
w
n
_
d
a
i
l
y


 
 
 
 
n
_
p
a
s
s
e
d
 
=
 
p
a
s
s
e
d
.
s
u
m
(
)

 
 
 
 
n
_
b
l
o
w
n
_
t
o
t
a
l
 
=
 
b
l
o
w
n
.
s
u
m
(
)

 
 
 
 
n
_
b
l
o
w
n
_
d
a
i
l
y
 
=
 
d
a
i
l
y
_
b
l
o
w
n
.
s
u
m
(
)

 
 
 
 
n
_
s
u
r
v
i
v
e
d
 
=
 
N
_
S
I
M
U
L
A
T
I
O
N
S
 
-
 
n
_
b
l
o
w
n
_
t
o
t
a
l
 
-
 
n
_
b
l
o
w
n
_
d
a
i
l
y
 
-
 
n
_
p
a
s
s
e
d


 
 
 
 
p
r
i
n
t
(
f
"
\
n
{
'
=
'
 
*
 
7
0
}
"
)

 
 
 
 
p
r
i
n
t
(
f
"
\
U
0
0
0
1
f
3
a
f
 
M
O
N
T
E
 
C
A
R
L
O
 
R
E
S
U
L
T
S
 
(
{
N
_
S
I
M
U
L
A
T
I
O
N
S
:
,
}
 
s
i
m
u
l
a
t
i
o
n
s
)
"
)

 
 
 
 
p
r
i
n
t
(
f
"
{
'
=
'
 
*
 
7
0
}
"
)

 
 
 
 
p
r
i
n
t
(
f
"
 
 
\
u
2
7
0
5
 
P
A
S
S
E
D
 
(
h
i
t
 
{
P
R
O
F
I
T
_
T
A
R
G
E
T
:
.
0
%
}
 
t
a
r
g
e
t
)
:
 
 
 
 
{
n
_
p
a
s
s
e
d
:
>
6
,
}
 
(
{
n
_
p
a
s
s
e
d
/
N
_
S
I
M
U
L
A
T
I
O
N
S
*
1
0
0
:
.
1
f
}
%
)
"
)

 
 
 
 
p
r
i
n
t
(
f
"
 
 
\
u
2
7
4
c
 
B
L
O
W
N
 
(
m
a
x
 
t
o
t
a
l
 
l
o
s
s
)
:
 
 
 
 
 
 
 
{
n
_
b
l
o
w
n
_
t
o
t
a
l
:
>
6
,
}
 
(
{
n
_
b
l
o
w
n
_
t
o
t
a
l
/
N
_
S
I
M
U
L
A
T
I
O
N
S
*
1
0
0
:
.
1
f
}
%
)
"
)

 
 
 
 
p
r
i
n
t
(
f
"
 
 
\
u
2
7
4
c
 
B
L
O
W
N
 
(
m
a
x
 
d
a
i
l
y
 
l
o
s
s
)
:
 
 
 
 
 
 
 
{
n
_
b
l
o
w
n
_
d
a
i
l
y
:
>
6
,
}
 
(
{
n
_
b
l
o
w
n
_
d
a
i
l
y
/
N
_
S
I
M
U
L
A
T
I
O
N
S
*
1
0
0
:
.
1
f
}
%
)
"
)

 
 
 
 
p
r
i
n
t
(
f
"
 
 
\
u
2
3
f
3
 
S
t
i
l
l
 
t
r
a
d
i
n
g
 
(
n
o
 
t
r
i
g
g
e
r
)
:
 
 
 
{
n
_
s
u
r
v
i
v
e
d
:
>
6
,
}
 
(
{
n
_
s
u
r
v
i
v
e
d
/
N
_
S
I
M
U
L
A
T
I
O
N
S
*
1
0
0
:
.
1
f
}
%
)
"
)

 
 
 
 
p
r
i
n
t
(
f
"
 
 
\
U
0
0
0
1
f
4
b
0
 
M
e
d
i
a
n
 
f
i
n
a
l
 
e
q
u
i
t
y
:
 
 
 
 
 
 
 
 
 
$
{
n
p
.
m
e
d
i
a
n
(
f
i
n
a
l
_
e
q
u
i
t
y
)
:
>
1
0
,
.
0
f
}
"
)

 
 
 
 
p
r
i
n
t
(
f
"
 
 
\
U
0
0
0
1
f
4
b
0
 
M
e
a
n
 
f
i
n
a
l
 
e
q
u
i
t
y
:
 
 
 
 
 
 
 
 
 
 
 
$
{
n
p
.
m
e
a
n
(
f
i
n
a
l
_
e
q
u
i
t
y
)
:
>
1
0
,
.
0
f
}
"
)

 
 
 
 
p
r
i
n
t
(
f
"
 
 
\
U
0
0
0
1
f
4
b
0
 
5
t
h
 
p
e
r
c
e
n
t
i
l
e
 
e
q
u
i
t
y
:
 
 
 
 
 
 
 
$
{
n
p
.
p
e
r
c
e
n
t
i
l
e
(
f
i
n
a
l
_
e
q
u
i
t
y
,
 
5
)
:
>
1
0
,
.
0
f
}
"
)

 
 
 
 
p
r
i
n
t
(
f
"
 
 
\
U
0
0
0
1
f
4
b
0
 
9
5
t
h
 
p
e
r
c
e
n
t
i
l
e
 
e
q
u
i
t
y
:
 
 
 
 
 
 
$
{
n
p
.
p
e
r
c
e
n
t
i
l
e
(
f
i
n
a
l
_
e
q
u
i
t
y
,
 
9
5
)
:
>
1
0
,
.
0
f
}
"
)

 
 
 
 
p
r
i
n
t
(
f
"
{
'
=
'
 
*
 
7
0
}
"
)


 
 
 
 
#
 
V
i
s
u
a
l
i
z
a
t
i
o
n

 
 
 
 
f
i
g
,
 
a
x
e
s
 
=
 
p
l
t
.
s
u
b
p
l
o
t
s
(
2
,
 
2
,
 
f
i
g
s
i
z
e
=
(
1
8
,
 
1
2
)
)

 
 
 
 
f
i
g
.
s
u
p
t
i
t
l
e
(
f
'
M
o
n
t
e
 
C
a
r
l
o
 
F
T
M
O
 
C
h
a
l
l
e
n
g
e
 
\
u
2
0
1
4
 
L
o
s
t
 
M
o
m
e
n
t
u
m
 
{
b
_
m
t
}
(
m
a
=
{
b
_
m
p
}
,
 
s
l
=
{
b
_
s
l
}
,
 
s
w
=
{
b
_
s
w
}
)
 
\
u
2
0
1
4
 
{
N
_
S
I
M
U
L
A
T
I
O
N
S
:
,
}
 
P
a
t
h
s
'
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
f
o
n
t
s
i
z
e
=
1
6
,
 
f
o
n
t
w
e
i
g
h
t
=
'
b
o
l
d
'
)


 
 
 
 
#
 
(
0
,
0
)
 
E
q
u
i
t
y
 
p
a
t
h
s
 
s
a
m
p
l
e
 
2
0
0

 
 
 
 
s
a
m
p
l
e
_
i
d
x
 
=
 
n
p
.
r
a
n
d
o
m
.
c
h
o
i
c
e
(
N
_
S
I
M
U
L
A
T
I
O
N
S
,
 
m
i
n
(
2
0
0
,
 
N
_
S
I
M
U
L
A
T
I
O
N
S
)
,
 
r
e
p
l
a
c
e
=
F
a
l
s
e
)

 
 
 
 
f
o
r
 
i
 
i
n
 
s
a
m
p
l
e
_
i
d
x
:

 
 
 
 
 
 
 
 
c
 
=
 
'
#
2
c
a
0
2
c
'
 
i
f
 
p
a
s
s
e
d
[
i
]
 
e
l
s
e
 
(
'
#
d
6
2
7
2
8
'
 
i
f
 
(
b
l
o
w
n
[
i
]
 
o
r
 
d
a
i
l
y
_
b
l
o
w
n
[
i
]
)
 
e
l
s
e
 
'
#
a
a
a
a
a
a
'
)

 
 
 
 
 
 
 
 
a
 
=
 
0
.
4
 
i
f
 
(
p
a
s
s
e
d
[
i
]
 
o
r
 
b
l
o
w
n
[
i
]
 
o
r
 
d
a
i
l
y
_
b
l
o
w
n
[
i
]
)
 
e
l
s
e
 
0
.
1

 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
 
0
]
.
p
l
o
t
(
r
a
n
g
e
(
T
R
A
D
I
N
G
_
D
A
Y
S
 
+
 
1
)
,
 
e
q
u
i
t
y
_
p
a
t
h
s
[
i
]
,
 
c
o
l
o
r
=
c
,
 
a
l
p
h
a
=
a
,
 
l
i
n
e
w
i
d
t
h
=
0
.
5
)

 
 
 
 
a
x
e
s
[
0
,
 
0
]
.
a
x
h
l
i
n
e
(
F
T
M
O
_
A
C
C
O
U
N
T
 
*
 
(
1
 
+
 
P
R
O
F
I
T
_
T
A
R
G
E
T
)
,
 
c
o
l
o
r
=
'
g
r
e
e
n
'
,
 
l
i
n
e
s
t
y
l
e
=
'
-
-
'
,
 
l
i
n
e
w
i
d
t
h
=
2
.
5
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
l
a
b
e
l
=
f
'
P
r
o
f
i
t
 
T
a
r
g
e
t
 
(
$
{
F
T
M
O
_
A
C
C
O
U
N
T
*
(
1
+
P
R
O
F
I
T
_
T
A
R
G
E
T
)
:
,
.
0
f
}
)
'
)

 
 
 
 
a
x
e
s
[
0
,
 
0
]
.
a
x
h
l
i
n
e
(
F
T
M
O
_
A
C
C
O
U
N
T
 
*
 
(
1
 
-
 
M
A
X
_
T
O
T
A
L
_
L
O
S
S
)
,
 
c
o
l
o
r
=
'
r
e
d
'
,
 
l
i
n
e
s
t
y
l
e
=
'
-
-
'
,
 
l
i
n
e
w
i
d
t
h
=
2
.
5
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
l
a
b
e
l
=
f
'
M
a
x
 
L
o
s
s
 
(
$
{
F
T
M
O
_
A
C
C
O
U
N
T
*
(
1
-
M
A
X
_
T
O
T
A
L
_
L
O
S
S
)
:
,
.
0
f
}
)
'
)

 
 
 
 
a
x
e
s
[
0
,
 
0
]
.
a
x
h
l
i
n
e
(
F
T
M
O
_
A
C
C
O
U
N
T
,
 
c
o
l
o
r
=
'
b
l
a
c
k
'
,
 
l
i
n
e
s
t
y
l
e
=
'
-
'
,
 
l
i
n
e
w
i
d
t
h
=
1
,
 
a
l
p
h
a
=
0
.
5
)

 
 
 
 
a
x
e
s
[
0
,
 
0
]
.
s
e
t
_
t
i
t
l
e
(
'
S
i
m
u
l
a
t
e
d
 
E
q
u
i
t
y
 
P
a
t
h
s
 
(
2
0
0
 
s
a
m
p
l
e
)
'
,
 
f
o
n
t
s
i
z
e
=
1
3
,
 
f
o
n
t
w
e
i
g
h
t
=
'
b
o
l
d
'
)

 
 
 
 
a
x
e
s
[
0
,
 
0
]
.
s
e
t
_
x
l
a
b
e
l
(
'
T
r
a
d
i
n
g
 
D
a
y
'
,
 
f
o
n
t
s
i
z
e
=
1
1
)

 
 
 
 
a
x
e
s
[
0
,
 
0
]
.
s
e
t
_
y
l
a
b
e
l
(
'
E
q
u
i
t
y
 
(
$
)
'
,
 
f
o
n
t
s
i
z
e
=
1
1
)

 
 
 
 
a
x
e
s
[
0
,
 
0
]
.
y
a
x
i
s
.
s
e
t
_
m
a
j
o
r
_
f
o
r
m
a
t
t
e
r
(
p
l
t
.
F
u
n
c
F
o
r
m
a
t
t
e
r
(
l
a
m
b
d
a
 
x
,
 
_
:
 
f
'
$
{
x
:
,
.
0
f
}
'
)
)

 
 
 
 
a
x
e
s
[
0
,
 
0
]
.
l
e
g
e
n
d
(
l
o
c
=
'
u
p
p
e
r
 
l
e
f
t
'
,
 
f
o
n
t
s
i
z
e
=
9
)

 
 
 
 
a
x
e
s
[
0
,
 
0
]
.
g
r
i
d
(
T
r
u
e
,
 
a
l
p
h
a
=
0
.
2
)


 
 
 
 
#
 
(
0
,
1
)
 
F
i
n
a
l
 
e
q
u
i
t
y
 
d
i
s
t
r
i
b
u
t
i
o
n

 
 
 
 
b
i
n
s
 
=
 
n
p
.
l
i
n
s
p
a
c
e
(
f
i
n
a
l
_
e
q
u
i
t
y
.
m
i
n
(
)
,
 
f
i
n
a
l
_
e
q
u
i
t
y
.
m
a
x
(
)
,
 
6
0
)

 
 
 
 
a
x
e
s
[
0
,
 
1
]
.
h
i
s
t
(
f
i
n
a
l
_
e
q
u
i
t
y
[
p
a
s
s
e
d
]
,
 
b
i
n
s
=
b
i
n
s
,
 
c
o
l
o
r
=
'
#
2
c
a
0
2
c
'
,
 
a
l
p
h
a
=
0
.
7
,
 
l
a
b
e
l
=
f
'
P
a
s
s
e
d
 
(
{
n
_
p
a
s
s
e
d
:
,
}
)
'
)

 
 
 
 
a
x
e
s
[
0
,
 
1
]
.
h
i
s
t
(
f
i
n
a
l
_
e
q
u
i
t
y
[
b
l
o
w
n
 
|
 
d
a
i
l
y
_
b
l
o
w
n
]
,
 
b
i
n
s
=
b
i
n
s
,
 
c
o
l
o
r
=
'
#
d
6
2
7
2
8
'
,
 
a
l
p
h
a
=
0
.
7
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
l
a
b
e
l
=
f
'
B
l
o
w
n
 
(
{
n
_
b
l
o
w
n
_
t
o
t
a
l
+
n
_
b
l
o
w
n
_
d
a
i
l
y
:
,
}
)
'
)

 
 
 
 
a
x
e
s
[
0
,
 
1
]
.
h
i
s
t
(
f
i
n
a
l
_
e
q
u
i
t
y
[
~
p
a
s
s
e
d
 
&
 
~
b
l
o
w
n
 
&
 
~
d
a
i
l
y
_
b
l
o
w
n
]
,
 
b
i
n
s
=
b
i
n
s
,
 
c
o
l
o
r
=
'
#
a
a
a
a
a
a
'
,
 
a
l
p
h
a
=
0
.
5
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
l
a
b
e
l
=
f
'
S
t
i
l
l
 
T
r
a
d
i
n
g
 
(
{
n
_
s
u
r
v
i
v
e
d
:
,
}
)
'
)

 
 
 
 
a
x
e
s
[
0
,
 
1
]
.
a
x
v
l
i
n
e
(
F
T
M
O
_
A
C
C
O
U
N
T
,
 
c
o
l
o
r
=
'
b
l
a
c
k
'
,
 
l
i
n
e
s
t
y
l
e
=
'
-
'
,
 
l
i
n
e
w
i
d
t
h
=
1
.
5
,
 
a
l
p
h
a
=
0
.
5
)

 
 
 
 
a
x
e
s
[
0
,
 
1
]
.
a
x
v
l
i
n
e
(
F
T
M
O
_
A
C
C
O
U
N
T
 
*
 
(
1
 
+
 
P
R
O
F
I
T
_
T
A
R
G
E
T
)
,
 
c
o
l
o
r
=
'
g
r
e
e
n
'
,
 
l
i
n
e
s
t
y
l
e
=
'
-
-
'
,
 
l
i
n
e
w
i
d
t
h
=
2
)

 
 
 
 
a
x
e
s
[
0
,
 
1
]
.
a
x
v
l
i
n
e
(
F
T
M
O
_
A
C
C
O
U
N
T
 
*
 
(
1
 
-
 
M
A
X
_
T
O
T
A
L
_
L
O
S
S
)
,
 
c
o
l
o
r
=
'
r
e
d
'
,
 
l
i
n
e
s
t
y
l
e
=
'
-
-
'
,
 
l
i
n
e
w
i
d
t
h
=
2
)

 
 
 
 
a
x
e
s
[
0
,
 
1
]
.
s
e
t
_
t
i
t
l
e
(
'
F
i
n
a
l
 
E
q
u
i
t
y
 
D
i
s
t
r
i
b
u
t
i
o
n
'
,
 
f
o
n
t
s
i
z
e
=
1
3
,
 
f
o
n
t
w
e
i
g
h
t
=
'
b
o
l
d
'
)

 
 
 
 
a
x
e
s
[
0
,
 
1
]
.
s
e
t
_
x
l
a
b
e
l
(
'
F
i
n
a
l
 
E
q
u
i
t
y
 
(
$
)
'
,
 
f
o
n
t
s
i
z
e
=
1
1
)

 
 
 
 
a
x
e
s
[
0
,
 
1
]
.
s
e
t
_
y
l
a
b
e
l
(
'
C
o
u
n
t
'
,
 
f
o
n
t
s
i
z
e
=
1
1
)

 
 
 
 
a
x
e
s
[
0
,
 
1
]
.
x
a
x
i
s
.
s
e
t
_
m
a
j
o
r
_
f
o
r
m
a
t
t
e
r
(
p
l
t
.
F
u
n
c
F
o
r
m
a
t
t
e
r
(
l
a
m
b
d
a
 
x
,
 
_
:
 
f
'
$
{
x
:
,
.
0
f
}
'
)
)

 
 
 
 
a
x
e
s
[
0
,
 
1
]
.
l
e
g
e
n
d
(
f
o
n
t
s
i
z
e
=
9
)

 
 
 
 
a
x
e
s
[
0
,
 
1
]
.
g
r
i
d
(
a
x
i
s
=
'
y
'
,
 
a
l
p
h
a
=
0
.
2
)


 
 
 
 
#
 
(
1
,
0
)
 
O
u
t
c
o
m
e
 
p
i
e
 
c
h
a
r
t

 
 
 
 
l
a
b
e
l
s
 
=
 
[
'
P
a
s
s
e
d
'
,
 
'
B
l
o
w
n
 
(
T
o
t
a
l
)
'
,
 
'
B
l
o
w
n
 
(
D
a
i
l
y
)
'
,
 
'
S
t
i
l
l
 
T
r
a
d
i
n
g
'
]

 
 
 
 
s
i
z
e
s
 
 
=
 
[
n
_
p
a
s
s
e
d
,
 
n
_
b
l
o
w
n
_
t
o
t
a
l
,
 
n
_
b
l
o
w
n
_
d
a
i
l
y
,
 
n
_
s
u
r
v
i
v
e
d
]

 
 
 
 
c
o
l
o
r
s
_
p
i
e
 
=
 
[
'
#
2
c
a
0
2
c
'
,
 
'
#
d
6
2
7
2
8
'
,
 
'
#
f
f
7
f
0
e
'
,
 
'
#
a
a
a
a
a
a
'
]

 
 
 
 
e
x
p
l
o
d
e
 
=
 
(
0
.
0
5
,
 
0
.
0
5
,
 
0
.
0
5
,
 
0
)

 
 
 
 
n
o
n
_
z
e
r
o
 
=
 
[
(
l
,
 
s
,
 
c
,
 
e
)
 
f
o
r
 
l
,
 
s
,
 
c
,
 
e
 
i
n
 
z
i
p
(
l
a
b
e
l
s
,
 
s
i
z
e
s
,
 
c
o
l
o
r
s
_
p
i
e
,
 
e
x
p
l
o
d
e
)
 
i
f
 
s
 
>
 
0
]

 
 
 
 
i
f
 
n
o
n
_
z
e
r
o
:

 
 
 
 
 
 
 
 
l
a
b
e
l
s
_
f
,
 
s
i
z
e
s
_
f
,
 
c
o
l
o
r
s
_
f
,
 
e
x
p
l
o
d
e
_
f
 
=
 
z
i
p
(
*
n
o
n
_
z
e
r
o
)

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
 
0
]
.
p
i
e
(
s
i
z
e
s
_
f
,
 
e
x
p
l
o
d
e
=
e
x
p
l
o
d
e
_
f
,
 
l
a
b
e
l
s
=
l
a
b
e
l
s
_
f
,
 
c
o
l
o
r
s
=
c
o
l
o
r
s
_
f
,
 
a
u
t
o
p
c
t
=
'
%
1
.
1
f
%
%
'
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
s
h
a
d
o
w
=
T
r
u
e
,
 
s
t
a
r
t
a
n
g
l
e
=
9
0
,
 
t
e
x
t
p
r
o
p
s
=
{
'
f
o
n
t
s
i
z
e
'
:
 
1
1
}
)

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
 
0
]
.
s
e
t
_
t
i
t
l
e
(
'
O
u
t
c
o
m
e
 
D
i
s
t
r
i
b
u
t
i
o
n
'
,
 
f
o
n
t
s
i
z
e
=
1
3
,
 
f
o
n
t
w
e
i
g
h
t
=
'
b
o
l
d
'
)


 
 
 
 
#
 
(
1
,
1
)
 
P
e
r
c
e
n
t
i
l
e
 
e
q
u
i
t
y
 
c
u
r
v
e
s

 
 
 
 
p
c
t
s
 
=
 
[
5
,
 
2
5
,
 
5
0
,
 
7
5
,
 
9
5
]

 
 
 
 
p
c
t
_
c
o
l
o
r
s
 
=
 
[
'
#
d
6
2
7
2
8
'
,
 
'
#
f
f
7
f
0
e
'
,
 
'
#
1
f
7
7
b
4
'
,
 
'
#
2
c
a
0
2
c
'
,
 
'
#
1
7
b
e
c
f
'
]

 
 
 
 
f
o
r
 
p
c
t
,
 
c
l
r
 
i
n
 
z
i
p
(
p
c
t
s
,
 
p
c
t
_
c
o
l
o
r
s
)
:

 
 
 
 
 
 
 
 
p
c
t
_
p
a
t
h
 
=
 
n
p
.
p
e
r
c
e
n
t
i
l
e
(
e
q
u
i
t
y
_
p
a
t
h
s
,
 
p
c
t
,
 
a
x
i
s
=
0
)

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
 
1
]
.
p
l
o
t
(
r
a
n
g
e
(
T
R
A
D
I
N
G
_
D
A
Y
S
 
+
 
1
)
,
 
p
c
t
_
p
a
t
h
,
 
c
o
l
o
r
=
c
l
r
,
 
l
i
n
e
w
i
d
t
h
=
2
,
 
l
a
b
e
l
=
f
'
P
{
p
c
t
}
'
)

 
 
 
 
a
x
e
s
[
1
,
 
1
]
.
a
x
h
l
i
n
e
(
F
T
M
O
_
A
C
C
O
U
N
T
 
*
 
(
1
 
+
 
P
R
O
F
I
T
_
T
A
R
G
E
T
)
,
 
c
o
l
o
r
=
'
g
r
e
e
n
'
,
 
l
i
n
e
s
t
y
l
e
=
'
-
-
'
,
 
l
i
n
e
w
i
d
t
h
=
2
,
 
a
l
p
h
a
=
0
.
7
)

 
 
 
 
a
x
e
s
[
1
,
 
1
]
.
a
x
h
l
i
n
e
(
F
T
M
O
_
A
C
C
O
U
N
T
 
*
 
(
1
 
-
 
M
A
X
_
T
O
T
A
L
_
L
O
S
S
)
,
 
c
o
l
o
r
=
'
r
e
d
'
,
 
l
i
n
e
s
t
y
l
e
=
'
-
-
'
,
 
l
i
n
e
w
i
d
t
h
=
2
,
 
a
l
p
h
a
=
0
.
7
)

 
 
 
 
a
x
e
s
[
1
,
 
1
]
.
a
x
h
l
i
n
e
(
F
T
M
O
_
A
C
C
O
U
N
T
,
 
c
o
l
o
r
=
'
b
l
a
c
k
'
,
 
l
i
n
e
s
t
y
l
e
=
'
-
'
,
 
l
i
n
e
w
i
d
t
h
=
1
,
 
a
l
p
h
a
=
0
.
3
)

 
 
 
 
a
x
e
s
[
1
,
 
1
]
.
s
e
t
_
t
i
t
l
e
(
'
P
e
r
c
e
n
t
i
l
e
 
E
q
u
i
t
y
 
C
u
r
v
e
s
'
,
 
f
o
n
t
s
i
z
e
=
1
3
,
 
f
o
n
t
w
e
i
g
h
t
=
'
b
o
l
d
'
)

 
 
 
 
a
x
e
s
[
1
,
 
1
]
.
s
e
t
_
x
l
a
b
e
l
(
'
T
r
a
d
i
n
g
 
D
a
y
'
,
 
f
o
n
t
s
i
z
e
=
1
1
)

 
 
 
 
a
x
e
s
[
1
,
 
1
]
.
s
e
t
_
y
l
a
b
e
l
(
'
E
q
u
i
t
y
 
(
$
)
'
,
 
f
o
n
t
s
i
z
e
=
1
1
)

 
 
 
 
a
x
e
s
[
1
,
 
1
]
.
y
a
x
i
s
.
s
e
t
_
m
a
j
o
r
_
f
o
r
m
a
t
t
e
r
(
p
l
t
.
F
u
n
c
F
o
r
m
a
t
t
e
r
(
l
a
m
b
d
a
 
x
,
 
_
:
 
f
'
$
{
x
:
,
.
0
f
}
'
)
)

 
 
 
 
a
x
e
s
[
1
,
 
1
]
.
l
e
g
e
n
d
(
f
o
n
t
s
i
z
e
=
9
)

 
 
 
 
a
x
e
s
[
1
,
 
1
]
.
g
r
i
d
(
T
r
u
e
,
 
a
l
p
h
a
=
0
.
2
)


 
 
 
 
p
l
t
.
t
i
g
h
t
_
l
a
y
o
u
t
(
)

 
 
 
 
p
l
t
.
s
h
o
w
(
)


 
 
 
 
#
 
F
T
M
O
 
V
e
r
d
i
c
t

 
 
 
 
p
a
s
s
_
r
a
t
e
 
=
 
n
_
p
a
s
s
e
d
 
/
 
N
_
S
I
M
U
L
A
T
I
O
N
S
 
*
 
1
0
0

 
 
 
 
p
r
i
n
t
(
f
"
\
n
{
'
=
'
 
*
 
7
0
}
"
)

 
 
 
 
i
f
 
p
a
s
s
_
r
a
t
e
 
>
=
 
5
0
:

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
\
U
0
0
0
1
f
7
e
2
 
F
T
M
O
 
V
E
R
D
I
C
T
:
 
F
A
V
O
R
A
B
L
E
 
\
u
2
0
1
4
 
{
p
a
s
s
_
r
a
t
e
:
.
1
f
}
%
 
p
a
s
s
 
r
a
t
e
"
)

 
 
 
 
e
l
i
f
 
p
a
s
s
_
r
a
t
e
 
>
=
 
2
5
:

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
\
U
0
0
0
1
f
7
e
1
 
F
T
M
O
 
V
E
R
D
I
C
T
:
 
P
O
S
S
I
B
L
E
 
\
u
2
0
1
4
 
{
p
a
s
s
_
r
a
t
e
:
.
1
f
}
%
 
p
a
s
s
 
r
a
t
e
"
)

 
 
 
 
e
l
i
f
 
p
a
s
s
_
r
a
t
e
 
>
=
 
1
0
:

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
\
U
0
0
0
1
f
7
e
0
 
F
T
M
O
 
V
E
R
D
I
C
T
:
 
C
H
A
L
L
E
N
G
I
N
G
 
\
u
2
0
1
4
 
{
p
a
s
s
_
r
a
t
e
:
.
1
f
}
%
 
p
a
s
s
 
r
a
t
e
"
)

 
 
 
 
e
l
s
e
:

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
\
U
0
0
0
1
f
5
3
4
 
F
T
M
O
 
V
E
R
D
I
C
T
:
 
U
N
L
I
K
E
L
Y
 
\
u
2
0
1
4
 
{
p
a
s
s
_
r
a
t
e
:
.
1
f
}
%
 
p
a
s
s
 
r
a
t
e
"
)

 
 
 
 
p
r
i
n
t
(
f
"
{
'
=
'
 
*
 
7
0
}
"
)

 
 
 
 
p
r
i
n
t
(
f
"
N
o
t
e
:
 
T
h
i
s
 
s
i
m
u
l
a
t
i
o
n
 
b
o
o
t
s
t
r
a
p
s
 
f
r
o
m
 
o
b
s
e
r
v
e
d
 
d
a
i
l
y
 
r
e
t
u
r
n
s
.
"
)

 
 
 
 
p
r
i
n
t
(
f
"
I
t
 
a
s
s
u
m
e
s
 
t
h
e
 
s
t
r
a
t
e
g
y
 
i
s
 
t
r
a
d
e
d
 
e
v
e
r
y
 
d
a
y
 
a
t
 
h
i
s
t
o
r
i
c
a
l
 
r
e
t
u
r
n
 
l
e
v
e
l
s
.
"
)

 
 
 
 
p
r
i
n
t
(
f
"
R
e
a
l
 
p
e
r
f
o
r
m
a
n
c
e
 
w
i
l
l
 
v
a
r
y
 
b
a
s
e
d
 
o
n
 
m
a
r
k
e
t
 
c
o
n
d
i
t
i
o
n
s
 
a
n
d
 
e
x
e
c
u
t
i
o
n
.
"
)

In [ ]:
#
 
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═

#
 
U
N
I
V
E
R
S
A
L
 
S
T
R
A
T
E
G
Y
 
E
X
P
O
R
T
 
—
 
D
a
t
a
 
F
i
l
e
s
 
+
 
P
D
F
 
T
e
a
r
s
h
e
e
t

#
 
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═


i
m
p
o
r
t
 
o
s
,
 
s
y
s
,
 
j
s
o
n
,
 
d
a
t
e
t
i
m
e
,
 
h
a
s
h
l
i
b
,
 
p
l
a
t
f
o
r
m
,
 
s
h
u
t
i
l

f
r
o
m
 
m
a
t
p
l
o
t
l
i
b
.
b
a
c
k
e
n
d
s
.
b
a
c
k
e
n
d
_
p
d
f
 
i
m
p
o
r
t
 
P
d
f
P
a
g
e
s


#
 
═
═
═
 
E
D
I
T
 
T
H
E
S
E
 
L
I
N
E
S
 
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═

S
T
R
A
T
E
G
Y
_
N
A
M
E
 
=
 
"
L
o
s
t
_
M
o
m
e
n
t
u
m
"

P
A
R
A
M
_
C
O
L
S
 
 
 
 
=
 
[
"
m
a
_
p
e
r
i
o
d
"
,
 
"
m
a
_
t
y
p
e
"
,
 
"
s
l
o
p
e
_
l
o
o
k
b
a
c
k
"
,
 
"
s
w
i
n
g
_
l
o
o
k
b
a
c
k
"
]

N
O
T
E
S
 
 
 
 
 
 
 
 
 
=
 
"
"

#
 
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═


I
N
I
T
_
C
A
S
H
 
=
 
1
0
0
_
0
0
0

F
E
E
S
 
 
 
 
 
 
=
 
0
.
0
0
0
5

S
L
I
P
P
A
G
E
 
 
=
 
0
.
0
0
0
5

F
R
E
Q
 
 
 
 
 
 
=
 
'
D
'


#
 
G
o
o
g
l
e
 
D
r
i
v
e
 
m
o
u
n
t

E
X
P
O
R
T
_
D
I
R
 
=
 
"
/
c
o
n
t
e
n
t
/
s
t
r
a
t
e
g
y
_
e
x
p
o
r
t
s
"

I
N
_
C
O
L
A
B
 
=
 
'
g
o
o
g
l
e
.
c
o
l
a
b
'
 
i
n
 
s
y
s
.
m
o
d
u
l
e
s

t
r
y
:

 
 
 
 
f
r
o
m
 
g
o
o
g
l
e
.
c
o
l
a
b
 
i
m
p
o
r
t
 
d
r
i
v
e

 
 
 
 
i
f
 
n
o
t
 
o
s
.
p
a
t
h
.
e
x
i
s
t
s
(
'
/
c
o
n
t
e
n
t
/
d
r
i
v
e
'
)
:

 
 
 
 
 
 
 
 
d
r
i
v
e
.
m
o
u
n
t
(
'
/
c
o
n
t
e
n
t
/
d
r
i
v
e
'
)

 
 
 
 
E
X
P
O
R
T
_
D
I
R
 
=
 
"
/
c
o
n
t
e
n
t
/
d
r
i
v
e
/
M
y
D
r
i
v
e
/
s
t
r
a
t
e
g
y
_
e
x
p
o
r
t
s
"

 
 
 
 
I
N
_
C
O
L
A
B
 
=
 
T
r
u
e

 
 
 
 
p
r
i
n
t
(
"
\
u
2
7
0
5
 
G
o
o
g
l
e
 
D
r
i
v
e
 
m
o
u
n
t
e
d
"
)

e
x
c
e
p
t
:

 
 
 
 
p
r
i
n
t
(
"
\
u
2
6
a
0
\
u
f
e
0
f
 
L
o
c
a
l
 
m
o
d
e
 
\
u
2
0
1
4
 
e
x
p
o
r
t
s
 
t
o
 
.
/
s
t
r
a
t
e
g
y
_
e
x
p
o
r
t
s
"
)


R
U
N
_
T
I
M
E
S
T
A
M
P
 
=
 
d
a
t
e
t
i
m
e
.
d
a
t
e
t
i
m
e
.
n
o
w
(
)

R
U
N
_
I
D
 
=
 
R
U
N
_
T
I
M
E
S
T
A
M
P
.
s
t
r
f
t
i
m
e
(
"
%
Y
%
m
%
d
_
%
H
%
M
%
S
"
)


#
 
F
o
l
d
e
r
 
s
t
r
u
c
t
u
r
e

S
T
R
A
T
_
D
I
R
 
 
 
=
 
o
s
.
p
a
t
h
.
j
o
i
n
(
E
X
P
O
R
T
_
D
I
R
,
 
S
T
R
A
T
E
G
Y
_
N
A
M
E
,
 
T
I
C
K
E
R
)

L
A
T
E
S
T
_
D
I
R
 
 
=
 
o
s
.
p
a
t
h
.
j
o
i
n
(
S
T
R
A
T
_
D
I
R
,
 
"
l
a
t
e
s
t
"
)

A
R
C
H
I
V
E
_
D
I
R
 
=
 
o
s
.
p
a
t
h
.
j
o
i
n
(
S
T
R
A
T
_
D
I
R
,
 
"
a
r
c
h
i
v
e
"
)

o
s
.
m
a
k
e
d
i
r
s
(
L
A
T
E
S
T
_
D
I
R
,
 
e
x
i
s
t
_
o
k
=
T
r
u
e
)

o
s
.
m
a
k
e
d
i
r
s
(
A
R
C
H
I
V
E
_
D
I
R
,
 
e
x
i
s
t
_
o
k
=
T
r
u
e
)


#
 
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═

#
 
B
u
i
l
d
 
p
o
r
t
f
o
l
i
o
s

#
 
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═

r
e
s
u
l
t
s
_
d
f
 
=
 
p
d
.
D
a
t
a
F
r
a
m
e
(
g
r
i
d
_
s
e
a
r
c
h
_
r
e
s
u
l
t
s
)

b
e
s
t
 
=
 
r
e
s
u
l
t
s
_
d
f
.
l
o
c
[
r
e
s
u
l
t
s
_
d
f
[
'
s
h
a
r
p
e
_
r
a
t
i
o
'
]
.
i
d
x
m
a
x
(
)
]

b
e
s
t
_
p
a
r
a
m
s
 
=
 
{
}

f
o
r
 
c
o
l
 
i
n
 
P
A
R
A
M
_
C
O
L
S
:

 
 
 
 
v
a
l
 
=
 
b
e
s
t
[
c
o
l
]

 
 
 
 
b
e
s
t
_
p
a
r
a
m
s
[
c
o
l
]
 
=
 
i
n
t
(
v
a
l
)
 
i
f
 
c
o
l
 
!
=
 
'
m
a
_
t
y
p
e
'
 
e
l
s
e
 
s
t
r
(
v
a
l
)

p
a
r
a
m
_
s
t
r
 
=
 
"
,
 
"
.
j
o
i
n
(
[
f
"
{
k
}
=
{
v
}
"
 
f
o
r
 
k
,
 
v
 
i
n
 
b
e
s
t
_
p
a
r
a
m
s
.
i
t
e
m
s
(
)
]
)


i
f
 
i
s
i
n
s
t
a
n
c
e
(
s
t
o
c
k
_
d
a
t
a
.
c
o
l
u
m
n
s
,
 
p
d
.
M
u
l
t
i
I
n
d
e
x
)
:

 
 
 
 
f
u
l
l
_
c
l
o
s
e
 
=
 
s
t
o
c
k
_
d
a
t
a
[
(
'
C
l
o
s
e
'
,
 
T
I
C
K
E
R
)
]
.
a
s
t
y
p
e
(
f
l
o
a
t
)
.
s
q
u
e
e
z
e
(
)

 
 
 
 
f
u
l
l
_
l
o
w
 
=
 
s
t
o
c
k
_
d
a
t
a
[
(
'
L
o
w
'
,
 
T
I
C
K
E
R
)
]
.
a
s
t
y
p
e
(
f
l
o
a
t
)
.
s
q
u
e
e
z
e
(
)

e
l
s
e
:

 
 
 
 
f
u
l
l
_
c
l
o
s
e
 
=
 
s
t
o
c
k
_
d
a
t
a
[
'
C
l
o
s
e
'
]
.
a
s
t
y
p
e
(
f
l
o
a
t
)
.
s
q
u
e
e
z
e
(
)

 
 
 
 
f
u
l
l
_
l
o
w
 
=
 
s
t
o
c
k
_
d
a
t
a
[
'
L
o
w
'
]
.
a
s
t
y
p
e
(
f
l
o
a
t
)
.
s
q
u
e
e
z
e
(
)

f
u
l
l
_
c
l
o
s
e
.
n
a
m
e
 
=
 
'
p
r
i
c
e
'
;
 
f
u
l
l
_
l
o
w
.
n
a
m
e
 
=
 
'
l
o
w
'


s
p
l
i
t
_
i
d
x
 
=
 
i
n
t
(
l
e
n
(
f
u
l
l
_
c
l
o
s
e
)
 
*
 
0
.
6
0
)

t
r
a
i
n
_
c
l
o
s
e
 
=
 
f
u
l
l
_
c
l
o
s
e
.
i
l
o
c
[
:
s
p
l
i
t
_
i
d
x
]
.
c
o
p
y
(
)

v
a
l
_
c
l
o
s
e
 
=
 
f
u
l
l
_
c
l
o
s
e
.
i
l
o
c
[
s
p
l
i
t
_
i
d
x
:
]
.
c
o
p
y
(
)

t
r
a
i
n
_
l
o
w
 
=
 
f
u
l
l
_
l
o
w
.
i
l
o
c
[
:
s
p
l
i
t
_
i
d
x
]
.
c
o
p
y
(
)

v
a
l
_
l
o
w
 
=
 
f
u
l
l
_
l
o
w
.
i
l
o
c
[
s
p
l
i
t
_
i
d
x
:
]
.
c
o
p
y
(
)


b
_
m
p
 
=
 
b
e
s
t
_
p
a
r
a
m
s
[
'
m
a
_
p
e
r
i
o
d
'
]
;
 
b
_
m
t
 
=
 
b
e
s
t
_
p
a
r
a
m
s
[
'
m
a
_
t
y
p
e
'
]

b
_
s
l
 
=
 
b
e
s
t
_
p
a
r
a
m
s
[
'
s
l
o
p
e
_
l
o
o
k
b
a
c
k
'
]
;
 
b
_
s
w
 
=
 
b
e
s
t
_
p
a
r
a
m
s
[
'
s
w
i
n
g
_
l
o
o
k
b
a
c
k
'
]


#
 
F
u
l
l
 
s
a
m
p
l
e

e
_
f
u
l
l
,
 
x
_
f
u
l
l
 
=
 
_
l
o
s
t
_
m
o
m
e
n
t
u
m
_
s
i
g
n
a
l
s
(
f
u
l
l
_
c
l
o
s
e
,
 
f
u
l
l
_
l
o
w
,
 
b
_
m
p
,
 
b
_
m
t
,
 
b
_
s
l
,
 
b
_
s
w
)

p
f
_
f
u
l
l
 
=
 
v
b
t
.
P
o
r
t
f
o
l
i
o
.
f
r
o
m
_
s
i
g
n
a
l
s
(
c
l
o
s
e
=
f
u
l
l
_
c
l
o
s
e
,
 
e
n
t
r
i
e
s
=
e
_
f
u
l
l
,
 
e
x
i
t
s
=
x
_
f
u
l
l
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
i
n
i
t
_
c
a
s
h
=
I
N
I
T
_
C
A
S
H
,
 
f
e
e
s
=
F
E
E
S
,
 
s
l
i
p
p
a
g
e
=
S
L
I
P
P
A
G
E
,
 
f
r
e
q
=
F
R
E
Q
)

#
 
I
S

e
_
i
s
,
 
x
_
i
s
 
=
 
_
l
o
s
t
_
m
o
m
e
n
t
u
m
_
s
i
g
n
a
l
s
(
t
r
a
i
n
_
c
l
o
s
e
,
 
t
r
a
i
n
_
l
o
w
,
 
b
_
m
p
,
 
b
_
m
t
,
 
b
_
s
l
,
 
b
_
s
w
)

p
f
_
i
s
 
=
 
v
b
t
.
P
o
r
t
f
o
l
i
o
.
f
r
o
m
_
s
i
g
n
a
l
s
(
c
l
o
s
e
=
t
r
a
i
n
_
c
l
o
s
e
,
 
e
n
t
r
i
e
s
=
e
_
i
s
,
 
e
x
i
t
s
=
x
_
i
s
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
i
n
i
t
_
c
a
s
h
=
I
N
I
T
_
C
A
S
H
,
 
f
e
e
s
=
F
E
E
S
,
 
s
l
i
p
p
a
g
e
=
S
L
I
P
P
A
G
E
,
 
f
r
e
q
=
F
R
E
Q
)

#
 
O
O
S

e
_
o
o
s
,
 
x
_
o
o
s
 
=
 
_
l
o
s
t
_
m
o
m
e
n
t
u
m
_
s
i
g
n
a
l
s
(
v
a
l
_
c
l
o
s
e
,
 
v
a
l
_
l
o
w
,
 
b
_
m
p
,
 
b
_
m
t
,
 
b
_
s
l
,
 
b
_
s
w
)

p
f
_
o
o
s
 
=
 
v
b
t
.
P
o
r
t
f
o
l
i
o
.
f
r
o
m
_
s
i
g
n
a
l
s
(
c
l
o
s
e
=
v
a
l
_
c
l
o
s
e
,
 
e
n
t
r
i
e
s
=
e
_
o
o
s
,
 
e
x
i
t
s
=
x
_
o
o
s
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
i
n
i
t
_
c
a
s
h
=
I
N
I
T
_
C
A
S
H
,
 
f
e
e
s
=
F
E
E
S
,
 
s
l
i
p
p
a
g
e
=
S
L
I
P
P
A
G
E
,
 
f
r
e
q
=
F
R
E
Q
)

#
 
B
u
y
 
&
 
H
o
l
d

b
h
_
e
 
=
 
p
d
.
S
e
r
i
e
s
(
F
a
l
s
e
,
 
i
n
d
e
x
=
f
u
l
l
_
c
l
o
s
e
.
i
n
d
e
x
,
 
d
t
y
p
e
=
b
o
o
l
)
;
 
b
h
_
e
.
i
l
o
c
[
0
]
 
=
 
T
r
u
e

b
h
_
x
 
=
 
p
d
.
S
e
r
i
e
s
(
F
a
l
s
e
,
 
i
n
d
e
x
=
f
u
l
l
_
c
l
o
s
e
.
i
n
d
e
x
,
 
d
t
y
p
e
=
b
o
o
l
)

p
f
_
b
h
 
=
 
v
b
t
.
P
o
r
t
f
o
l
i
o
.
f
r
o
m
_
s
i
g
n
a
l
s
(
c
l
o
s
e
=
f
u
l
l
_
c
l
o
s
e
,
 
e
n
t
r
i
e
s
=
b
h
_
e
,
 
e
x
i
t
s
=
b
h
_
x
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
i
n
i
t
_
c
a
s
h
=
I
N
I
T
_
C
A
S
H
,
 
f
e
e
s
=
F
E
E
S
,
 
s
l
i
p
p
a
g
e
=
S
L
I
P
P
A
G
E
,
 
f
r
e
q
=
F
R
E
Q
)


#
 
E
x
t
r
a
c
t
 
m
e
t
r
i
c
s

t
r
a
d
e
s
_
o
b
j
 
=
 
p
f
_
f
u
l
l
.
t
r
a
d
e
s

t
r
 
=
 
n
p
.
a
s
a
r
r
a
y
(
t
r
a
d
e
s
_
o
b
j
.
r
e
t
u
r
n
s
.
v
a
l
u
e
s
 
i
f
 
h
a
s
a
t
t
r
(
t
r
a
d
e
s
_
o
b
j
.
r
e
t
u
r
n
s
,
 
'
v
a
l
u
e
s
'
)
 
e
l
s
e
 
t
r
a
d
e
s
_
o
b
j
.
r
e
t
u
r
n
s
)
.
r
a
v
e
l
(
)

p
n
l
 
=
 
n
p
.
a
s
a
r
r
a
y
(
t
r
a
d
e
s
_
o
b
j
.
p
n
l
.
v
a
l
u
e
s
 
i
f
 
h
a
s
a
t
t
r
(
t
r
a
d
e
s
_
o
b
j
.
p
n
l
,
 
'
v
a
l
u
e
s
'
)
 
e
l
s
e
 
t
r
a
d
e
s
_
o
b
j
.
p
n
l
)
.
r
a
v
e
l
(
)

p
o
s
,
 
n
e
g
 
=
 
t
r
[
t
r
 
>
 
0
]
,
 
t
r
[
t
r
 
<
 
0
]

y
e
a
r
s
_
f
u
l
l
 
=
 
m
a
x
(
(
f
u
l
l
_
c
l
o
s
e
.
i
n
d
e
x
[
-
1
]
 
-
 
f
u
l
l
_
c
l
o
s
e
.
i
n
d
e
x
[
0
]
)
.
d
a
y
s
 
/
 
3
6
5
.
2
5
,
 
1
e
-
9
)

d
a
i
l
y
_
r
e
t
s
 
=
 
p
f
_
f
u
l
l
.
r
e
t
u
r
n
s
(
)


d
e
f
 
s
a
f
e
(
f
n
,
 
d
e
f
a
u
l
t
=
N
o
n
e
)
:

 
 
 
 
t
r
y
:
 
r
e
t
u
r
n
 
f
l
o
a
t
(
f
n
(
)
)

 
 
 
 
e
x
c
e
p
t
:
 
r
e
t
u
r
n
 
d
e
f
a
u
l
t


M
 
=
 
{

 
 
 
 
'
t
o
t
a
l
_
r
e
t
u
r
n
'
:
 
s
a
f
e
(
p
f
_
f
u
l
l
.
t
o
t
a
l
_
r
e
t
u
r
n
)
,
 
'
a
n
n
_
r
e
t
u
r
n
'
:
 
s
a
f
e
(
l
a
m
b
d
a
:
 
p
f
_
f
u
l
l
.
a
n
n
u
a
l
i
z
e
d
_
r
e
t
u
r
n
(
f
r
e
q
=
F
R
E
Q
)
)
,

 
 
 
 
'
s
h
a
r
p
e
'
:
 
s
a
f
e
(
l
a
m
b
d
a
:
 
p
f
_
f
u
l
l
.
s
h
a
r
p
e
_
r
a
t
i
o
(
f
r
e
q
=
F
R
E
Q
)
)
,
 
'
s
o
r
t
i
n
o
'
:
 
s
a
f
e
(
l
a
m
b
d
a
:
 
p
f
_
f
u
l
l
.
s
o
r
t
i
n
o
_
r
a
t
i
o
(
f
r
e
q
=
F
R
E
Q
)
)
,

 
 
 
 
'
m
a
x
_
d
d
'
:
 
s
a
f
e
(
p
f
_
f
u
l
l
.
m
a
x
_
d
r
a
w
d
o
w
n
)
,
 
'
v
o
l
a
t
i
l
i
t
y
'
:
 
s
a
f
e
(
l
a
m
b
d
a
:
 
p
f
_
f
u
l
l
.
a
n
n
u
a
l
i
z
e
d
_
v
o
l
a
t
i
l
i
t
y
(
f
r
e
q
=
F
R
E
Q
)
)
,

 
 
 
 
'
c
a
l
m
a
r
'
:
 
s
a
f
e
(
l
a
m
b
d
a
:
 
p
f
_
f
u
l
l
.
a
n
n
u
a
l
i
z
e
d
_
r
e
t
u
r
n
(
f
r
e
q
=
F
R
E
Q
)
)
 
/
 
a
b
s
(
s
a
f
e
(
p
f
_
f
u
l
l
.
m
a
x
_
d
r
a
w
d
o
w
n
)
)
 
i
f
 
a
b
s
(
s
a
f
e
(
p
f
_
f
u
l
l
.
m
a
x
_
d
r
a
w
d
o
w
n
,
 
0
)
)
 
>
 
1
e
-
9
 
e
l
s
e
 
N
o
n
e
,

 
 
 
 
'
t
r
a
d
e
s
'
:
 
l
e
n
(
t
r
a
d
e
s
_
o
b
j
)
,
 
'
t
r
a
d
e
s
_
y
r
'
:
 
l
e
n
(
t
r
a
d
e
s
_
o
b
j
)
 
/
 
y
e
a
r
s
_
f
u
l
l
,

 
 
 
 
'
w
i
n
_
r
a
t
e
'
:
 
f
l
o
a
t
(
l
e
n
(
p
o
s
)
 
/
 
l
e
n
(
t
r
)
 
*
 
1
0
0
)
 
i
f
 
l
e
n
(
t
r
)
 
>
 
0
 
e
l
s
e
 
N
o
n
e
,

 
 
 
 
'
p
f
'
:
 
f
l
o
a
t
(
p
o
s
.
s
u
m
(
)
 
/
 
a
b
s
(
n
e
g
.
s
u
m
(
)
)
)
 
i
f
 
l
e
n
(
n
e
g
)
 
>
 
0
 
a
n
d
 
a
b
s
(
n
e
g
.
s
u
m
(
)
)
 
>
 
0
 
e
l
s
e
 
N
o
n
e
,

 
 
 
 
'
e
x
p
e
c
t
a
n
c
y
'
:
 
f
l
o
a
t
(
t
r
.
m
e
a
n
(
)
)
 
i
f
 
l
e
n
(
t
r
)
 
>
 
0
 
e
l
s
e
 
N
o
n
e
,

 
 
 
 
'
a
v
g
_
w
i
n
'
:
 
f
l
o
a
t
(
p
o
s
.
m
e
a
n
(
)
)
 
i
f
 
l
e
n
(
p
o
s
)
 
>
 
0
 
e
l
s
e
 
N
o
n
e
,
 
'
a
v
g
_
l
o
s
s
'
:
 
f
l
o
a
t
(
n
e
g
.
m
e
a
n
(
)
)
 
i
f
 
l
e
n
(
n
e
g
)
 
>
 
0
 
e
l
s
e
 
N
o
n
e
,

 
 
 
 
'
l
a
r
g
e
s
t
_
w
i
n
'
:
 
f
l
o
a
t
(
p
o
s
.
m
a
x
(
)
)
 
i
f
 
l
e
n
(
p
o
s
)
 
>
 
0
 
e
l
s
e
 
N
o
n
e
,
 
'
l
a
r
g
e
s
t
_
l
o
s
s
'
:
 
f
l
o
a
t
(
n
e
g
.
m
i
n
(
)
)
 
i
f
 
l
e
n
(
n
e
g
)
 
>
 
0
 
e
l
s
e
 
N
o
n
e
,

 
 
 
 
'
p
a
y
o
f
f
'
:
 
f
l
o
a
t
(
a
b
s
(
p
o
s
.
m
e
a
n
(
)
 
/
 
n
e
g
.
m
e
a
n
(
)
)
)
 
i
f
 
l
e
n
(
p
o
s
)
 
>
 
0
 
a
n
d
 
l
e
n
(
n
e
g
)
 
>
 
0
 
e
l
s
e
 
N
o
n
e
,

 
 
 
 
'
i
s
_
s
h
a
r
p
e
'
:
 
s
a
f
e
(
l
a
m
b
d
a
:
 
p
f
_
i
s
.
s
h
a
r
p
e
_
r
a
t
i
o
(
f
r
e
q
=
F
R
E
Q
)
)
,
 
'
i
s
_
r
e
t
u
r
n
'
:
 
s
a
f
e
(
p
f
_
i
s
.
t
o
t
a
l
_
r
e
t
u
r
n
)
,

 
 
 
 
'
i
s
_
d
d
'
:
 
s
a
f
e
(
p
f
_
i
s
.
m
a
x
_
d
r
a
w
d
o
w
n
)
,
 
'
i
s
_
t
r
a
d
e
s
'
:
 
l
e
n
(
p
f
_
i
s
.
t
r
a
d
e
s
)
,

 
 
 
 
'
o
o
s
_
s
h
a
r
p
e
'
:
 
s
a
f
e
(
l
a
m
b
d
a
:
 
p
f
_
o
o
s
.
s
h
a
r
p
e
_
r
a
t
i
o
(
f
r
e
q
=
F
R
E
Q
)
)
,
 
'
o
o
s
_
r
e
t
u
r
n
'
:
 
s
a
f
e
(
p
f
_
o
o
s
.
t
o
t
a
l
_
r
e
t
u
r
n
)
,

 
 
 
 
'
o
o
s
_
d
d
'
:
 
s
a
f
e
(
p
f
_
o
o
s
.
m
a
x
_
d
r
a
w
d
o
w
n
)
,
 
'
o
o
s
_
t
r
a
d
e
s
'
:
 
l
e
n
(
p
f
_
o
o
s
.
t
r
a
d
e
s
)
,

 
 
 
 
'
b
h
_
r
e
t
u
r
n
'
:
 
s
a
f
e
(
p
f
_
b
h
.
t
o
t
a
l
_
r
e
t
u
r
n
)
,
 
'
b
h
_
s
h
a
r
p
e
'
:
 
s
a
f
e
(
l
a
m
b
d
a
:
 
p
f
_
b
h
.
s
h
a
r
p
e
_
r
a
t
i
o
(
f
r
e
q
=
F
R
E
Q
)
)
,

 
 
 
 
'
b
h
_
d
d
'
:
 
s
a
f
e
(
p
f
_
b
h
.
m
a
x
_
d
r
a
w
d
o
w
n
)
,

}


#
 
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═

#
 
1
.
 
S
A
V
E
 
S
T
R
U
C
T
U
R
E
D
 
D
A
T
A
 
F
I
L
E
S

#
 
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═

e
x
p
o
r
t
_
j
s
o
n
 
=
 
{

 
 
 
 
"
m
e
t
a
d
a
t
a
"
:
 
{

 
 
 
 
 
 
 
 
"
r
u
n
_
i
d
"
:
 
R
U
N
_
I
D
,
 
"
e
x
p
o
r
t
_
t
i
m
e
s
t
a
m
p
"
:
 
R
U
N
_
T
I
M
E
S
T
A
M
P
.
i
s
o
f
o
r
m
a
t
(
)
,

 
 
 
 
 
 
 
 
"
e
x
p
o
r
t
_
d
a
t
e
_
h
u
m
a
n
"
:
 
R
U
N
_
T
I
M
E
S
T
A
M
P
.
s
t
r
f
t
i
m
e
(
"
%
B
 
%
d
,
 
%
Y
 
a
t
 
%
I
:
%
M
 
%
p
"
)
,

 
 
 
 
 
 
 
 
"
s
t
r
a
t
e
g
y
_
n
a
m
e
"
:
 
S
T
R
A
T
E
G
Y
_
N
A
M
E
,
 
"
s
t
r
a
t
e
g
y
_
f
a
m
i
l
y
"
:
 
"
L
o
s
t
_
M
o
m
e
n
t
u
m
"
,

 
 
 
 
 
 
 
 
"
t
i
c
k
e
r
"
:
 
T
I
C
K
E
R
,

 
 
 
 
 
 
 
 
"
i
n
s
t
r
u
m
e
n
t
_
t
y
p
e
"
:
 
(
"
c
r
y
p
t
o
"
 
i
f
 
"
-
U
S
D
"
 
i
n
 
T
I
C
K
E
R
 
a
n
d
 
T
I
C
K
E
R
.
r
e
p
l
a
c
e
(
"
-
U
S
D
"
,
"
"
)
.
i
s
a
l
p
h
a
(
)

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
e
l
s
e
 
"
f
o
r
e
x
"
 
i
f
 
"
/
"
 
i
n
 
T
I
C
K
E
R
 
o
r
 
(
l
e
n
(
T
I
C
K
E
R
)
 
=
=
 
6
 
a
n
d
 
T
I
C
K
E
R
.
i
s
a
l
p
h
a
(
)
)

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
e
l
s
e
 
"
e
q
u
i
t
y
/
e
t
f
"
)
,

 
 
 
 
 
 
 
 
"
d
a
t
a
_
s
o
u
r
c
e
"
:
 
"
y
f
i
n
a
n
c
e
"
,
 
"
d
a
t
a
_
i
n
t
e
r
v
a
l
"
:
 
"
1
d
"
,
 
"
c
u
r
r
e
n
c
y
"
:
 
"
U
S
D
"
,

 
 
 
 
 
 
 
 
"
s
t
a
r
t
_
d
a
t
e
"
:
 
s
t
r
(
f
u
l
l
_
c
l
o
s
e
.
i
n
d
e
x
[
0
]
.
d
a
t
e
(
)
)
,
 
"
e
n
d
_
d
a
t
e
"
:
 
s
t
r
(
f
u
l
l
_
c
l
o
s
e
.
i
n
d
e
x
[
-
1
]
.
d
a
t
e
(
)
)
,

 
 
 
 
 
 
 
 
"
t
o
t
a
l
_
b
a
r
s
"
:
 
l
e
n
(
f
u
l
l
_
c
l
o
s
e
)
,
 
"
t
o
t
a
l
_
y
e
a
r
s
"
:
 
r
o
u
n
d
(
y
e
a
r
s
_
f
u
l
l
,
 
2
)
,

 
 
 
 
 
 
 
 
"
t
r
a
i
n
_
s
t
a
r
t
"
:
 
s
t
r
(
t
r
a
i
n
_
c
l
o
s
e
.
i
n
d
e
x
[
0
]
.
d
a
t
e
(
)
)
,
 
"
t
r
a
i
n
_
e
n
d
"
:
 
s
t
r
(
t
r
a
i
n
_
c
l
o
s
e
.
i
n
d
e
x
[
-
1
]
.
d
a
t
e
(
)
)
,

 
 
 
 
 
 
 
 
"
t
r
a
i
n
_
b
a
r
s
"
:
 
l
e
n
(
t
r
a
i
n
_
c
l
o
s
e
)
,
 
"
v
a
l
_
s
t
a
r
t
"
:
 
s
t
r
(
v
a
l
_
c
l
o
s
e
.
i
n
d
e
x
[
0
]
.
d
a
t
e
(
)
)
,

 
 
 
 
 
 
 
 
"
v
a
l
_
e
n
d
"
:
 
s
t
r
(
v
a
l
_
c
l
o
s
e
.
i
n
d
e
x
[
-
1
]
.
d
a
t
e
(
)
)
,
 
"
v
a
l
_
b
a
r
s
"
:
 
l
e
n
(
v
a
l
_
c
l
o
s
e
)
,
 
"
t
r
a
i
n
_
r
a
t
i
o
"
:
 
0
.
6
0
,

 
 
 
 
 
 
 
 
"
i
n
i
t
_
c
a
s
h
"
:
 
I
N
I
T
_
C
A
S
H
,
 
"
f
e
e
s
_
p
c
t
"
:
 
F
E
E
S
,
 
"
s
l
i
p
p
a
g
e
_
p
c
t
"
:
 
S
L
I
P
P
A
G
E
,
 
"
f
r
e
q
u
e
n
c
y
"
:
 
F
R
E
Q
,

 
 
 
 
 
 
 
 
"
f
i
r
s
t
_
c
l
o
s
e
"
:
 
r
o
u
n
d
(
f
l
o
a
t
(
f
u
l
l
_
c
l
o
s
e
.
i
l
o
c
[
0
]
)
,
 
4
)
,
 
"
l
a
s
t
_
c
l
o
s
e
"
:
 
r
o
u
n
d
(
f
l
o
a
t
(
f
u
l
l
_
c
l
o
s
e
.
i
l
o
c
[
-
1
]
)
,
 
4
)
,

 
 
 
 
 
 
 
 
"
p
y
t
h
o
n
_
v
e
r
s
i
o
n
"
:
 
s
y
s
.
v
e
r
s
i
o
n
.
s
p
l
i
t
(
)
[
0
]
,
 
"
e
n
v
i
r
o
n
m
e
n
t
"
:
 
"
c
o
l
a
b
_
p
r
o
"
 
i
f
 
I
N
_
C
O
L
A
B
 
e
l
s
e
 
"
l
o
c
a
l
"
,

 
 
 
 
 
 
 
 
"
g
r
i
d
_
c
o
m
b
o
s
_
t
e
s
t
e
d
"
:
 
l
e
n
(
r
e
s
u
l
t
s
_
d
f
)
,
 
"
p
a
r
a
m
_
c
o
l
u
m
n
s
"
:
 
P
A
R
A
M
_
C
O
L
S
,
 
"
n
o
t
e
s
"
:
 
N
O
T
E
S
,

 
 
 
 
}
,

 
 
 
 
"
b
e
s
t
_
p
a
r
a
m
s
"
:
 
b
e
s
t
_
p
a
r
a
m
s
,

 
 
 
 
"
m
e
t
r
i
c
s
_
f
u
l
l
_
s
a
m
p
l
e
"
:
 
{
k
:
 
v
 
f
o
r
 
k
,
 
v
 
i
n
 
M
.
i
t
e
m
s
(
)
 
i
f
 
n
o
t
 
k
.
s
t
a
r
t
s
w
i
t
h
(
'
i
s
_
'
)
 
a
n
d
 
n
o
t
 
k
.
s
t
a
r
t
s
w
i
t
h
(
'
o
o
s
_
'
)
 
a
n
d
 
n
o
t
 
k
.
s
t
a
r
t
s
w
i
t
h
(
'
b
h
_
'
)
}
,

 
 
 
 
"
m
e
t
r
i
c
s
_
i
n
_
s
a
m
p
l
e
"
:
 
{
k
.
r
e
p
l
a
c
e
(
'
i
s
_
'
,
'
'
)
:
 
v
 
f
o
r
 
k
,
 
v
 
i
n
 
M
.
i
t
e
m
s
(
)
 
i
f
 
k
.
s
t
a
r
t
s
w
i
t
h
(
'
i
s
_
'
)
}
,

 
 
 
 
"
m
e
t
r
i
c
s
_
o
u
t
_
o
f
_
s
a
m
p
l
e
"
:
 
{
k
.
r
e
p
l
a
c
e
(
'
o
o
s
_
'
,
'
'
)
:
 
v
 
f
o
r
 
k
,
 
v
 
i
n
 
M
.
i
t
e
m
s
(
)
 
i
f
 
k
.
s
t
a
r
t
s
w
i
t
h
(
'
o
o
s
_
'
)
}
,

 
 
 
 
"
m
e
t
r
i
c
s
_
b
u
y
_
h
o
l
d
"
:
 
{
k
.
r
e
p
l
a
c
e
(
'
b
h
_
'
,
'
'
)
:
 
v
 
f
o
r
 
k
,
 
v
 
i
n
 
M
.
i
t
e
m
s
(
)
 
i
f
 
k
.
s
t
a
r
t
s
w
i
t
h
(
'
b
h
_
'
)
}
,

 
 
 
 
"
g
r
i
d
_
s
e
a
r
c
h
_
s
u
m
m
a
r
y
"
:
 
{

 
 
 
 
 
 
 
 
"
t
o
p
5
"
:
 
r
e
s
u
l
t
s
_
d
f
.
n
l
a
r
g
e
s
t
(
5
,
 
'
s
h
a
r
p
e
_
r
a
t
i
o
'
)
[
P
A
R
A
M
_
C
O
L
S
 
+
 
[
'
s
h
a
r
p
e
_
r
a
t
i
o
'
,
'
t
o
t
a
l
_
r
e
t
u
r
n
'
,
'
m
a
x
_
d
r
a
w
d
o
w
n
'
]
]
.
t
o
_
d
i
c
t
(
'
r
e
c
o
r
d
s
'
)
,

 
 
 
 
}

}


#
 
S
a
v
e
 
J
S
O
N

w
i
t
h
 
o
p
e
n
(
o
s
.
p
a
t
h
.
j
o
i
n
(
L
A
T
E
S
T
_
D
I
R
,
 
"
s
u
m
m
a
r
y
.
j
s
o
n
"
)
,
 
'
w
'
)
 
a
s
 
f
:

 
 
 
 
j
s
o
n
.
d
u
m
p
(
e
x
p
o
r
t
_
j
s
o
n
,
 
f
,
 
i
n
d
e
n
t
=
2
,
 
d
e
f
a
u
l
t
=
s
t
r
)

w
i
t
h
 
o
p
e
n
(
o
s
.
p
a
t
h
.
j
o
i
n
(
A
R
C
H
I
V
E
_
D
I
R
,
 
f
"
{
R
U
N
_
I
D
}
_
s
u
m
m
a
r
y
.
j
s
o
n
"
)
,
 
'
w
'
)
 
a
s
 
f
:

 
 
 
 
j
s
o
n
.
d
u
m
p
(
e
x
p
o
r
t
_
j
s
o
n
,
 
f
,
 
i
n
d
e
n
t
=
2
,
 
d
e
f
a
u
l
t
=
s
t
r
)

p
r
i
n
t
(
f
"
\
u
2
7
0
5
 
s
u
m
m
a
r
y
.
j
s
o
n
"
)


#
 
S
a
v
e
 
C
S
V
s

p
d
.
D
a
t
a
F
r
a
m
e
(
{
'
d
a
t
e
'
:
 
f
u
l
l
_
c
l
o
s
e
.
i
n
d
e
x
.
s
t
r
f
t
i
m
e
(
'
%
Y
-
%
m
-
%
d
'
)
,
 
'
s
t
r
a
t
e
g
y
_
r
e
t
u
r
n
'
:
 
d
a
i
l
y
_
r
e
t
s
.
v
a
l
u
e
s
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
'
c
l
o
s
e
'
:
 
f
u
l
l
_
c
l
o
s
e
.
v
a
l
u
e
s
,
 
'
p
o
r
t
f
o
l
i
o
_
v
a
l
u
e
'
:
 
p
f
_
f
u
l
l
.
v
a
l
u
e
(
)
.
v
a
l
u
e
s

}
)
.
t
o
_
c
s
v
(
o
s
.
p
a
t
h
.
j
o
i
n
(
L
A
T
E
S
T
_
D
I
R
,
 
"
d
a
i
l
y
_
r
e
t
u
r
n
s
.
c
s
v
"
)
,
 
i
n
d
e
x
=
F
a
l
s
e
)

p
r
i
n
t
(
f
"
\
u
2
7
0
5
 
d
a
i
l
y
_
r
e
t
u
r
n
s
.
c
s
v
"
)


p
d
.
D
a
t
a
F
r
a
m
e
(
{
'
t
r
a
d
e
_
n
u
m
'
:
 
r
a
n
g
e
(
1
,
 
l
e
n
(
t
r
)
+
1
)
,
 
'
r
e
t
u
r
n
_
p
c
t
'
:
 
t
r
*
1
0
0
,
 
'
p
n
l
_
u
s
d
'
:
 
p
n
l
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
'
c
u
m
u
l
a
t
i
v
e
_
p
n
l
'
:
 
n
p
.
c
u
m
s
u
m
(
p
n
l
)
,
 
'
i
s
_
w
i
n
n
e
r
'
:
 
t
r
 
>
 
0

}
)
.
t
o
_
c
s
v
(
o
s
.
p
a
t
h
.
j
o
i
n
(
L
A
T
E
S
T
_
D
I
R
,
 
"
t
r
a
d
e
s
.
c
s
v
"
)
,
 
i
n
d
e
x
=
F
a
l
s
e
)

p
r
i
n
t
(
f
"
\
u
2
7
0
5
 
t
r
a
d
e
s
.
c
s
v
"
)


r
e
s
u
l
t
s
_
d
f
.
t
o
_
c
s
v
(
o
s
.
p
a
t
h
.
j
o
i
n
(
L
A
T
E
S
T
_
D
I
R
,
 
"
g
r
i
d
_
r
e
s
u
l
t
s
.
c
s
v
"
)
,
 
i
n
d
e
x
=
F
a
l
s
e
)

p
r
i
n
t
(
f
"
\
u
2
7
0
5
 
g
r
i
d
_
r
e
s
u
l
t
s
.
c
s
v
"
)


#
 
R
u
n
 
l
o
g

l
o
g
_
p
a
t
h
 
=
 
o
s
.
p
a
t
h
.
j
o
i
n
(
E
X
P
O
R
T
_
D
I
R
,
 
"
r
u
n
_
l
o
g
.
c
s
v
"
)

l
o
g
_
e
n
t
r
y
 
=
 
p
d
.
D
a
t
a
F
r
a
m
e
(
[
{

 
 
 
 
"
r
u
n
_
i
d
"
:
 
R
U
N
_
I
D
,
 
"
t
i
m
e
s
t
a
m
p
"
:
 
R
U
N
_
T
I
M
E
S
T
A
M
P
.
i
s
o
f
o
r
m
a
t
(
)
,
 
"
s
t
r
a
t
e
g
y
"
:
 
S
T
R
A
T
E
G
Y
_
N
A
M
E
,

 
 
 
 
"
t
i
c
k
e
r
"
:
 
T
I
C
K
E
R
,
 
"
b
e
s
t
_
p
a
r
a
m
s
"
:
 
s
t
r
(
b
e
s
t
_
p
a
r
a
m
s
)
,

 
 
 
 
"
s
h
a
r
p
e
_
f
u
l
l
"
:
 
r
o
u
n
d
(
M
[
'
s
h
a
r
p
e
'
]
 
o
r
 
0
,
 
4
)
,
 
"
s
h
a
r
p
e
_
i
s
"
:
 
r
o
u
n
d
(
M
[
'
i
s
_
s
h
a
r
p
e
'
]
 
o
r
 
0
,
 
4
)
,

 
 
 
 
"
s
h
a
r
p
e
_
o
o
s
"
:
 
r
o
u
n
d
(
M
[
'
o
o
s
_
s
h
a
r
p
e
'
]
 
o
r
 
0
,
 
4
)
,
 
"
t
o
t
a
l
_
r
e
t
u
r
n
"
:
 
r
o
u
n
d
(
M
[
'
t
o
t
a
l
_
r
e
t
u
r
n
'
]
 
o
r
 
0
,
 
4
)
,

 
 
 
 
"
m
a
x
_
d
r
a
w
d
o
w
n
"
:
 
r
o
u
n
d
(
M
[
'
m
a
x
_
d
d
'
]
 
o
r
 
0
,
 
4
)
,
 
"
t
o
t
a
l
_
t
r
a
d
e
s
"
:
 
M
[
'
t
r
a
d
e
s
'
]
,

 
 
 
 
"
w
i
n
_
r
a
t
e
"
:
 
r
o
u
n
d
(
M
[
'
w
i
n
_
r
a
t
e
'
]
 
o
r
 
0
,
 
1
)
,
 
"
p
r
o
f
i
t
_
f
a
c
t
o
r
"
:
 
r
o
u
n
d
(
M
[
'
p
f
'
]
 
o
r
 
0
,
 
2
)
 
i
f
 
M
[
'
p
f
'
]
 
e
l
s
e
 
N
o
n
e
,

 
 
 
 
"
n
o
t
e
s
"
:
 
N
O
T
E
S
,
 
"
e
x
p
o
r
t
_
p
a
t
h
"
:
 
S
T
R
A
T
_
D
I
R
,

}
]
)

i
f
 
o
s
.
p
a
t
h
.
e
x
i
s
t
s
(
l
o
g
_
p
a
t
h
)
:

 
 
 
 
l
o
g
_
c
o
m
b
i
n
e
d
 
=
 
p
d
.
c
o
n
c
a
t
(
[
p
d
.
r
e
a
d
_
c
s
v
(
l
o
g
_
p
a
t
h
)
,
 
l
o
g
_
e
n
t
r
y
]
,
 
i
g
n
o
r
e
_
i
n
d
e
x
=
T
r
u
e
)

e
l
s
e
:

 
 
 
 
l
o
g
_
c
o
m
b
i
n
e
d
 
=
 
l
o
g
_
e
n
t
r
y

l
o
g
_
c
o
m
b
i
n
e
d
.
t
o
_
c
s
v
(
l
o
g
_
p
a
t
h
,
 
i
n
d
e
x
=
F
a
l
s
e
)

p
r
i
n
t
(
f
"
\
u
2
7
0
5
 
r
u
n
_
l
o
g
.
c
s
v
 
(
{
l
e
n
(
l
o
g
_
c
o
m
b
i
n
e
d
)
}
 
r
u
n
s
)
"
)


#
 
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═

#
 
2
.
 
G
E
N
E
R
A
T
E
 
P
D
F
 
T
E
A
R
S
H
E
E
T
 
—
 
C
l
e
a
n
 
W
h
i
t
e
 
C
a
r
d
 
D
e
s
i
g
n

#
 
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═

p
d
f
_
p
a
t
h
 
=
 
o
s
.
p
a
t
h
.
j
o
i
n
(
L
A
T
E
S
T
_
D
I
R
,
 
"
t
e
a
r
s
h
e
e
t
.
p
d
f
"
)

p
d
f
_
a
r
c
h
i
v
e
 
=
 
o
s
.
p
a
t
h
.
j
o
i
n
(
A
R
C
H
I
V
E
_
D
I
R
,
 
f
"
{
R
U
N
_
I
D
}
_
t
e
a
r
s
h
e
e
t
.
p
d
f
"
)


f
m
t
 
=
 
l
a
m
b
d
a
 
v
,
 
d
=
2
:
 
f
"
{
v
:
.
{
d
}
f
}
"
 
i
f
 
v
 
i
s
 
n
o
t
 
N
o
n
e
 
a
n
d
 
n
o
t
 
n
p
.
i
s
n
a
n
(
v
)
 
e
l
s
e
 
"
N
/
A
"

f
m
t
p
 
=
 
l
a
m
b
d
a
 
v
:
 
f
"
{
v
:
.
2
%
}
"
 
i
f
 
v
 
i
s
 
n
o
t
 
N
o
n
e
 
a
n
d
 
n
o
t
 
n
p
.
i
s
n
a
n
(
v
)
 
e
l
s
e
 
"
N
/
A
"


#
 
─
─
 
D
e
s
i
g
n
 
t
o
k
e
n
s
 
─
─

B
G
 
 
 
 
 
 
 
=
 
'
#
F
F
F
F
F
F
'

C
A
R
D
_
B
G
 
 
=
 
'
#
F
7
F
8
F
A
'

C
A
R
D
_
B
R
D
 
=
 
'
#
E
2
E
5
E
B
'

T
E
X
T
_
P
R
I
 
=
 
'
#
1
A
1
D
2
3
'

T
E
X
T
_
S
E
C
 
=
 
'
#
5
A
6
2
7
0
'

T
E
X
T
_
M
U
T
 
=
 
'
#
8
C
9
5
A
3
'

A
C
C
E
N
T
 
 
 
=
 
'
#
2
5
6
3
E
B
'
 
 
 
#
 
B
l
u
e

G
R
E
E
N
 
 
 
 
=
 
'
#
0
5
9
6
6
9
'

R
E
D
 
 
 
 
 
 
=
 
'
#
D
C
2
6
2
6
'

O
R
A
N
G
E
 
 
 
=
 
'
#
E
A
5
8
0
C
'

G
R
I
D
_
C
L
R
 
=
 
'
#
E
5
E
7
E
B
'


d
e
f
 
_
d
r
a
w
_
c
a
r
d
(
a
x
_
f
i
g
,
 
x
,
 
y
,
 
w
,
 
h
,
 
l
a
b
e
l
,
 
v
a
l
u
e
,
 
c
o
l
o
r
=
A
C
C
E
N
T
,
 
f
o
n
t
s
i
z
e
_
v
a
l
=
2
2
)
:

 
 
 
 
"
"
"
D
r
a
w
 
a
 
r
o
u
n
d
e
d
 
m
e
t
r
i
c
 
c
a
r
d
 
o
n
 
t
h
e
 
f
i
g
u
r
e
.
"
"
"

 
 
 
 
f
r
o
m
 
m
a
t
p
l
o
t
l
i
b
.
p
a
t
c
h
e
s
 
i
m
p
o
r
t
 
F
a
n
c
y
B
b
o
x
P
a
t
c
h

 
 
 
 
r
e
c
t
 
=
 
F
a
n
c
y
B
b
o
x
P
a
t
c
h
(
(
x
,
 
y
)
,
 
w
,
 
h
,
 
b
o
x
s
t
y
l
e
=
"
r
o
u
n
d
,
p
a
d
=
0
.
0
0
8
"
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
f
a
c
e
c
o
l
o
r
=
C
A
R
D
_
B
G
,
 
e
d
g
e
c
o
l
o
r
=
C
A
R
D
_
B
R
D
,
 
l
i
n
e
w
i
d
t
h
=
1
.
2
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
t
r
a
n
s
f
o
r
m
=
a
x
_
f
i
g
.
t
r
a
n
s
A
x
e
s
,
 
z
o
r
d
e
r
=
2
)

 
 
 
 
a
x
_
f
i
g
.
a
d
d
_
p
a
t
c
h
(
r
e
c
t
)

 
 
 
 
a
x
_
f
i
g
.
t
e
x
t
(
x
 
+
 
w
/
2
,
 
y
 
+
 
h
*
0
.
6
8
,
 
v
a
l
u
e
,
 
h
a
=
'
c
e
n
t
e
r
'
,
 
v
a
=
'
c
e
n
t
e
r
'
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
f
o
n
t
s
i
z
e
=
f
o
n
t
s
i
z
e
_
v
a
l
,
 
f
o
n
t
w
e
i
g
h
t
=
'
b
o
l
d
'
,
 
c
o
l
o
r
=
c
o
l
o
r
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
t
r
a
n
s
f
o
r
m
=
a
x
_
f
i
g
.
t
r
a
n
s
A
x
e
s
,
 
z
o
r
d
e
r
=
3
)

 
 
 
 
a
x
_
f
i
g
.
t
e
x
t
(
x
 
+
 
w
/
2
,
 
y
 
+
 
h
*
0
.
2
5
,
 
l
a
b
e
l
,
 
h
a
=
'
c
e
n
t
e
r
'
,
 
v
a
=
'
c
e
n
t
e
r
'
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
f
o
n
t
s
i
z
e
=
8
,
 
c
o
l
o
r
=
T
E
X
T
_
S
E
C
,
 
t
r
a
n
s
f
o
r
m
=
a
x
_
f
i
g
.
t
r
a
n
s
A
x
e
s
,
 
z
o
r
d
e
r
=
3
)


d
e
f
 
_
s
t
y
l
e
_
a
x
(
a
x
,
 
t
i
t
l
e
=
N
o
n
e
)
:

 
 
 
 
"
"
"
A
p
p
l
y
 
c
l
e
a
n
 
w
h
i
t
e
 
s
t
y
l
i
n
g
 
t
o
 
a
n
 
a
x
i
s
.
"
"
"

 
 
 
 
a
x
.
s
e
t
_
f
a
c
e
c
o
l
o
r
(
B
G
)

 
 
 
 
a
x
.
t
i
c
k
_
p
a
r
a
m
s
(
c
o
l
o
r
s
=
T
E
X
T
_
S
E
C
,
 
l
a
b
e
l
s
i
z
e
=
8
)

 
 
 
 
a
x
.
g
r
i
d
(
T
r
u
e
,
 
a
l
p
h
a
=
0
.
4
,
 
c
o
l
o
r
=
G
R
I
D
_
C
L
R
,
 
l
i
n
e
w
i
d
t
h
=
0
.
8
)

 
 
 
 
f
o
r
 
s
p
i
n
e
 
i
n
 
a
x
.
s
p
i
n
e
s
.
v
a
l
u
e
s
(
)
:

 
 
 
 
 
 
 
 
s
p
i
n
e
.
s
e
t
_
c
o
l
o
r
(
C
A
R
D
_
B
R
D
)

 
 
 
 
 
 
 
 
s
p
i
n
e
.
s
e
t
_
l
i
n
e
w
i
d
t
h
(
0
.
8
)

 
 
 
 
i
f
 
t
i
t
l
e
:

 
 
 
 
 
 
 
 
a
x
.
s
e
t
_
t
i
t
l
e
(
t
i
t
l
e
,
 
c
o
l
o
r
=
T
E
X
T
_
P
R
I
,
 
f
o
n
t
s
i
z
e
=
1
1
,
 
f
o
n
t
w
e
i
g
h
t
=
'
b
o
l
d
'
,
 
p
a
d
=
1
0
)


w
i
t
h
 
P
d
f
P
a
g
e
s
(
p
d
f
_
p
a
t
h
)
 
a
s
 
p
d
f
:


 
 
 
 
#
 
─
─
 
P
A
G
E
 
1
:
 
D
a
s
h
b
o
a
r
d
 
w
i
t
h
 
M
e
t
r
i
c
 
C
a
r
d
s
 
+
 
S
u
m
m
a
r
y
 
T
a
b
l
e
 
─
─

 
 
 
 
f
i
g
 
=
 
p
l
t
.
f
i
g
u
r
e
(
f
i
g
s
i
z
e
=
(
1
1
,
 
8
.
5
)
)

 
 
 
 
f
i
g
.
p
a
t
c
h
.
s
e
t
_
f
a
c
e
c
o
l
o
r
(
B
G
)

 
 
 
 
a
x
 
=
 
f
i
g
.
a
d
d
_
a
x
e
s
(
[
0
,
 
0
,
 
1
,
 
1
]
)

 
 
 
 
a
x
.
s
e
t
_
x
l
i
m
(
0
,
 
1
)
;
 
a
x
.
s
e
t
_
y
l
i
m
(
0
,
 
1
)
;
 
a
x
.
a
x
i
s
(
'
o
f
f
'
)


 
 
 
 
#
 
H
e
a
d
e
r
 
b
a
r

 
 
 
 
f
r
o
m
 
m
a
t
p
l
o
t
l
i
b
.
p
a
t
c
h
e
s
 
i
m
p
o
r
t
 
F
a
n
c
y
B
b
o
x
P
a
t
c
h
,
 
R
e
c
t
a
n
g
l
e

 
 
 
 
h
e
a
d
e
r
 
=
 
R
e
c
t
a
n
g
l
e
(
(
0
,
 
0
.
9
1
)
,
 
1
,
 
0
.
0
9
,
 
f
a
c
e
c
o
l
o
r
=
A
C
C
E
N
T
,
 
t
r
a
n
s
f
o
r
m
=
a
x
.
t
r
a
n
s
A
x
e
s
,
 
z
o
r
d
e
r
=
1
)

 
 
 
 
a
x
.
a
d
d
_
p
a
t
c
h
(
h
e
a
d
e
r
)

 
 
 
 
a
x
.
t
e
x
t
(
0
.
0
4
,
 
0
.
9
5
5
,
 
"
S
T
R
A
T
E
G
Y
 
T
E
A
R
S
H
E
E
T
"
,
 
h
a
=
'
l
e
f
t
'
,
 
v
a
=
'
c
e
n
t
e
r
'
,
 
f
o
n
t
s
i
z
e
=
2
0
,

 
 
 
 
 
 
 
 
 
 
 
 
f
o
n
t
w
e
i
g
h
t
=
'
b
o
l
d
'
,
 
c
o
l
o
r
=
'
w
h
i
t
e
'
,
 
t
r
a
n
s
f
o
r
m
=
a
x
.
t
r
a
n
s
A
x
e
s
,
 
z
o
r
d
e
r
=
2
)

 
 
 
 
a
x
.
t
e
x
t
(
0
.
9
6
,
 
0
.
9
5
5
,
 
f
"
{
S
T
R
A
T
E
G
Y
_
N
A
M
E
}
 
 
|
 
 
{
T
I
C
K
E
R
}
"
,
 
h
a
=
'
r
i
g
h
t
'
,
 
v
a
=
'
c
e
n
t
e
r
'
,

 
 
 
 
 
 
 
 
 
 
 
 
f
o
n
t
s
i
z
e
=
1
2
,
 
c
o
l
o
r
=
'
r
g
b
a
(
2
5
5
,
2
5
5
,
2
5
5
,
0
.
8
5
)
'
,
 
t
r
a
n
s
f
o
r
m
=
a
x
.
t
r
a
n
s
A
x
e
s
,
 
z
o
r
d
e
r
=
2
,
 
f
a
m
i
l
y
=
'
m
o
n
o
s
p
a
c
e
'
)

 
 
 
 
a
x
.
t
e
x
t
(
0
.
9
6
,
 
0
.
9
2
,
 
f
"
{
f
u
l
l
_
c
l
o
s
e
.
i
n
d
e
x
[
0
]
.
d
a
t
e
(
)
}
 
t
o
 
{
f
u
l
l
_
c
l
o
s
e
.
i
n
d
e
x
[
-
1
]
.
d
a
t
e
(
)
}
 
 
|
 
 
{
l
e
n
(
f
u
l
l
_
c
l
o
s
e
)
}
 
b
a
r
s
 
 
|
 
 
{
p
a
r
a
m
_
s
t
r
}
"
,

 
 
 
 
 
 
 
 
 
 
 
 
h
a
=
'
r
i
g
h
t
'
,
 
v
a
=
'
c
e
n
t
e
r
'
,
 
f
o
n
t
s
i
z
e
=
8
,
 
c
o
l
o
r
=
'
r
g
b
a
(
2
5
5
,
2
5
5
,
2
5
5
,
0
.
6
5
)
'
,
 
t
r
a
n
s
f
o
r
m
=
a
x
.
t
r
a
n
s
A
x
e
s
,
 
z
o
r
d
e
r
=
2
)


 
 
 
 
#
 
M
e
t
r
i
c
 
c
a
r
d
s
 
r
o
w
 
—
 
t
o
p
 
K
P
I
s

 
 
 
 
c
a
r
d
_
w
,
 
c
a
r
d
_
h
 
=
 
0
.
1
4
5
,
 
0
.
0
9

 
 
 
 
c
a
r
d
_
y
 
=
 
0
.
7
9

 
 
 
 
c
a
r
d
s
 
=
 
[

 
 
 
 
 
 
 
 
(
"
S
H
A
R
P
E
"
,
 
f
m
t
(
M
[
'
s
h
a
r
p
e
'
]
,
 
3
)
,
 
A
C
C
E
N
T
)
,

 
 
 
 
 
 
 
 
(
"
R
E
T
U
R
N
"
,
 
f
m
t
p
(
M
[
'
t
o
t
a
l
_
r
e
t
u
r
n
'
]
)
,
 
G
R
E
E
N
 
i
f
 
(
M
[
'
t
o
t
a
l
_
r
e
t
u
r
n
'
]
 
o
r
 
0
)
 
>
 
0
 
e
l
s
e
 
R
E
D
)
,

 
 
 
 
 
 
 
 
(
"
M
A
X
 
D
D
"
,
 
f
m
t
p
(
M
[
'
m
a
x
_
d
d
'
]
)
,
 
R
E
D
 
i
f
 
(
M
[
'
m
a
x
_
d
d
'
]
 
o
r
 
0
)
 
<
 
-
0
.
1
5
 
e
l
s
e
 
O
R
A
N
G
E
)
,

 
 
 
 
 
 
 
 
(
"
W
I
N
 
R
A
T
E
"
,
 
f
"
{
M
[
'
w
i
n
_
r
a
t
e
'
]
:
.
1
f
}
%
"
 
i
f
 
M
[
'
w
i
n
_
r
a
t
e
'
]
 
e
l
s
e
 
"
N
/
A
"
,
 
G
R
E
E
N
 
i
f
 
(
M
[
'
w
i
n
_
r
a
t
e
'
]
 
o
r
 
0
)
 
>
 
5
0
 
e
l
s
e
 
R
E
D
)
,

 
 
 
 
 
 
 
 
(
"
T
R
A
D
E
S
"
,
 
s
t
r
(
M
[
'
t
r
a
d
e
s
'
]
)
,
 
T
E
X
T
_
P
R
I
)
,

 
 
 
 
 
 
 
 
(
"
P
R
O
F
I
T
 
F
A
C
T
O
R
"
,
 
f
m
t
(
M
[
'
p
f
'
]
)
,
 
G
R
E
E
N
 
i
f
 
(
M
[
'
p
f
'
]
 
o
r
 
0
)
 
>
 
1
.
5
 
e
l
s
e
 
(
O
R
A
N
G
E
 
i
f
 
(
M
[
'
p
f
'
]
 
o
r
 
0
)
 
>
 
1
 
e
l
s
e
 
R
E
D
)
)
,

 
 
 
 
]

 
 
 
 
x
_
s
t
a
r
t
 
=
 
0
.
0
3

 
 
 
 
g
a
p
 
=
 
(
0
.
9
4
 
-
 
l
e
n
(
c
a
r
d
s
)
 
*
 
c
a
r
d
_
w
)
 
/
 
(
l
e
n
(
c
a
r
d
s
)
 
-
 
1
)

 
 
 
 
f
o
r
 
i
,
 
(
l
a
b
e
l
,
 
v
a
l
u
e
,
 
c
o
l
o
r
)
 
i
n
 
e
n
u
m
e
r
a
t
e
(
c
a
r
d
s
)
:

 
 
 
 
 
 
 
 
c
x
 
=
 
x
_
s
t
a
r
t
 
+
 
i
 
*
 
(
c
a
r
d
_
w
 
+
 
g
a
p
)

 
 
 
 
 
 
 
 
_
d
r
a
w
_
c
a
r
d
(
a
x
,
 
c
x
,
 
c
a
r
d
_
y
,
 
c
a
r
d
_
w
,
 
c
a
r
d
_
h
,
 
l
a
b
e
l
,
 
v
a
l
u
e
,
 
c
o
l
o
r
)


 
 
 
 
#
 
I
S
 
v
s
 
O
O
S
 
c
o
m
p
a
r
i
s
o
n
 
t
a
b
l
e

 
 
 
 
t
a
b
l
e
_
y
 
=
 
0
.
0
4

 
 
 
 
t
a
b
l
e
_
h
 
=
 
0
.
7
0

 
 
 
 
r
o
w
s
 
=
 
[

 
 
 
 
 
 
 
 
[
"
M
E
T
R
I
C
"
,
 
"
F
U
L
L
 
S
A
M
P
L
E
"
,
 
"
I
N
-
S
A
M
P
L
E
"
,
 
"
O
U
T
-
O
F
-
S
A
M
P
L
E
"
,
 
"
B
U
Y
 
&
 
H
O
L
D
"
]
,

 
 
 
 
 
 
 
 
[
"
T
o
t
a
l
 
R
e
t
u
r
n
"
,
 
f
m
t
p
(
M
[
'
t
o
t
a
l
_
r
e
t
u
r
n
'
]
)
,
 
f
m
t
p
(
M
[
'
i
s
_
r
e
t
u
r
n
'
]
)
,
 
f
m
t
p
(
M
[
'
o
o
s
_
r
e
t
u
r
n
'
]
)
,
 
f
m
t
p
(
M
[
'
b
h
_
r
e
t
u
r
n
'
]
)
]
,

 
 
 
 
 
 
 
 
[
"
S
h
a
r
p
e
 
R
a
t
i
o
"
,
 
f
m
t
(
M
[
'
s
h
a
r
p
e
'
]
,
 
3
)
,
 
f
m
t
(
M
[
'
i
s
_
s
h
a
r
p
e
'
]
,
 
3
)
,
 
f
m
t
(
M
[
'
o
o
s
_
s
h
a
r
p
e
'
]
,
 
3
)
,
 
f
m
t
(
M
[
'
b
h
_
s
h
a
r
p
e
'
]
,
 
3
)
]
,

 
 
 
 
 
 
 
 
[
"
S
o
r
t
i
n
o
 
R
a
t
i
o
"
,
 
f
m
t
(
M
[
'
s
o
r
t
i
n
o
'
]
,
 
3
)
,
 
"
—
"
,
 
"
—
"
,
 
"
—
"
]
,

 
 
 
 
 
 
 
 
[
"
M
a
x
 
D
r
a
w
d
o
w
n
"
,
 
f
m
t
p
(
M
[
'
m
a
x
_
d
d
'
]
)
,
 
f
m
t
p
(
M
[
'
i
s
_
d
d
'
]
)
,
 
f
m
t
p
(
M
[
'
o
o
s
_
d
d
'
]
)
,
 
f
m
t
p
(
M
[
'
b
h
_
d
d
'
]
)
]
,

 
 
 
 
 
 
 
 
[
"
C
a
l
m
a
r
 
R
a
t
i
o
"
,
 
f
m
t
(
M
[
'
c
a
l
m
a
r
'
]
,
 
3
)
,
 
"
—
"
,
 
"
—
"
,
 
"
—
"
]
,

 
 
 
 
 
 
 
 
[
"
V
o
l
a
t
i
l
i
t
y
 
(
A
n
n
.
)
"
,
 
f
m
t
p
(
M
[
'
v
o
l
a
t
i
l
i
t
y
'
]
)
,
 
"
—
"
,
 
"
—
"
,
 
"
—
"
]
,

 
 
 
 
 
 
 
 
[
"
W
i
n
 
R
a
t
e
"
,
 
f
"
{
M
[
'
w
i
n
_
r
a
t
e
'
]
:
.
1
f
}
%
"
 
i
f
 
M
[
'
w
i
n
_
r
a
t
e
'
]
 
e
l
s
e
 
"
N
/
A
"
,
 
"
—
"
,
 
"
—
"
,
 
"
—
"
]
,

 
 
 
 
 
 
 
 
[
"
P
r
o
f
i
t
 
F
a
c
t
o
r
"
,
 
f
m
t
(
M
[
'
p
f
'
]
)
,
 
"
—
"
,
 
"
—
"
,
 
"
—
"
]
,

 
 
 
 
 
 
 
 
[
"
E
x
p
e
c
t
a
n
c
y
"
,
 
f
m
t
(
M
[
'
e
x
p
e
c
t
a
n
c
y
'
]
,
 
4
)
,
 
"
—
"
,
 
"
—
"
,
 
"
—
"
]
,

 
 
 
 
 
 
 
 
[
"
P
a
y
o
f
f
 
R
a
t
i
o
"
,
 
f
m
t
(
M
[
'
p
a
y
o
f
f
'
]
)
,
 
"
—
"
,
 
"
—
"
,
 
"
—
"
]
,

 
 
 
 
 
 
 
 
[
"
T
o
t
a
l
 
T
r
a
d
e
s
"
,
 
s
t
r
(
M
[
'
t
r
a
d
e
s
'
]
)
,
 
s
t
r
(
M
[
'
i
s
_
t
r
a
d
e
s
'
]
)
,
 
s
t
r
(
M
[
'
o
o
s
_
t
r
a
d
e
s
'
]
)
,
 
"
1
"
]
,

 
 
 
 
 
 
 
 
[
"
T
r
a
d
e
s
/
Y
e
a
r
"
,
 
f
m
t
(
M
[
'
t
r
a
d
e
s
_
y
r
'
]
,
 
1
)
,
 
"
—
"
,
 
"
—
"
,
 
"
—
"
]
,

 
 
 
 
 
 
 
 
[
"
A
v
g
 
W
i
n
"
,
 
f
m
t
p
(
M
[
'
a
v
g
_
w
i
n
'
]
)
,
 
"
—
"
,
 
"
—
"
,
 
"
—
"
]
,

 
 
 
 
 
 
 
 
[
"
A
v
g
 
L
o
s
s
"
,
 
f
m
t
p
(
M
[
'
a
v
g
_
l
o
s
s
'
]
)
,
 
"
—
"
,
 
"
—
"
,
 
"
—
"
]
,

 
 
 
 
 
 
 
 
[
"
L
a
r
g
e
s
t
 
W
i
n
"
,
 
f
m
t
p
(
M
[
'
l
a
r
g
e
s
t
_
w
i
n
'
]
)
,
 
"
—
"
,
 
"
—
"
,
 
"
—
"
]
,

 
 
 
 
 
 
 
 
[
"
L
a
r
g
e
s
t
 
L
o
s
s
"
,
 
f
m
t
p
(
M
[
'
l
a
r
g
e
s
t
_
l
o
s
s
'
]
)
,
 
"
—
"
,
 
"
—
"
,
 
"
—
"
]
,

 
 
 
 
]


 
 
 
 
t
a
b
l
e
 
=
 
a
x
.
t
a
b
l
e
(
c
e
l
l
T
e
x
t
=
r
o
w
s
[
1
:
]
,
 
c
o
l
L
a
b
e
l
s
=
r
o
w
s
[
0
]
,
 
c
e
l
l
L
o
c
=
'
c
e
n
t
e
r
'
,
 
l
o
c
=
'
c
e
n
t
e
r
'
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
b
b
o
x
=
[
0
.
0
3
,
 
t
a
b
l
e
_
y
,
 
0
.
9
4
,
 
t
a
b
l
e
_
h
]
)

 
 
 
 
t
a
b
l
e
.
a
u
t
o
_
s
e
t
_
f
o
n
t
_
s
i
z
e
(
F
a
l
s
e
)

 
 
 
 
t
a
b
l
e
.
s
e
t
_
f
o
n
t
s
i
z
e
(
9
)

 
 
 
 
f
o
r
 
(
r
o
w
,
 
c
o
l
)
,
 
c
e
l
l
 
i
n
 
t
a
b
l
e
.
g
e
t
_
c
e
l
l
d
(
)
.
i
t
e
m
s
(
)
:

 
 
 
 
 
 
 
 
c
e
l
l
.
s
e
t
_
e
d
g
e
c
o
l
o
r
(
C
A
R
D
_
B
R
D
)

 
 
 
 
 
 
 
 
c
e
l
l
.
s
e
t
_
l
i
n
e
w
i
d
t
h
(
0
.
6
)

 
 
 
 
 
 
 
 
i
f
 
r
o
w
 
=
=
 
0
:

 
 
 
 
 
 
 
 
 
 
 
 
c
e
l
l
.
s
e
t
_
f
a
c
e
c
o
l
o
r
(
A
C
C
E
N
T
)

 
 
 
 
 
 
 
 
 
 
 
 
c
e
l
l
.
s
e
t
_
t
e
x
t
_
p
r
o
p
s
(
c
o
l
o
r
=
'
w
h
i
t
e
'
,
 
f
o
n
t
w
e
i
g
h
t
=
'
b
o
l
d
'
,
 
f
o
n
t
s
i
z
e
=
8
)

 
 
 
 
 
 
 
 
e
l
s
e
:

 
 
 
 
 
 
 
 
 
 
 
 
c
e
l
l
.
s
e
t
_
f
a
c
e
c
o
l
o
r
(
B
G
 
i
f
 
r
o
w
 
%
 
2
 
=
=
 
1
 
e
l
s
e
 
C
A
R
D
_
B
G
)

 
 
 
 
 
 
 
 
 
 
 
 
c
e
l
l
.
s
e
t
_
t
e
x
t
_
p
r
o
p
s
(
c
o
l
o
r
=
T
E
X
T
_
P
R
I
,
 
f
o
n
t
s
i
z
e
=
8
,
 
f
a
m
i
l
y
=
'
m
o
n
o
s
p
a
c
e
'
)

 
 
 
 
 
 
 
 
 
 
 
 
i
f
 
c
o
l
 
=
=
 
0
:

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
c
e
l
l
.
s
e
t
_
t
e
x
t
_
p
r
o
p
s
(
c
o
l
o
r
=
T
E
X
T
_
P
R
I
,
 
f
o
n
t
s
i
z
e
=
8
,
 
f
o
n
t
w
e
i
g
h
t
=
'
b
o
l
d
'
)

 
 
 
 
 
 
 
 
c
e
l
l
.
s
e
t
_
h
e
i
g
h
t
(
0
.
0
5
2
)


 
 
 
 
#
 
F
o
o
t
e
r

 
 
 
 
a
x
.
t
e
x
t
(
0
.
5
,
 
0
.
0
1
,
 
f
"
R
u
n
 
{
R
U
N
_
I
D
}
 
 
|
 
 
Q
S
 
F
i
n
a
n
c
e
"
,
 
h
a
=
'
c
e
n
t
e
r
'
,
 
v
a
=
'
b
o
t
t
o
m
'
,

 
 
 
 
 
 
 
 
 
 
 
 
f
o
n
t
s
i
z
e
=
7
,
 
c
o
l
o
r
=
T
E
X
T
_
M
U
T
,
 
t
r
a
n
s
f
o
r
m
=
a
x
.
t
r
a
n
s
A
x
e
s
)


 
 
 
 
p
d
f
.
s
a
v
e
f
i
g
(
f
i
g
,
 
f
a
c
e
c
o
l
o
r
=
B
G
)

 
 
 
 
p
l
t
.
c
l
o
s
e
(
f
i
g
)


 
 
 
 
#
 
─
─
 
P
A
G
E
 
2
:
 
E
q
u
i
t
y
 
C
u
r
v
e
 
+
 
D
r
a
w
d
o
w
n
 
─
─

 
 
 
 
f
i
g
,
 
(
a
x
1
,
 
a
x
2
)
 
=
 
p
l
t
.
s
u
b
p
l
o
t
s
(
2
,
 
1
,
 
f
i
g
s
i
z
e
=
(
1
1
,
 
8
.
5
)
,
 
g
r
i
d
s
p
e
c
_
k
w
=
{
'
h
e
i
g
h
t
_
r
a
t
i
o
s
'
:
 
[
3
,
 
1
]
}
)

 
 
 
 
f
i
g
.
p
a
t
c
h
.
s
e
t
_
f
a
c
e
c
o
l
o
r
(
B
G
)

 
 
 
 
f
i
g
.
s
u
p
t
i
t
l
e
(
f
'
{
S
T
R
A
T
E
G
Y
_
N
A
M
E
}
 
o
n
 
{
T
I
C
K
E
R
}
 
—
 
E
q
u
i
t
y
 
&
 
D
r
a
w
d
o
w
n
'
,
 
f
o
n
t
s
i
z
e
=
1
4
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
f
o
n
t
w
e
i
g
h
t
=
'
b
o
l
d
'
,
 
c
o
l
o
r
=
T
E
X
T
_
P
R
I
,
 
y
=
0
.
9
7
)


 
 
 
 
e
q
_
s
t
r
a
t
 
=
 
p
f
_
f
u
l
l
.
v
a
l
u
e
(
)
;
 
e
q
_
b
h
 
=
 
p
f
_
b
h
.
v
a
l
u
e
(
)


 
 
 
 
#
 
E
q
u
i
t
y
 
c
u
r
v
e
 
w
i
t
h
 
g
r
a
d
i
e
n
t
 
f
i
l
l

 
 
 
 
_
s
t
y
l
e
_
a
x
(
a
x
1
)

 
 
 
 
a
x
1
.
p
l
o
t
(
f
u
l
l
_
c
l
o
s
e
.
i
n
d
e
x
[
:
s
p
l
i
t
_
i
d
x
]
,
 
e
q
_
s
t
r
a
t
.
i
l
o
c
[
:
s
p
l
i
t
_
i
d
x
]
.
v
a
l
u
e
s
,

 
 
 
 
 
 
 
 
 
 
 
 
 
c
o
l
o
r
=
A
C
C
E
N
T
,
 
l
i
n
e
w
i
d
t
h
=
2
,
 
l
a
b
e
l
=
'
S
t
r
a
t
e
g
y
 
(
I
S
)
'
,
 
s
o
l
i
d
_
c
a
p
s
t
y
l
e
=
'
r
o
u
n
d
'
)

 
 
 
 
a
x
1
.
p
l
o
t
(
f
u
l
l
_
c
l
o
s
e
.
i
n
d
e
x
[
s
p
l
i
t
_
i
d
x
:
]
,
 
e
q
_
s
t
r
a
t
.
i
l
o
c
[
s
p
l
i
t
_
i
d
x
:
]
.
v
a
l
u
e
s
,

 
 
 
 
 
 
 
 
 
 
 
 
 
c
o
l
o
r
=
O
R
A
N
G
E
,
 
l
i
n
e
w
i
d
t
h
=
2
,
 
l
a
b
e
l
=
'
S
t
r
a
t
e
g
y
 
(
O
O
S
)
'
,
 
s
o
l
i
d
_
c
a
p
s
t
y
l
e
=
'
r
o
u
n
d
'
)

 
 
 
 
a
x
1
.
p
l
o
t
(
f
u
l
l
_
c
l
o
s
e
.
i
n
d
e
x
,
 
e
q
_
b
h
.
v
a
l
u
e
s
,
 
c
o
l
o
r
=
T
E
X
T
_
M
U
T
,
 
l
i
n
e
w
i
d
t
h
=
1
.
2
,

 
 
 
 
 
 
 
 
 
 
 
 
 
a
l
p
h
a
=
0
.
6
,
 
l
i
n
e
s
t
y
l
e
=
'
-
-
'
,
 
l
a
b
e
l
=
'
B
u
y
 
&
 
H
o
l
d
'
)

 
 
 
 
a
x
1
.
a
x
v
l
i
n
e
(
x
=
f
u
l
l
_
c
l
o
s
e
.
i
n
d
e
x
[
s
p
l
i
t
_
i
d
x
]
,
 
c
o
l
o
r
=
R
E
D
,
 
l
i
n
e
s
t
y
l
e
=
'
:
'
,
 
a
l
p
h
a
=
0
.
3
,
 
l
i
n
e
w
i
d
t
h
=
1
)

 
 
 
 
a
x
1
.
f
i
l
l
_
b
e
t
w
e
e
n
(
f
u
l
l
_
c
l
o
s
e
.
i
n
d
e
x
[
:
s
p
l
i
t
_
i
d
x
]
,
 
e
q
_
s
t
r
a
t
.
i
l
o
c
[
:
s
p
l
i
t
_
i
d
x
]
.
v
a
l
u
e
s
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
e
q
_
s
t
r
a
t
.
i
l
o
c
[
:
s
p
l
i
t
_
i
d
x
]
.
v
a
l
u
e
s
.
m
i
n
(
)
,
 
a
l
p
h
a
=
0
.
0
6
,
 
c
o
l
o
r
=
A
C
C
E
N
T
)

 
 
 
 
a
x
1
.
f
i
l
l
_
b
e
t
w
e
e
n
(
f
u
l
l
_
c
l
o
s
e
.
i
n
d
e
x
[
s
p
l
i
t
_
i
d
x
:
]
,
 
e
q
_
s
t
r
a
t
.
i
l
o
c
[
s
p
l
i
t
_
i
d
x
:
]
.
v
a
l
u
e
s
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
e
q
_
s
t
r
a
t
.
i
l
o
c
[
s
p
l
i
t
_
i
d
x
:
]
.
v
a
l
u
e
s
.
m
i
n
(
)
,
 
a
l
p
h
a
=
0
.
0
6
,
 
c
o
l
o
r
=
O
R
A
N
G
E
)

 
 
 
 
a
x
1
.
s
e
t
_
y
l
a
b
e
l
(
'
P
o
r
t
f
o
l
i
o
 
V
a
l
u
e
 
(
$
)
'
,
 
c
o
l
o
r
=
T
E
X
T
_
S
E
C
,
 
f
o
n
t
s
i
z
e
=
9
)

 
 
 
 
a
x
1
.
l
e
g
e
n
d
(
f
o
n
t
s
i
z
e
=
8
,
 
f
a
c
e
c
o
l
o
r
=
B
G
,
 
e
d
g
e
c
o
l
o
r
=
C
A
R
D
_
B
R
D
,
 
l
a
b
e
l
c
o
l
o
r
=
T
E
X
T
_
P
R
I
,
 
f
r
a
m
e
a
l
p
h
a
=
0
.
9
5
)

 
 
 
 
a
x
1
.
y
a
x
i
s
.
s
e
t
_
m
a
j
o
r
_
f
o
r
m
a
t
t
e
r
(
p
l
t
.
F
u
n
c
F
o
r
m
a
t
t
e
r
(
l
a
m
b
d
a
 
x
,
 
_
:
 
f
'
$
{
x
:
,
.
0
f
}
'
)
)


 
 
 
 
#
 
S
t
a
t
s
 
b
a
n
n
e
r

 
 
 
 
s
t
a
t
s
_
t
e
x
t
 
=
 
f
"
S
h
a
r
p
e
:
 
{
f
m
t
(
M
[
'
s
h
a
r
p
e
'
]
,
3
)
}
 
 
 
|
 
 
 
R
e
t
u
r
n
:
 
{
f
m
t
p
(
M
[
'
t
o
t
a
l
_
r
e
t
u
r
n
'
]
)
}
 
 
 
|
 
 
 
M
a
x
D
D
:
 
{
f
m
t
p
(
M
[
'
m
a
x
_
d
d
'
]
)
}
 
 
 
|
 
 
 
W
R
:
 
{
M
[
'
w
i
n
_
r
a
t
e
'
]
:
.
1
f
}
%
 
 
 
|
 
 
 
P
F
:
 
{
f
m
t
(
M
[
'
p
f
'
]
)
}
"

 
 
 
 
a
x
1
.
t
e
x
t
(
0
.
5
,
 
0
.
0
3
,
 
s
t
a
t
s
_
t
e
x
t
,
 
t
r
a
n
s
f
o
r
m
=
a
x
1
.
t
r
a
n
s
A
x
e
s
,
 
f
o
n
t
s
i
z
e
=
8
,
 
h
a
=
'
c
e
n
t
e
r
'
,

 
 
 
 
 
 
 
 
 
 
 
 
 
c
o
l
o
r
=
T
E
X
T
_
S
E
C
,
 
f
a
m
i
l
y
=
'
m
o
n
o
s
p
a
c
e
'
,

 
 
 
 
 
 
 
 
 
 
 
 
 
b
b
o
x
=
d
i
c
t
(
b
o
x
s
t
y
l
e
=
'
r
o
u
n
d
,
p
a
d
=
0
.
5
'
,
 
f
a
c
e
c
o
l
o
r
=
C
A
R
D
_
B
G
,
 
e
d
g
e
c
o
l
o
r
=
C
A
R
D
_
B
R
D
,
 
a
l
p
h
a
=
0
.
9
5
)
)


 
 
 
 
#
 
D
r
a
w
d
o
w
n

 
 
 
 
_
s
t
y
l
e
_
a
x
(
a
x
2
)

 
 
 
 
d
d
 
=
 
p
f
_
f
u
l
l
.
d
r
a
w
d
o
w
n
(
)
 
*
 
1
0
0

 
 
 
 
a
x
2
.
f
i
l
l
_
b
e
t
w
e
e
n
(
f
u
l
l
_
c
l
o
s
e
.
i
n
d
e
x
,
 
d
d
.
v
a
l
u
e
s
,
 
0
,
 
c
o
l
o
r
=
R
E
D
,
 
a
l
p
h
a
=
0
.
2
)

 
 
 
 
a
x
2
.
p
l
o
t
(
f
u
l
l
_
c
l
o
s
e
.
i
n
d
e
x
,
 
d
d
.
v
a
l
u
e
s
,
 
c
o
l
o
r
=
R
E
D
,
 
l
i
n
e
w
i
d
t
h
=
0
.
8
,
 
a
l
p
h
a
=
0
.
7
)

 
 
 
 
a
x
2
.
s
e
t
_
y
l
a
b
e
l
(
'
D
r
a
w
d
o
w
n
 
%
'
,
 
c
o
l
o
r
=
T
E
X
T
_
S
E
C
,
 
f
o
n
t
s
i
z
e
=
9
)

 
 
 
 
a
x
2
.
s
e
t
_
x
l
a
b
e
l
(
'
D
a
t
e
'
,
 
c
o
l
o
r
=
T
E
X
T
_
S
E
C
,
 
f
o
n
t
s
i
z
e
=
9
)


 
 
 
 
p
l
t
.
t
i
g
h
t
_
l
a
y
o
u
t
(
r
e
c
t
=
[
0
,
 
0
,
 
1
,
 
0
.
9
5
]
)

 
 
 
 
p
d
f
.
s
a
v
e
f
i
g
(
f
i
g
,
 
f
a
c
e
c
o
l
o
r
=
B
G
)

 
 
 
 
p
l
t
.
c
l
o
s
e
(
f
i
g
)


 
 
 
 
#
 
─
─
 
P
A
G
E
 
3
:
 
T
r
a
d
e
 
A
n
a
l
y
s
i
s
 
—
 
2
x
2
 
G
r
i
d
 
─
─

 
 
 
 
i
f
 
l
e
n
(
t
r
)
 
>
 
0
:

 
 
 
 
 
 
 
 
f
i
g
,
 
a
x
e
s
 
=
 
p
l
t
.
s
u
b
p
l
o
t
s
(
2
,
 
2
,
 
f
i
g
s
i
z
e
=
(
1
1
,
 
8
.
5
)
)

 
 
 
 
 
 
 
 
f
i
g
.
p
a
t
c
h
.
s
e
t
_
f
a
c
e
c
o
l
o
r
(
B
G
)

 
 
 
 
 
 
 
 
f
i
g
.
s
u
p
t
i
t
l
e
(
f
'
T
r
a
d
e
-
b
y
-
T
r
a
d
e
 
A
n
a
l
y
s
i
s
 
—
 
{
l
e
n
(
t
r
)
}
 
T
r
a
d
e
s
'
,
 
f
o
n
t
s
i
z
e
=
1
4
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
f
o
n
t
w
e
i
g
h
t
=
'
b
o
l
d
'
,
 
c
o
l
o
r
=
T
E
X
T
_
P
R
I
,
 
y
=
0
.
9
7
)


 
 
 
 
 
 
 
 
n
 
=
 
l
e
n
(
t
r
)

 
 
 
 
 
 
 
 
c
o
l
o
r
s
_
b
a
r
 
=
 
[
G
R
E
E
N
 
i
f
 
r
 
>
 
0
 
e
l
s
e
 
R
E
D
 
f
o
r
 
r
 
i
n
 
t
r
]

 
 
 
 
 
 
 
 
c
o
l
o
r
s
_
p
n
l
 
=
 
[
G
R
E
E
N
 
i
f
 
p
 
>
 
0
 
e
l
s
e
 
R
E
D
 
f
o
r
 
p
 
i
n
 
p
n
l
]


 
 
 
 
 
 
 
 
f
o
r
 
a
 
i
n
 
a
x
e
s
.
f
l
a
t
:

 
 
 
 
 
 
 
 
 
 
 
 
_
s
t
y
l
e
_
a
x
(
a
)


 
 
 
 
 
 
 
 
#
 
T
r
a
d
e
 
r
e
t
u
r
n
s
 
b
a
r
 
c
h
a
r
t

 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
0
]
.
b
a
r
(
r
a
n
g
e
(
n
)
,
 
t
r
*
1
0
0
,
 
c
o
l
o
r
=
c
o
l
o
r
s
_
b
a
r
,
 
e
d
g
e
c
o
l
o
r
=
'
n
o
n
e
'
,
 
w
i
d
t
h
=
0
.
8
,
 
a
l
p
h
a
=
0
.
8
5
)

 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
0
]
.
a
x
h
l
i
n
e
(
n
p
.
m
e
a
n
(
t
r
)
*
1
0
0
,
 
c
o
l
o
r
=
A
C
C
E
N
T
,
 
l
i
n
e
s
t
y
l
e
=
'
-
-
'
,
 
l
i
n
e
w
i
d
t
h
=
1
.
5
,
 
l
a
b
e
l
=
f
'
A
v
g
:
 
{
n
p
.
m
e
a
n
(
t
r
)
*
1
0
0
:
.
2
f
}
%
'
)

 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
0
]
.
a
x
h
l
i
n
e
(
0
,
 
c
o
l
o
r
=
T
E
X
T
_
M
U
T
,
 
l
i
n
e
w
i
d
t
h
=
0
.
5
)

 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
0
]
.
s
e
t
_
t
i
t
l
e
(
'
T
r
a
d
e
 
R
e
t
u
r
n
s
 
(
%
)
'
,
 
c
o
l
o
r
=
T
E
X
T
_
P
R
I
,
 
f
o
n
t
s
i
z
e
=
1
1
,
 
f
o
n
t
w
e
i
g
h
t
=
'
b
o
l
d
'
)

 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
0
]
.
s
e
t
_
x
l
a
b
e
l
(
'
T
r
a
d
e
 
#
'
,
 
c
o
l
o
r
=
T
E
X
T
_
S
E
C
,
 
f
o
n
t
s
i
z
e
=
8
)

 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
0
]
.
l
e
g
e
n
d
(
f
o
n
t
s
i
z
e
=
7
,
 
f
a
c
e
c
o
l
o
r
=
B
G
,
 
e
d
g
e
c
o
l
o
r
=
C
A
R
D
_
B
R
D
)


 
 
 
 
 
 
 
 
#
 
T
r
a
d
e
 
P
&
L
 
b
a
r
 
c
h
a
r
t

 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
1
]
.
b
a
r
(
r
a
n
g
e
(
n
)
,
 
p
n
l
,
 
c
o
l
o
r
=
c
o
l
o
r
s
_
p
n
l
,
 
e
d
g
e
c
o
l
o
r
=
'
n
o
n
e
'
,
 
w
i
d
t
h
=
0
.
8
,
 
a
l
p
h
a
=
0
.
8
5
)

 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
1
]
.
a
x
h
l
i
n
e
(
n
p
.
m
e
a
n
(
p
n
l
)
,
 
c
o
l
o
r
=
A
C
C
E
N
T
,
 
l
i
n
e
s
t
y
l
e
=
'
-
-
'
,
 
l
i
n
e
w
i
d
t
h
=
1
.
5
,
 
l
a
b
e
l
=
f
'
A
v
g
:
 
$
{
n
p
.
m
e
a
n
(
p
n
l
)
:
,
.
0
f
}
'
)

 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
1
]
.
a
x
h
l
i
n
e
(
0
,
 
c
o
l
o
r
=
T
E
X
T
_
M
U
T
,
 
l
i
n
e
w
i
d
t
h
=
0
.
5
)

 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
1
]
.
s
e
t
_
t
i
t
l
e
(
'
T
r
a
d
e
 
P
&
L
 
(
$
)
'
,
 
c
o
l
o
r
=
T
E
X
T
_
P
R
I
,
 
f
o
n
t
s
i
z
e
=
1
1
,
 
f
o
n
t
w
e
i
g
h
t
=
'
b
o
l
d
'
)

 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
1
]
.
y
a
x
i
s
.
s
e
t
_
m
a
j
o
r
_
f
o
r
m
a
t
t
e
r
(
p
l
t
.
F
u
n
c
F
o
r
m
a
t
t
e
r
(
l
a
m
b
d
a
 
x
,
 
_
:
 
f
'
$
{
x
:
,
.
0
f
}
'
)
)

 
 
 
 
 
 
 
 
a
x
e
s
[
0
,
1
]
.
l
e
g
e
n
d
(
f
o
n
t
s
i
z
e
=
7
,
 
f
a
c
e
c
o
l
o
r
=
B
G
,
 
e
d
g
e
c
o
l
o
r
=
C
A
R
D
_
B
R
D
)


 
 
 
 
 
 
 
 
#
 
C
u
m
u
l
a
t
i
v
e
 
P
&
L
 
w
i
t
h
 
g
r
a
d
i
e
n
t

 
 
 
 
 
 
 
 
c
u
m
_
p
n
l
 
=
 
n
p
.
c
u
m
s
u
m
(
p
n
l
)

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
0
]
.
p
l
o
t
(
r
a
n
g
e
(
1
,
 
n
+
1
)
,
 
c
u
m
_
p
n
l
,
 
c
o
l
o
r
=
A
C
C
E
N
T
,
 
l
i
n
e
w
i
d
t
h
=
2
.
5
,
 
s
o
l
i
d
_
c
a
p
s
t
y
l
e
=
'
r
o
u
n
d
'
)

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
0
]
.
f
i
l
l
_
b
e
t
w
e
e
n
(
r
a
n
g
e
(
1
,
 
n
+
1
)
,
 
c
u
m
_
p
n
l
,
 
0
,
 
w
h
e
r
e
=
c
u
m
_
p
n
l
>
=
0
,
 
a
l
p
h
a
=
0
.
1
2
,
 
c
o
l
o
r
=
G
R
E
E
N
)

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
0
]
.
f
i
l
l
_
b
e
t
w
e
e
n
(
r
a
n
g
e
(
1
,
 
n
+
1
)
,
 
c
u
m
_
p
n
l
,
 
0
,
 
w
h
e
r
e
=
c
u
m
_
p
n
l
<
0
,
 
a
l
p
h
a
=
0
.
1
2
,
 
c
o
l
o
r
=
R
E
D
)

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
0
]
.
a
x
h
l
i
n
e
(
0
,
 
c
o
l
o
r
=
T
E
X
T
_
M
U
T
,
 
l
i
n
e
w
i
d
t
h
=
0
.
5
)

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
0
]
.
s
e
t
_
t
i
t
l
e
(
'
C
u
m
u
l
a
t
i
v
e
 
P
&
L
'
,
 
c
o
l
o
r
=
T
E
X
T
_
P
R
I
,
 
f
o
n
t
s
i
z
e
=
1
1
,
 
f
o
n
t
w
e
i
g
h
t
=
'
b
o
l
d
'
)

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
0
]
.
s
e
t
_
x
l
a
b
e
l
(
'
T
r
a
d
e
 
#
'
,
 
c
o
l
o
r
=
T
E
X
T
_
S
E
C
,
 
f
o
n
t
s
i
z
e
=
8
)

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
0
]
.
y
a
x
i
s
.
s
e
t
_
m
a
j
o
r
_
f
o
r
m
a
t
t
e
r
(
p
l
t
.
F
u
n
c
F
o
r
m
a
t
t
e
r
(
l
a
m
b
d
a
 
x
,
 
_
:
 
f
'
$
{
x
:
,
.
0
f
}
'
)
)


 
 
 
 
 
 
 
 
#
 
R
e
t
u
r
n
 
d
i
s
t
r
i
b
u
t
i
o
n
 
h
i
s
t
o
g
r
a
m

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
1
]
.
h
i
s
t
(
t
r
*
1
0
0
,
 
b
i
n
s
=
m
i
n
(
3
0
,
 
m
a
x
(
1
0
,
 
n
/
/
3
)
)
,
 
c
o
l
o
r
=
A
C
C
E
N
T
,
 
e
d
g
e
c
o
l
o
r
=
'
w
h
i
t
e
'
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
a
l
p
h
a
=
0
.
7
5
,
 
l
i
n
e
w
i
d
t
h
=
0
.
5
)

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
1
]
.
a
x
v
l
i
n
e
(
n
p
.
m
e
a
n
(
t
r
)
*
1
0
0
,
 
c
o
l
o
r
=
R
E
D
,
 
l
i
n
e
s
t
y
l
e
=
'
-
-
'
,
 
l
i
n
e
w
i
d
t
h
=
2
,
 
l
a
b
e
l
=
f
'
M
e
a
n
:
 
{
n
p
.
m
e
a
n
(
t
r
)
*
1
0
0
:
.
2
f
}
%
'
)

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
1
]
.
a
x
v
l
i
n
e
(
0
,
 
c
o
l
o
r
=
T
E
X
T
_
M
U
T
,
 
l
i
n
e
w
i
d
t
h
=
0
.
8
,
 
a
l
p
h
a
=
0
.
5
)

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
1
]
.
s
e
t
_
t
i
t
l
e
(
'
R
e
t
u
r
n
 
D
i
s
t
r
i
b
u
t
i
o
n
'
,
 
c
o
l
o
r
=
T
E
X
T
_
P
R
I
,
 
f
o
n
t
s
i
z
e
=
1
1
,
 
f
o
n
t
w
e
i
g
h
t
=
'
b
o
l
d
'
)

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
1
]
.
s
e
t
_
x
l
a
b
e
l
(
'
R
e
t
u
r
n
 
%
'
,
 
c
o
l
o
r
=
T
E
X
T
_
S
E
C
,
 
f
o
n
t
s
i
z
e
=
8
)

 
 
 
 
 
 
 
 
a
x
e
s
[
1
,
1
]
.
l
e
g
e
n
d
(
f
o
n
t
s
i
z
e
=
7
,
 
f
a
c
e
c
o
l
o
r
=
B
G
,
 
e
d
g
e
c
o
l
o
r
=
C
A
R
D
_
B
R
D
)


 
 
 
 
 
 
 
 
p
l
t
.
t
i
g
h
t
_
l
a
y
o
u
t
(
r
e
c
t
=
[
0
,
 
0
,
 
1
,
 
0
.
9
5
]
)

 
 
 
 
 
 
 
 
p
d
f
.
s
a
v
e
f
i
g
(
f
i
g
,
 
f
a
c
e
c
o
l
o
r
=
B
G
)

 
 
 
 
 
 
 
 
p
l
t
.
c
l
o
s
e
(
f
i
g
)


 
 
 
 
#
 
─
─
 
P
A
G
E
 
4
:
 
M
o
n
t
e
 
C
a
r
l
o
 
F
T
M
O
 
S
i
m
u
l
a
t
i
o
n
 
─
─

 
 
 
 
d
r
 
=
 
d
a
i
l
y
_
r
e
t
s
.
v
a
l
u
e
s
.
r
a
v
e
l
(
)
;
 
d
r
 
=
 
d
r
[
~
n
p
.
i
s
n
a
n
(
d
r
)
]

 
 
 
 
N
_
S
I
M
 
=
 
5
0
0
0
;
 
D
A
Y
S
 
=
 
3
0
;
 
A
C
C
O
U
N
T
 
=
 
1
0
0
0
0
0

 
 
 
 
n
p
.
r
a
n
d
o
m
.
s
e
e
d
(
4
2
)

 
 
 
 
n
_
p
a
s
s
e
d
 
=
 
n
_
b
l
o
w
n
_
t
 
=
 
n
_
b
l
o
w
n
_
d
 
=
 
0

 
 
 
 
f
i
n
a
l
_
e
q
s
 
=
 
n
p
.
z
e
r
o
s
(
N
_
S
I
M
)
;
 
s
a
m
p
l
e
_
p
a
t
h
s
 
=
 
[
]


 
 
 
 
f
o
r
 
s
 
i
n
 
r
a
n
g
e
(
N
_
S
I
M
)
:

 
 
 
 
 
 
 
 
e
q
 
=
 
A
C
C
O
U
N
T
;
 
p
a
t
h
 
=
 
[
A
C
C
O
U
N
T
]

 
 
 
 
 
 
 
 
s
i
m
_
r
e
t
s
 
=
 
n
p
.
r
a
n
d
o
m
.
c
h
o
i
c
e
(
d
r
,
 
s
i
z
e
=
D
A
Y
S
,
 
r
e
p
l
a
c
e
=
T
r
u
e
)

 
 
 
 
 
 
 
 
b
l
o
w
n
 
=
 
F
a
l
s
e

 
 
 
 
 
 
 
 
f
o
r
 
d
 
i
n
 
r
a
n
g
e
(
D
A
Y
S
)
:

 
 
 
 
 
 
 
 
 
 
 
 
d
a
y
_
s
t
a
r
t
 
=
 
e
q
;
 
e
q
 
*
=
 
(
1
 
+
 
s
i
m
_
r
e
t
s
[
d
]
)
;
 
p
a
t
h
.
a
p
p
e
n
d
(
e
q
)

 
 
 
 
 
 
 
 
 
 
 
 
i
f
 
(
e
q
 
-
 
d
a
y
_
s
t
a
r
t
)
/
A
C
C
O
U
N
T
 
<
 
-
0
.
0
5
:
 
n
_
b
l
o
w
n
_
d
 
+
=
 
1
;
 
b
l
o
w
n
 
=
 
T
r
u
e
;
 
b
r
e
a
k

 
 
 
 
 
 
 
 
 
 
 
 
i
f
 
(
e
q
 
-
 
A
C
C
O
U
N
T
)
/
A
C
C
O
U
N
T
 
<
 
-
0
.
1
0
:
 
n
_
b
l
o
w
n
_
t
 
+
=
 
1
;
 
b
l
o
w
n
 
=
 
T
r
u
e
;
 
b
r
e
a
k

 
 
 
 
 
 
 
 
 
 
 
 
i
f
 
(
e
q
 
-
 
A
C
C
O
U
N
T
)
/
A
C
C
O
U
N
T
 
>
=
 
0
.
1
0
:
 
n
_
p
a
s
s
e
d
 
+
=
 
1
;
 
b
l
o
w
n
 
=
 
T
r
u
e
;
 
b
r
e
a
k

 
 
 
 
 
 
 
 
w
h
i
l
e
 
l
e
n
(
p
a
t
h
)
 
<
 
D
A
Y
S
 
+
 
1
:
 
p
a
t
h
.
a
p
p
e
n
d
(
p
a
t
h
[
-
1
]
)

 
 
 
 
 
 
 
 
f
i
n
a
l
_
e
q
s
[
s
]
 
=
 
p
a
t
h
[
-
1
]

 
 
 
 
 
 
 
 
i
f
 
s
 
<
 
1
5
0
:
 
s
a
m
p
l
e
_
p
a
t
h
s
.
a
p
p
e
n
d
(
p
a
t
h
)


 
 
 
 
n
_
s
t
i
l
l
 
=
 
N
_
S
I
M
 
-
 
n
_
p
a
s
s
e
d
 
-
 
n
_
b
l
o
w
n
_
t
 
-
 
n
_
b
l
o
w
n
_
d

 
 
 
 
p
a
s
s
_
r
a
t
e
 
=
 
n
_
p
a
s
s
e
d
 
/
 
N
_
S
I
M
 
*
 
1
0
0


 
 
 
 
v
e
r
d
i
c
t
 
=
 
"
F
A
V
O
R
A
B
L
E
"
 
i
f
 
p
a
s
s
_
r
a
t
e
 
>
=
 
5
0
 
e
l
s
e
 
"
P
O
S
S
I
B
L
E
"
 
i
f
 
p
a
s
s
_
r
a
t
e
 
>
=
 
2
5
 
e
l
s
e
 
"
C
H
A
L
L
E
N
G
I
N
G
"
 
i
f
 
p
a
s
s
_
r
a
t
e
 
>
=
 
1
0
 
e
l
s
e
 
"
U
N
L
I
K
E
L
Y
"

 
 
 
 
v
e
r
d
i
c
t
_
c
o
l
o
r
 
=
 
G
R
E
E
N
 
i
f
 
p
a
s
s
_
r
a
t
e
 
>
=
 
5
0
 
e
l
s
e
 
O
R
A
N
G
E
 
i
f
 
p
a
s
s
_
r
a
t
e
 
>
=
 
2
5
 
e
l
s
e
 
(
O
R
A
N
G
E
 
i
f
 
p
a
s
s
_
r
a
t
e
 
>
=
 
1
0
 
e
l
s
e
 
R
E
D
)


 
 
 
 
f
i
g
 
=
 
p
l
t
.
f
i
g
u
r
e
(
f
i
g
s
i
z
e
=
(
1
1
,
 
8
.
5
)
)

 
 
 
 
f
i
g
.
p
a
t
c
h
.
s
e
t
_
f
a
c
e
c
o
l
o
r
(
B
G
)


 
 
 
 
#
 
T
o
p
 
s
e
c
t
i
o
n
:
 
M
C
 
r
e
s
u
l
t
 
c
a
r
d
s

 
 
 
 
a
x
_
t
o
p
 
=
 
f
i
g
.
a
d
d
_
a
x
e
s
(
[
0
,
 
0
.
8
2
,
 
1
,
 
0
.
1
8
]
)

 
 
 
 
a
x
_
t
o
p
.
s
e
t
_
x
l
i
m
(
0
,
 
1
)
;
 
a
x
_
t
o
p
.
s
e
t
_
y
l
i
m
(
0
,
 
1
)
;
 
a
x
_
t
o
p
.
a
x
i
s
(
'
o
f
f
'
)


 
 
 
 
#
 
T
i
t
l
e

 
 
 
 
a
x
_
t
o
p
.
t
e
x
t
(
0
.
5
,
 
0
.
8
5
,
 
f
'
F
T
M
O
 
M
o
n
t
e
 
C
a
r
l
o
 
—
 
{
N
_
S
I
M
:
,
}
 
S
i
m
u
l
a
t
i
o
n
s
'
,
 
h
a
=
'
c
e
n
t
e
r
'
,
 
v
a
=
'
c
e
n
t
e
r
'
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
f
o
n
t
s
i
z
e
=
1
6
,
 
f
o
n
t
w
e
i
g
h
t
=
'
b
o
l
d
'
,
 
c
o
l
o
r
=
T
E
X
T
_
P
R
I
,
 
t
r
a
n
s
f
o
r
m
=
a
x
_
t
o
p
.
t
r
a
n
s
A
x
e
s
)


 
 
 
 
#
 
R
e
s
u
l
t
 
c
a
r
d
s

 
 
 
 
m
c
_
c
a
r
d
s
 
=
 
[

 
 
 
 
 
 
 
 
(
"
P
A
S
S
 
R
A
T
E
"
,
 
f
"
{
p
a
s
s
_
r
a
t
e
:
.
1
f
}
%
"
,
 
v
e
r
d
i
c
t
_
c
o
l
o
r
)
,

 
 
 
 
 
 
 
 
(
"
V
E
R
D
I
C
T
"
,
 
v
e
r
d
i
c
t
,
 
v
e
r
d
i
c
t
_
c
o
l
o
r
)
,

 
 
 
 
 
 
 
 
(
"
P
A
S
S
E
D
"
,
 
f
"
{
n
_
p
a
s
s
e
d
:
,
}
"
,
 
G
R
E
E
N
)
,

 
 
 
 
 
 
 
 
(
"
B
L
O
W
N
 
(
T
O
T
A
L
)
"
,
 
f
"
{
n
_
b
l
o
w
n
_
t
:
,
}
"
,
 
R
E
D
)
,

 
 
 
 
 
 
 
 
(
"
B
L
O
W
N
 
(
D
A
I
L
Y
)
"
,
 
f
"
{
n
_
b
l
o
w
n
_
d
:
,
}
"
,
 
R
E
D
)
,

 
 
 
 
 
 
 
 
(
"
S
T
I
L
L
 
T
R
A
D
I
N
G
"
,
 
f
"
{
n
_
s
t
i
l
l
:
,
}
"
,
 
T
E
X
T
_
S
E
C
)
,

 
 
 
 
]

 
 
 
 
m
c
_
c
w
 
=
 
0
.
1
4

 
 
 
 
m
c
_
g
a
p
 
=
 
(
0
.
9
2
 
-
 
l
e
n
(
m
c
_
c
a
r
d
s
)
 
*
 
m
c
_
c
w
)
 
/
 
(
l
e
n
(
m
c
_
c
a
r
d
s
)
 
-
 
1
)

 
 
 
 
f
o
r
 
i
,
 
(
l
a
b
e
l
,
 
v
a
l
u
e
,
 
c
o
l
o
r
)
 
i
n
 
e
n
u
m
e
r
a
t
e
(
m
c
_
c
a
r
d
s
)
:

 
 
 
 
 
 
 
 
c
x
 
=
 
0
.
0
4
 
+
 
i
 
*
 
(
m
c
_
c
w
 
+
 
m
c
_
g
a
p
)

 
 
 
 
 
 
 
 
_
d
r
a
w
_
c
a
r
d
(
a
x
_
t
o
p
,
 
c
x
,
 
0
.
0
5
,
 
m
c
_
c
w
,
 
0
.
6
5
,
 
l
a
b
e
l
,
 
v
a
l
u
e
,
 
c
o
l
o
r
,
 
f
o
n
t
s
i
z
e
_
v
a
l
=
1
6
)


 
 
 
 
#
 
B
o
t
t
o
m
 
s
e
c
t
i
o
n
:
 
E
q
u
i
t
y
 
p
a
t
h
s

 
 
 
 
a
x
_
m
c
 
=
 
f
i
g
.
a
d
d
_
a
x
e
s
(
[
0
.
0
8
,
 
0
.
0
8
,
 
0
.
8
6
,
 
0
.
7
0
]
)

 
 
 
 
_
s
t
y
l
e
_
a
x
(
a
x
_
m
c
)


 
 
 
 
f
o
r
 
p
a
t
h
 
i
n
 
s
a
m
p
l
e
_
p
a
t
h
s
:

 
 
 
 
 
 
 
 
c
 
=
 
G
R
E
E
N
 
i
f
 
p
a
t
h
[
-
1
]
 
>
=
 
1
1
0
0
0
0
 
e
l
s
e
 
(
R
E
D
 
i
f
 
p
a
t
h
[
-
1
]
 
<
=
 
9
0
0
0
0
 
e
l
s
e
 
T
E
X
T
_
M
U
T
)

 
 
 
 
 
 
 
 
a
x
_
m
c
.
p
l
o
t
(
r
a
n
g
e
(
D
A
Y
S
+
1
)
,
 
p
a
t
h
,
 
c
o
l
o
r
=
c
,
 
a
l
p
h
a
=
0
.
1
5
,
 
l
i
n
e
w
i
d
t
h
=
0
.
5
)

 
 
 
 
a
x
_
m
c
.
a
x
h
l
i
n
e
(
1
1
0
0
0
0
,
 
c
o
l
o
r
=
G
R
E
E
N
,
 
l
i
n
e
s
t
y
l
e
=
'
-
-
'
,
 
l
i
n
e
w
i
d
t
h
=
2
.
5
,
 
l
a
b
e
l
=
f
'
1
0
%
 
T
a
r
g
e
t
 
(
$
1
1
0
k
)
'
)

 
 
 
 
a
x
_
m
c
.
a
x
h
l
i
n
e
(
9
0
0
0
0
,
 
c
o
l
o
r
=
R
E
D
,
 
l
i
n
e
s
t
y
l
e
=
'
-
-
'
,
 
l
i
n
e
w
i
d
t
h
=
2
.
5
,
 
l
a
b
e
l
=
f
'
1
0
%
 
M
a
x
 
L
o
s
s
 
(
$
9
0
k
)
'
)

 
 
 
 
a
x
_
m
c
.
a
x
h
l
i
n
e
(
1
0
0
0
0
0
,
 
c
o
l
o
r
=
T
E
X
T
_
M
U
T
,
 
l
i
n
e
s
t
y
l
e
=
'
-
'
,
 
l
i
n
e
w
i
d
t
h
=
0
.
8
,
 
a
l
p
h
a
=
0
.
4
)


 
 
 
 
a
x
_
m
c
.
s
e
t
_
x
l
a
b
e
l
(
'
T
r
a
d
i
n
g
 
D
a
y
'
,
 
c
o
l
o
r
=
T
E
X
T
_
S
E
C
,
 
f
o
n
t
s
i
z
e
=
9
)

 
 
 
 
a
x
_
m
c
.
s
e
t
_
y
l
a
b
e
l
(
'
E
q
u
i
t
y
 
(
$
)
'
,
 
c
o
l
o
r
=
T
E
X
T
_
S
E
C
,
 
f
o
n
t
s
i
z
e
=
9
)

 
 
 
 
a
x
_
m
c
.
y
a
x
i
s
.
s
e
t
_
m
a
j
o
r
_
f
o
r
m
a
t
t
e
r
(
p
l
t
.
F
u
n
c
F
o
r
m
a
t
t
e
r
(
l
a
m
b
d
a
 
x
,
 
_
:
 
f
'
$
{
x
:
,
.
0
f
}
'
)
)

 
 
 
 
a
x
_
m
c
.
l
e
g
e
n
d
(
f
o
n
t
s
i
z
e
=
9
,
 
f
a
c
e
c
o
l
o
r
=
B
G
,
 
e
d
g
e
c
o
l
o
r
=
C
A
R
D
_
B
R
D
,
 
l
a
b
e
l
c
o
l
o
r
=
T
E
X
T
_
P
R
I
,
 
f
r
a
m
e
a
l
p
h
a
=
0
.
9
5
)


 
 
 
 
#
 
M
e
d
i
a
n
 
f
i
n
a
l
 
e
q
u
i
t
y
 
a
n
n
o
t
a
t
i
o
n

 
 
 
 
a
x
_
m
c
.
t
e
x
t
(
0
.
5
,
 
0
.
0
3
,
 
f
"
M
e
d
i
a
n
 
F
i
n
a
l
 
E
q
u
i
t
y
:
 
$
{
n
p
.
m
e
d
i
a
n
(
f
i
n
a
l
_
e
q
s
)
:
,
.
0
f
}
"
,
 
t
r
a
n
s
f
o
r
m
=
a
x
_
m
c
.
t
r
a
n
s
A
x
e
s
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
f
o
n
t
s
i
z
e
=
9
,
 
h
a
=
'
c
e
n
t
e
r
'
,
 
c
o
l
o
r
=
T
E
X
T
_
S
E
C
,
 
f
a
m
i
l
y
=
'
m
o
n
o
s
p
a
c
e
'
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
b
b
o
x
=
d
i
c
t
(
b
o
x
s
t
y
l
e
=
'
r
o
u
n
d
,
p
a
d
=
0
.
5
'
,
 
f
a
c
e
c
o
l
o
r
=
C
A
R
D
_
B
G
,
 
e
d
g
e
c
o
l
o
r
=
C
A
R
D
_
B
R
D
,
 
a
l
p
h
a
=
0
.
9
5
)
)


 
 
 
 
p
d
f
.
s
a
v
e
f
i
g
(
f
i
g
,
 
f
a
c
e
c
o
l
o
r
=
B
G
)

 
 
 
 
p
l
t
.
c
l
o
s
e
(
f
i
g
)


#
 
C
o
p
y
 
P
D
F
 
t
o
 
a
r
c
h
i
v
e

s
h
u
t
i
l
.
c
o
p
y
2
(
p
d
f
_
p
a
t
h
,
 
p
d
f
_
a
r
c
h
i
v
e
)


#
 
A
d
d
 
M
C
 
r
e
s
u
l
t
s
 
t
o
 
J
S
O
N

m
c
_
d
a
t
a
 
=
 
{
"
p
a
s
s
_
r
a
t
e
"
:
 
r
o
u
n
d
(
p
a
s
s
_
r
a
t
e
_
m
c
,
 
1
)
,
 
"
n
_
s
i
m
u
l
a
t
i
o
n
s
"
:
 
N
_
S
I
M
,
 
"
n
_
p
a
s
s
e
d
"
:
 
n
_
p
a
s
s
e
d
_
m
c
,

 
 
 
 
 
 
 
 
 
 
 
"
n
_
b
l
o
w
n
_
t
o
t
a
l
"
:
 
n
_
b
l
o
w
n
_
t
,
 
"
n
_
b
l
o
w
n
_
d
a
i
l
y
"
:
 
n
_
b
l
o
w
n
_
d
,
 
"
n
_
s
t
i
l
l
_
t
r
a
d
i
n
g
"
:
 
n
_
s
t
i
l
l
,

 
 
 
 
 
 
 
 
 
 
 
"
m
e
d
i
a
n
_
f
i
n
a
l
_
e
q
u
i
t
y
"
:
 
r
o
u
n
d
(
f
l
o
a
t
(
n
p
.
m
e
d
i
a
n
(
f
i
n
a
l
_
e
q
s
)
)
,
 
2
)
,
 
"
v
e
r
d
i
c
t
"
:
 
v
e
r
d
i
c
t
}

e
x
p
o
r
t
_
j
s
o
n
[
"
m
o
n
t
e
_
c
a
r
l
o
_
f
t
m
o
"
]
 
=
 
m
c
_
d
a
t
a

w
i
t
h
 
o
p
e
n
(
o
s
.
p
a
t
h
.
j
o
i
n
(
L
A
T
E
S
T
_
D
I
R
,
 
"
s
u
m
m
a
r
y
.
j
s
o
n
"
)
,
 
'
w
'
)
 
a
s
 
f
:

 
 
 
 
j
s
o
n
.
d
u
m
p
(
e
x
p
o
r
t
_
j
s
o
n
,
 
f
,
 
i
n
d
e
n
t
=
2
,
 
d
e
f
a
u
l
t
=
s
t
r
)


#
 
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═

#
 
L
O
G
 
T
O
 
G
O
O
G
L
E
 
S
H
E
E
T
S

#
 
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═

t
r
y
:

 
 
 
 
f
r
o
m
 
l
i
b
.
s
h
e
e
t
s
_
l
o
g
g
e
r
 
i
m
p
o
r
t
 
S
h
e
e
t
s
L
o
g
g
e
r

 
 
 
 
_
l
o
g
g
e
r
 
=
 
S
h
e
e
t
s
L
o
g
g
e
r
(
)

 
 
 
 
_
l
o
g
g
e
r
.
l
o
g
_
r
e
s
u
l
t
(
e
x
p
o
r
t
_
j
s
o
n
)

 
 
 
 
_
t
r
a
d
e
s
_
p
a
t
h
 
=
 
o
s
.
p
a
t
h
.
j
o
i
n
(
L
A
T
E
S
T
_
D
I
R
,
 
"
t
r
a
d
e
s
.
c
s
v
"
)

 
 
 
 
i
f
 
o
s
.
p
a
t
h
.
e
x
i
s
t
s
(
_
t
r
a
d
e
s
_
p
a
t
h
)
:

 
 
 
 
 
 
 
 
_
t
r
a
d
e
s
_
d
f
 
=
 
p
d
.
r
e
a
d
_
c
s
v
(
_
t
r
a
d
e
s
_
p
a
t
h
)

 
 
 
 
 
 
 
 
_
l
o
g
g
e
r
.
l
o
g
_
t
r
a
d
e
s
(
T
I
C
K
E
R
,
 
S
T
R
A
T
E
G
Y
_
N
A
M
E
,
 
R
U
N
_
I
D
,
 
_
t
r
a
d
e
s
_
d
f
)

e
x
c
e
p
t
 
E
x
c
e
p
t
i
o
n
 
a
s
 
_
e
:

 
 
 
 
p
r
i
n
t
(
f
"
\
u
2
6
a
0
\
u
f
e
0
f
 
S
h
e
e
t
s
 
l
o
g
g
i
n
g
 
s
k
i
p
p
e
d
:
 
{
_
e
}
"
)


#
 
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═

#
 
F
I
N
A
L
 
S
U
M
M
A
R
Y

#
 
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═

p
r
i
n
t
(
f
"
\
n
{
'
=
'
*
7
0
}
"
)

p
r
i
n
t
(
f
"
\
U
0
0
0
1
f
4
e
6
 
E
X
P
O
R
T
 
C
O
M
P
L
E
T
E
 
\
u
2
0
1
4
 
{
S
T
R
A
T
E
G
Y
_
N
A
M
E
}
 
o
n
 
{
T
I
C
K
E
R
}
"
)

p
r
i
n
t
(
f
"
{
'
=
'
*
7
0
}
"
)

p
r
i
n
t
(
f
"
 
 
R
u
n
 
I
D
:
 
 
 
 
 
 
 
{
R
U
N
_
I
D
}
"
)

p
r
i
n
t
(
f
"
 
 
T
i
m
e
s
t
a
m
p
:
 
 
 
 
{
R
U
N
_
T
I
M
E
S
T
A
M
P
.
s
t
r
f
t
i
m
e
(
'
%
B
 
%
d
,
 
%
Y
 
a
t
 
%
I
:
%
M
:
%
S
 
%
p
'
)
}
"
)

p
r
i
n
t
(
f
"
 
 
I
n
s
t
r
u
m
e
n
t
:
 
 
 
{
T
I
C
K
E
R
}
 
(
{
e
x
p
o
r
t
_
j
s
o
n
[
'
m
e
t
a
d
a
t
a
'
]
[
'
i
n
s
t
r
u
m
e
n
t
_
t
y
p
e
'
]
}
)
"
)

p
r
i
n
t
(
f
"
 
 
P
a
r
a
m
s
:
 
 
 
 
 
 
 
{
p
a
r
a
m
_
s
t
r
}
"
)

p
r
i
n
t
(
f
"
 
 
S
h
a
r
p
e
:
 
 
 
 
 
 
 
{
f
m
t
(
M
[
'
s
h
a
r
p
e
'
]
,
3
)
}
 
(
I
S
:
 
{
f
m
t
(
M
[
'
i
s
_
s
h
a
r
p
e
'
]
,
3
)
}
 
-
>
 
O
O
S
:
 
{
f
m
t
(
M
[
'
o
o
s
_
s
h
a
r
p
e
'
]
,
3
)
}
)
"
)

p
r
i
n
t
(
f
"
 
 
F
T
M
O
 
V
e
r
d
i
c
t
:
 
{
v
e
r
d
i
c
t
}
 
(
{
p
a
s
s
_
r
a
t
e
_
m
c
:
.
1
f
}
%
 
p
a
s
s
 
r
a
t
e
)
"
)

p
r
i
n
t
(
f
"
{
'
=
'
*
7
0
}
"
)

p
r
i
n
t
(
f
"
\
n
\
U
0
0
0
1
f
4
c
2
 
{
S
T
R
A
T
_
D
I
R
}
/
"
)

p
r
i
n
t
(
f
"
 
 
\
u
2
5
1
c
\
u
2
5
0
0
\
u
2
5
0
0
 
l
a
t
e
s
t
/
"
)

p
r
i
n
t
(
f
"
 
 
\
u
2
5
0
2
 
 
 
\
u
2
5
1
c
\
u
2
5
0
0
\
u
2
5
0
0
 
s
u
m
m
a
r
y
.
j
s
o
n
"
)

p
r
i
n
t
(
f
"
 
 
\
u
2
5
0
2
 
 
 
\
u
2
5
1
c
\
u
2
5
0
0
\
u
2
5
0
0
 
d
a
i
l
y
_
r
e
t
u
r
n
s
.
c
s
v
"
)

p
r
i
n
t
(
f
"
 
 
\
u
2
5
0
2
 
 
 
\
u
2
5
1
c
\
u
2
5
0
0
\
u
2
5
0
0
 
t
r
a
d
e
s
.
c
s
v
"
)

p
r
i
n
t
(
f
"
 
 
\
u
2
5
0
2
 
 
 
\
u
2
5
1
c
\
u
2
5
0
0
\
u
2
5
0
0
 
g
r
i
d
_
r
e
s
u
l
t
s
.
c
s
v
"
)

p
r
i
n
t
(
f
"
 
 
\
u
2
5
0
2
 
 
 
\
u
2
5
1
4
\
u
2
5
0
0
\
u
2
5
0
0
 
t
e
a
r
s
h
e
e
t
.
p
d
f
 
 
 
 
 
 
 
\
u
2
1
9
0
 
S
h
a
r
e
 
t
h
i
s
 
w
i
t
h
 
C
l
a
u
d
e
!
"
)

p
r
i
n
t
(
f
"
 
 
\
u
2
5
1
4
\
u
2
5
0
0
\
u
2
5
0
0
 
a
r
c
h
i
v
e
/
"
)

p
r
i
n
t
(
f
"
 
 
 
 
 
 
\
u
2
5
1
c
\
u
2
5
0
0
\
u
2
5
0
0
 
{
R
U
N
_
I
D
}
_
s
u
m
m
a
r
y
.
j
s
o
n
"
)

p
r
i
n
t
(
f
"
 
 
 
 
 
 
\
u
2
5
1
4
\
u
2
5
0
0
\
u
2
5
0
0
 
{
R
U
N
_
I
D
}
_
t
e
a
r
s
h
e
e
t
.
p
d
f
"
)

p
r
i
n
t
(
f
"
\
n
\
U
0
0
0
1
f
4
c
b
 
r
u
n
_
l
o
g
.
c
s
v
 
(
{
l
e
n
(
l
o
g
_
c
o
m
b
i
n
e
d
)
}
 
t
o
t
a
l
 
r
u
n
s
)
"
)

p
r
i
n
t
(
f
"
\
n
\
U
0
0
0
1
f
4
a
1
 
T
o
 
g
e
t
 
m
y
 
a
n
a
l
y
s
i
s
:
 
u
p
l
o
a
d
 
t
h
e
 
t
e
a
r
s
h
e
e
t
.
p
d
f
 
t
o
 
o
u
r
 
c
h
a
t
.
"
)

p
r
i
n
t
(
f
"
 
 
 
F
o
r
 
d
e
e
p
e
r
 
a
n
a
l
y
s
i
s
:
 
a
l
s
o
 
u
p
l
o
a
d
 
s
u
m
m
a
r
y
.
j
s
o
n
 
+
 
d
a
i
l
y
_
r
e
t
u
r
n
s
.
c
s
v
"
)